In [ ]:
!pip install gitpython kagglehub --quiet

In [ ]:
!pip install ultralytics deep-sort-realtime opencv-python matplotlib --q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 91.3 MB/s eta 0:00:00


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("yeeandres/pets2009")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'pets2009' dataset.
Path to dataset files: /kaggle/input/pets2009


In [ ]:
import git
import os

# Cloner le repo GitHub si pas déjà cloné
repo_path = "/content/OpenTraj"

if not os.path.exists(repo_path):
    git.Repo.clone_from("https://github.com/crowdbotp/OpenTraj.git", repo_path)
else:
    print("Le repo GitHub existe déjà.")

# Chemin vers PETS-2009
github_dataset_path = os.path.join(repo_path, "datasets", "PETS-2009")
print("Chemin GitHub PETS-2009 :", github_dataset_path)

Chemin GitHub PETS-2009 : /content/OpenTraj/datasets/PETS-2009


#tracking

In [ ]:
# full_pipeline_with_pretrained_reid.py
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import xml.etree.ElementTree as ET
from collections import defaultdict, deque
import numpy as np
import time
from ultralytics import YOLO
import os
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from scipy.optimize import linear_sum_assignment
import json
import csv
from datetime import datetime
import matplotlib
matplotlib.use('Agg')

# New imports
import networkx as nx
from sklearn.ensemble import IsolationForest
from scipy import ndimage
from sklearn.neighbors import KernelDensity

# ============================================================================
# CONFIGURATION
# ============================================================================

view_path = "/kaggle/input/pets2009/Crowd_PETS09/S2/L1/Time_12-34/View_001"
calibration_file = "/content/OpenTraj/datasets/PETS-2009/data/calibration/View_001.xml"
output_visualization_dir = "/content/visualizations/"
output_trajectories_csv = "/content/trajectories_corrected.csv"
output_metrics_csv = "/content/graph_metrics.csv"
output_trajectories_json = "/content/trajectories_corrected.json"

# Gestion robuste du modèle
MODEL_PERSON_PATH = "/content/best.pt"
os.makedirs(output_visualization_dir, exist_ok=True)

# ============================================================================
# MODÈLE REID PRÉ-ENTRAÎNÉ
# ============================================================================

class PretrainedReID:
    """Modèle ReID pré-entraîné pour l'extraction de features d'apparence"""

    def __init__(self, model_name='resnet50', device='auto'):
        self.device = torch.device('cuda' if torch.cuda.is_available() and device != 'cpu' else 'cpu')
        self.model_name = model_name
        self.model = None
        self.preprocess = None

        self._load_pretrained_model()
        print(f"✅ Modèle ReID {model_name} chargé sur {self.device}")

    def _load_pretrained_model(self):
        """Charge un modèle pré-entraîné pour l'extraction de features"""
        try:
            if self.model_name.startswith('resnet'):
                # Chargement ResNet pré-entraîné
                if self.model_name == 'resnet50':
                    self.model = models.resnet50(pretrained=True)
                elif self.model_name == 'resnet34':
                    self.model = models.resnet34(pretrained=True)
                elif self.model_name == 'resnet18':
                    self.model = models.resnet18(pretrained=True)

                # Supprimer la dernière couche (classification)
                self.model = nn.Sequential(*list(self.model.children())[:-1])

            elif self.model_name.startswith('mobilenet'):
                # MobileNet
                if self.model_name == 'mobilenet_v2':
                    self.model = models.mobilenet_v2(pretrained=True)
                    self.model.classifier = nn.Sequential(
                        nn.AdaptiveAvgPool2d((1, 1)),
                        nn.Flatten()
                    )

            self.model.eval()
            self.model.to(self.device)

            # Transformation des images
            self.preprocess = transforms.Compose([
                transforms.ToPILImage(),
                transforms.Resize((256, 128)),  # Taille standard pour ReID
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]
                )
            ])

        except Exception as e:
            print(f"❌ Erreur chargement modèle ReID: {e}")
            self._load_fallback_model()

    def _load_fallback_model(self):
        """Charge un modèle de repli plus simple"""
        try:
            self.model = models.resnet18(pretrained=True)
            self.model = nn.Sequential(*list(self.model.children())[:-1])
            self.model.eval()
            self.model.to(self.device)

            self.preprocess = transforms.Compose([
                transforms.ToPILImage(),
                transforms.Resize((128, 64)),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]
                )
            ])
            print("✅ Modèle de repli chargé")
        except Exception as e:
            print(f"❌ Erreur chargement modèle de repli: {e}")
            self.model = None

    def extract_features(self, image):
        """Extrait les features d'apparence d'une image"""
        if self.model is None or image.size == 0:
            return np.zeros(512, dtype=np.float32)

        try:
            # Prétraitement
            input_tensor = self.preprocess(image).unsqueeze(0).to(self.device)

            # Extraction des features
            with torch.no_grad():
                features = self.model(input_tensor)
                features = features.squeeze().cpu().numpy()

            # Normalisation L2
            if np.linalg.norm(features) > 0:
                features = features / np.linalg.norm(features)

            return features.astype(np.float32)

        except Exception as e:
            print(f"⚠ Erreur extraction features: {e}")
            return np.zeros(512, dtype=np.float32)

# ============================================================================
# CALIBRATION CAMÉRA
# ============================================================================

class CameraCalibration:
    """Calibration caméra corrigée pour PETS2009"""

    def __init__(self, calibration_file):
        self.calibration_file = calibration_file
        self.camera_params = {}
        self.homography_matrix = None

        self.load_calibration()
        self.compute_accurate_homography()
        print("✅ Calibration caméra corrigée initialisée")

    def load_calibration(self):
        """Charge les paramètres de calibration avec gestion d'erreur"""
        try:
            tree = ET.parse(self.calibration_file)
            root = tree.getroot()

            # Paramètres géométriques
            geometry = root.find('Geometry')
            if geometry is not None:
                self.camera_params.update({
                    'width': int(geometry.get('width', '768')),
                    'height': int(geometry.get('height', '576')),
                    'cx': float(geometry.get('ncx', '795.0')),
                    'cy': float(geometry.get('nfx', '752.0')),
                    'dx': float(geometry.get('dx', '0.00485')),
                    'dy': float(geometry.get('dy', '0.00465'))
                })

            # Paramètres intrinsèques
            intrinsic = root.find('Intrinsic')
            if intrinsic is not None:
                self.camera_params.update({
                    'focal': float(intrinsic.get('focal', '5.5549183034')),
                    'kappa1': float(intrinsic.get('kappa1', '0.0051113043639')),
                    'cx_pixel': float(intrinsic.get('cx', '324.22149053')),
                    'cy_pixel': float(intrinsic.get('cy', '282.56650051')),
                    'sx': float(intrinsic.get('sx', '1.0937855397'))
                })

            print(f"📷 Calibration chargée: {self.camera_params.get('width','?')}x{self.camera_params.get('height','?')}")

        except Exception as e:
            print(f"❌ Erreur calibration: {e}")
            self.camera_params = {
                'width': 768, 'height': 576,
                'cx_pixel': 324.22, 'cy_pixel': 282.57,
                'focal': 5.55
            }

    def compute_accurate_homography(self):
        """Calcule une homographie précise pour PETS2009 S2/L1"""
        try:
            src_points = np.array([
                [184, 441], [583, 441], [384, 323], [284, 382], [483, 382]
            ], dtype=np.float32)

            dst_points = np.array([
                [0, 0], [15, 0], [7.5, 12], [3, 6], [12, 6]
            ], dtype=np.float32)

            self.homography_matrix, status = cv2.findHomography(
                src_points, dst_points, cv2.RANSAC, 5.0
            )

            if self.homography_matrix is not None:
                print("✅ Homographie précise calculée")
            else:
                raise Exception("Calcul homographie échoué")

        except Exception as e:
            print(f"❌ Erreur calcul homographie: {e}")
            self.homography_matrix = np.eye(3)

    def pixel_to_world_corrected(self, pixel_x, pixel_y, foot_position=True):
        """Convertit pixels → monde avec correction de la position des pieds"""
        try:
            if foot_position:
                y_foot = pixel_y
            else:
                y_foot = pixel_y

            point = np.array([pixel_x, y_foot, 1.0])
            world_point = np.dot(self.homography_matrix, point)

            if world_point[2] != 0:
                world_x = world_point[0] / world_point[2]
                world_y = world_point[1] / world_point[2]
            else:
                world_x, world_y = pixel_x, pixel_y

            return float(world_x), float(world_y)

        except Exception as e:
            print(f"⚠ Erreur conversion: {e}")
            return float(pixel_x), float(pixel_y)

# ============================================================================
# DÉTECTEUR
# ============================================================================

class PersonDetector:
    def __init__(self, model_path=MODEL_PERSON_PATH, conf_threshold=0.4):
        self.conf_threshold = conf_threshold

        try:
            if os.path.exists(model_path) and os.path.getsize(model_path) > 1024 * 1024:
                self.yolo_model = YOLO(model_path)
                print(f"✅ Modèle personnalisé chargé: {model_path}")
            else:
                self.yolo_model = YOLO('yolov8n.pt')
                print("🔄 Utilisation de YOLOv8n (modèle par défaut)")
        except:
            self.yolo_model = YOLO('yolov8n.pt')
            print("🔄 Utilisation de YOLOv8n (fallback)")

        self.class_names = self.yolo_model.names
        print(f"📋 Classes: {self.class_names}")

    def detect_people(self, frame):
        """Détection avec positions corrigées"""
        try:
            results = self.yolo_model(
                frame, conf=self.conf_threshold, verbose=False, imgsz=640
            )

            detections = []
            for result in results:
                boxes = result.boxes
                if boxes is not None:
                    for box in boxes:
                        class_id = int(box.cls[0])
                        class_name = self.class_names[class_id]

                        if 'person' in class_name.lower() or class_id == 0:
                            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                            confidence = float(box.conf[0].cpu().numpy())

                            x_center = (x1 + x2) / 2
                            y_center = (y1 + y2) / 2
                            y_foot = y2

                            detections.append({
                                'bbox': [x1, y1, x2, y2],
                                'center': [x_center, y_center],
                                'foot': [x_center, y_foot],
                                'confidence': confidence,
                                'class_id': class_id,
                                'width': x2 - x1,
                                'height': y2 - y1
                            })

            print(f"🔍 Détections: {len(detections)} personnes")
            return detections

        except Exception as e:
            print(f"❌ Erreur détection: {e}")
            return []

# ============================================================================
# BOT-SORT AVEC REID PRÉ-ENTRAÎNÉ
# ============================================================================

class BotSortTrackerWithReID:
    def __init__(self, track_thresh=0.4, track_buffer=30, match_thresh=0.7,
                 appearance_thresh=0.25, motion_model=True, reid_model='resnet50'):
        self.track_thresh = track_thresh
        self.track_buffer = track_buffer
        self.match_thresh = match_thresh
        self.appearance_thresh = appearance_thresh
        self.motion_model = motion_model

        self.tracked_tracks = []
        self.lost_tracks = []
        self.removed_tracks = []

        self.frame_id = 0
        self.max_time_lost = track_buffer
        self.next_id = 1

        # Modèle ReID pré-entraîné
        self.reid_model = PretrainedReID(model_name=reid_model)
        self.appearance_features = {}
        self.velocity_estimates = {}

        print("✅ Tracker Bot-SORT avec ReID pré-entraîné initialisé")

    def _extract_appearance_features(self, detection, frame):
        """Extrait les features d'apparence avec le modèle pré-entraîné"""
        try:
            x1, y1, x2, y2 = [int(coord) for coord in detection['bbox']]
            h, w = frame.shape[:2]
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w-1, x2), min(h-1, y2)

            if x2 <= x1 or y2 <= y1:
                return np.zeros(512, dtype=np.float32)

            roi = frame[y1:y2, x1:x2]
            if roi.size == 0:
                return np.zeros(512, dtype=np.float32)

            # Extraction avec modèle pré-entraîné
            features = self.reid_model.extract_features(roi)
            return features

        except Exception as e:
            print(f"⚠ Erreur extraction apparence: {e}")
            return np.zeros(512, dtype=np.float32)

    def _appearance_similarity(self, features1, features2):
        """Calcule la similarité cosine entre features d'apparence"""
        if features1 is None or features2 is None:
            return 0.0

        norm1 = np.linalg.norm(features1)
        norm2 = np.linalg.norm(features2)

        if norm1 == 0 or norm2 == 0:
            return 0.0

        similarity = np.dot(features1, features2) / (norm1 * norm2)
        return max(0.0, min(1.0, similarity))

    def _predict_with_motion(self, track):
        """Prédit la position suivante avec modèle de mouvement"""
        track_id = track['track_id']

        if track_id not in self.velocity_estimates:
            self.velocity_estimates[track_id] = {'vx': 0, 'vy': 0, 'count': 0}
            return track['center']

        if len(track.get('position_history', [])) < 2:
            return track['center']

        vel = self.velocity_estimates[track_id]
        pred_x = track['center'][0] + vel['vx']
        pred_y = track['center'][1] + vel['vy']

        return [pred_x, pred_y]

    def update(self, detections, frame):
        """Met à jour avec apparence et modèle de mouvement"""
        self.frame_id += 1

        detections_list = []
        for det in detections:
            # Extraction des features d'apparence avec modèle pré-entraîné
            appearance_feat = self._extract_appearance_features(det, frame)

            detections_list.append({
                'bbox': det['bbox'],
                'score': det['confidence'],
                'center': det['center'],
                'foot': det['foot'],
                'class_id': det['class_id'],
                'appearance': appearance_feat
            })

        self._match_detections_to_tracks(detections_list)
        self._manage_tracks()

        return self._get_active_tracks()

    def _match_detections_to_tracks(self, detections):
        """Association avec apparence et mouvement"""
        for track in self.tracked_tracks:
            track['active'] = True

        active_tracks = [t for t in self.tracked_tracks if t['active']]

        if not detections:
            for track in active_tracks:
                track['active'] = False
            return

        if not active_tracks:
            for det in detections:
                if det['score'] > self.track_thresh:
                    self._create_new_track(det)
            return

        # Matrice de coût combinée
        cost_matrix = np.ones((len(active_tracks), len(detections)))

        for i, track in enumerate(active_tracks):
            predicted_pos = self._predict_with_motion(track)

            for j, det in enumerate(detections):
                # Coût IoU
                iou_cost = 1 - self._calculate_iou(track['bbox'], det['bbox'])

                # Coût apparence
                appearance_cost = 0
                if track['track_id'] in self.appearance_features:
                    similarity = self._appearance_similarity(
                        self.appearance_features[track['track_id']],
                        det['appearance']
                    )
                    appearance_cost = 1 - similarity

                # Coût mouvement
                motion_cost = np.linalg.norm(
                    np.array(predicted_pos) - np.array(det['center'])
                ) / 100.0

                # Combinaison pondérée
                cost_matrix[i, j] = (
                    0.4 * iou_cost +
                    0.4 * appearance_cost +
                    0.2 * motion_cost
                )

        row_ind, col_ind = linear_sum_assignment(cost_matrix)

        matched_pairs = []
        for i, j in zip(row_ind, col_ind):
            if cost_matrix[i, j] < (1 - self.match_thresh):
                matched_pairs.append((i, j))

        # Mise à jour des tracks
        for i, j in matched_pairs:
            track = active_tracks[i]
            det = detections[j]

            # Mise à jour du modèle de mouvement
            self._update_motion_model(track, det['center'])

            # Sauvegarde des features d'apparence
            self.appearance_features[track['track_id']] = det['appearance']

            # Mise à jour historique des positions
            if 'position_history' not in track:
                track['position_history'] = []
            track['position_history'].append(det['center'])

            track.update({
                'bbox': det['bbox'],
                'score': det['score'],
                'center': det['center'],
                'foot': det['foot'],
                'last_update': self.frame_id,
                'active': True
            })

        matched_track_indices = [i for i, _ in matched_pairs]
        for i, track in enumerate(active_tracks):
            if i not in matched_track_indices:
                track['active'] = False

        matched_det_indices = [j for _, j in matched_pairs]
        for j, det in enumerate(detections):
            if j not in matched_det_indices and det['score'] > self.track_thresh:
                self._create_new_track(det)

    def _update_motion_model(self, track, new_position):
        """Met à jour le modèle de mouvement"""
        track_id = track['track_id']

        if 'position_history' not in track:
            track['position_history'] = [new_position]
            return

        track['position_history'].append(new_position)

        if len(track['position_history']) > 10:
            track['position_history'] = track['position_history'][-10:]

        if len(track['position_history']) >= 2:
            positions = np.array(track['position_history'])
            velocities = np.diff(positions, axis=0)
            avg_velocity = np.mean(velocities[-3:], axis=0) if len(velocities) >= 3 else velocities[-1]

            self.velocity_estimates[track_id] = {
                'vx': avg_velocity[0],
                'vy': avg_velocity[1],
                'count': len(velocities)
            }

    def _create_new_track(self, detection):
        """Crée un nouveau track avec apparence"""
        new_track = {
            'track_id': self.next_id,
            'bbox': detection['bbox'],
            'score': detection['score'],
            'center': detection['center'],
            'foot': detection['foot'],
            'start_frame': self.frame_id,
            'last_update': self.frame_id,
            'active': True,
            'position_history': [detection['center']]
        }
        self.tracked_tracks.append(new_track)

        # Sauvegarde des features d'apparence
        self.appearance_features[self.next_id] = detection['appearance']

        print(f"🆕 Track {self.next_id} créé (ReID features: {len(detection['appearance'])})")
        self.next_id += 1

    def _calculate_iou(self, box1, box2):
        x1_1, y1_1, x2_1, y2_1 = box1
        x1_2, y1_2, x2_2, y2_2 = box2

        xi1 = max(x1_1, x1_2)
        yi1 = max(y1_1, y1_2)
        xi2 = min(x2_1, x2_2)
        yi2 = min(y2_1, y2_2)

        inter_area = max(0, xi2 - xi1) * max(0, yi2 - yi1)
        box1_area = (x2_1 - x1_1) * (y2_1 - y1_1)
        box2_area = (x2_2 - x1_2) * (y2_2 - y1_2)
        union_area = box1_area + box2_area - inter_area

        return inter_area / union_area if union_area > 0 else 0

    def _manage_tracks(self):
        current_time = self.frame_id
        tracks_to_remove = []

        for track in self.tracked_tracks:
            if not track['active']:
                if current_time - track['last_update'] > self.max_time_lost:
                    tracks_to_remove.append(track)

        for track in tracks_to_remove:
            self.tracked_tracks.remove(track)
            if track['track_id'] in self.appearance_features:
                del self.appearance_features[track['track_id']]
            if track['track_id'] in self.velocity_estimates:
                del self.velocity_estimates[track['track_id']]
            self.removed_tracks.append(track)

    def _get_active_tracks(self):
        """Retourne les tracks actifs avec historique"""
        active_tracks = []
        for track in self.tracked_tracks:
            if track['active']:
                x1, y1, x2, y2 = track['bbox']
                w = x2 - x1
                h = y2 - y1

                active_tracks.append({
                    'id': track['track_id'],
                    'frame': self.frame_id,
                    'bbox': track['bbox'],
                    'center': track['center'],
                    'foot': track['foot'],
                    'width': w,
                    'height': h,
                    'score': track['score'],
                    'position_history': track.get('position_history', [])
                })

        return active_tracks

# ============================================================================
# EXTRACTEUR DE TRAJECTOIRES
# ============================================================================

class TrajectoryExtractor:
    def __init__(self, calibration=None):
        self.calibration = calibration
        self.trajectories = defaultdict(list)
        self.trajectories_data = []
        self.complete_trajectories = defaultdict(list)

        print("✅ Extracteur de trajectoires corrigé initialisé")

    def extract_trajectories(self, tracks, frame_width, frame_height):
        """Extrait les trajectoires avec positions corrigées"""
        current_trajectories = {}

        for track in tracks:
            track_id = track['id']

            positions = {
                'bbox_center': track['center'],
                'foot_position': track['foot'],
                'bbox': track['bbox']
            }

            world_positions = {}
            if self.calibration:
                for pos_name, pos in positions.items():
                    if pos_name == 'foot_position':
                        world_x, world_y = self.calibration.pixel_to_world_corrected(
                            pos[0], pos[1], foot_position=True
                        )
                    else:
                        world_x, world_y = self.calibration.pixel_to_world_corrected(
                            pos[0], pos[1], foot_position=False
                        )
                    world_positions[pos_name] = (world_x, world_y)

            trajectory_point = {
                'id': track_id,
                'frame': track['frame'],
                'pixel_center_x': float(positions['bbox_center'][0]),
                'pixel_center_y': float(positions['bbox_center'][1]),
                'pixel_foot_x': float(positions['foot_position'][0]),
                'pixel_foot_y': float(positions['foot_position'][1]),
                'world_center_x': float(world_positions.get('bbox_center', (0, 0))[0]),
                'world_center_y': float(world_positions.get('bbox_center', (0, 0))[1]),
                'world_foot_x': float(world_positions.get('foot_position', (0, 0))[0]),
                'world_foot_y': float(world_positions.get('foot_position', (0, 0))[1]),
                'width': float(track['width']),
                'height': float(track['height']),
                'score': float(track['score']),
                'has_calibration': self.calibration is not None
            }

            self.trajectories_data.append(trajectory_point)
            self.trajectories[track_id].append(trajectory_point)

            self.complete_trajectories[track_id].append({
                'frame': track['frame'],
                'pixel_foot_x': float(positions['foot_position'][0]),
                'pixel_foot_y': float(positions['foot_position'][1]),
                'world_foot_x': float(world_positions.get('foot_position', (0, 0))[0]),
                'world_foot_y': float(world_positions.get('foot_position', (0, 0))[1])
            })

            current_trajectories[track_id] = trajectory_point

        print(f"📈 Trajectoires extraites: {len(current_trajectories)}")
        return current_trajectories

    def save_trajectories_to_csv(self, filename):
        """Sauvegarde avec toutes les positions"""
        try:
            with open(filename, 'w', newline='') as csvfile:
                fieldnames = [
                    'id', 'frame',
                    'pixel_center_x', 'pixel_center_y',
                    'pixel_foot_x', 'pixel_foot_y',
                    'world_center_x', 'world_center_y',
                    'world_foot_x', 'world_foot_y',
                    'width', 'height', 'score', 'has_calibration'
                ]
                writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
                writer.writeheader()
                for trajectory in self.trajectories_data:
                    writer.writerow(trajectory)

            print(f"✅ Trajectoires corrigées sauvegardées: {filename}")
            return True
        except Exception as e:
            print(f"❌ Erreur sauvegarde CSV: {e}")
            return False

# ============================================================================
# VISUALISATION AVEC HEATMAP ET TRAJETS
# ============================================================================

class AdvancedVisualizer:
    """Visualisation avancée avec heatmap et trajets superposés"""

    def __init__(self, output_dir):
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)

    def create_density_heatmap(self, trajectories_data, frame_shape, filename_suffix=""):
        """Crée une heatmap de densité des positions"""
        try:
            foot_positions = []
            for point in trajectories_data:
                foot_positions.append([point['pixel_foot_x'], point['pixel_foot_y']])

            if not foot_positions:
                print("❌ Aucune position pour la heatmap")
                return None

            foot_positions = np.array(foot_positions)

            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

            h, w = frame_shape[:2]
            heatmap, xedges, yedges = np.histogram2d(
                foot_positions[:, 0], foot_positions[:, 1],
                bins=[50, 50], range=[[0, w], [0, h]]
            )

            heatmap = ndimage.gaussian_filter(heatmap, sigma=1.5)

            im1 = ax1.imshow(heatmap.T, origin='lower', aspect='auto',
                           extent=[0, w, 0, h], cmap='hot_r')
            ax1.set_title('🔥 Heatmap de Densité - Positions Pieds', fontweight='bold')
            ax1.set_xlabel('X (pixels)')
            ax1.set_ylabel('Y (pixels)')
            plt.colorbar(im1, ax=ax1, label='Densité')

            if len(foot_positions) > 10:
                kde = KernelDensity(kernel='gaussian', bandwidth=20).fit(foot_positions)

                x_grid = np.linspace(0, w, 100)
                y_grid = np.linspace(0, h, 100)
                X, Y = np.meshgrid(x_grid, y_grid)
                grid_points = np.c_[X.ravel(), Y.ravel()]

                log_density = kde.score_samples(grid_points)
                density = np.exp(log_density).reshape(X.shape)

                im2 = ax2.contourf(X, Y, density, levels=20, cmap='viridis')
                ax2.set_title('📊 Heatmap KDE - Distribution Probabiliste', fontweight='bold')
                ax2.set_xlabel('X (pixels)')
                ax2.set_ylabel('Y (pixels)')
                plt.colorbar(im2, ax=ax2, label='Densité Log')
            else:
                ax2.text(0.5, 0.5, 'Pas assez de données\npour KDE',
                        ha='center', va='center', transform=ax2.transAxes)
                ax2.set_title('📊 Heatmap KDE', fontweight='bold')

            plt.tight_layout()
            filename = f"{self.output_dir}density_heatmap{filename_suffix}.png"
            plt.savefig(filename, dpi=150, bbox_inches='tight')
            plt.close()

            print(f"🔥 Heatmap sauvegardée: {filename}")
            return filename

        except Exception as e:
            print(f"❌ Erreur création heatmap: {e}")
            return None

    def create_trajectory_overlay(self, complete_trajectories, background_frame, filename_suffix=""):
        """Crée une visualisation des trajets superposés"""
        try:
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

            ax1.imshow(background_frame)
            ax1.set_title('🛤 Trajets Superposés - Vue Image', fontweight='bold')

            ax2.set_title('📈 Diagramme des Trajets', fontweight='bold')
            ax2.set_xlabel('X (pixels)')
            ax2.set_ylabel('Y (pixels)')
            ax2.grid(True, alpha=0.3)
            ax2.invert_yaxis()

            colors = plt.cm.tab10(np.linspace(0, 1, len(complete_trajectories)))

            for i, (track_id, trajectory) in enumerate(complete_trajectories.items()):
                if len(trajectory) < 2:
                    continue

                color = colors[i % len(colors)]

                x_coords = [point['pixel_foot_x'] for point in trajectory]
                y_coords = [point['pixel_foot_y'] for point in trajectory]

                ax1.plot(x_coords, y_coords, linewidth=2, color=color, alpha=0.7)
                ax1.scatter(x_coords[-1], y_coords[-1], color=color, s=50,
                           label=f'ID {track_id}' if i < 10 else "")

                ax2.plot(x_coords, y_coords, linewidth=2, color=color, alpha=0.7,
                        label=f'ID {track_id}' if i < 10 else "")
                ax2.scatter(x_coords[-1], y_coords[-1], color=color, s=30)

            if len(complete_trajectories) <= 10:
                ax1.legend(loc='upper right')
                ax2.legend(loc='upper right')
            else:
                ax1.legend(loc='upper right', ncol=2)
                ax2.legend(loc='upper right', ncol=2)

            plt.tight_layout()
            filename = f"{self.output_dir}trajectory_overlay{filename_suffix}.png"
            plt.savefig(filename, dpi=150, bbox_inches='tight')
            plt.close()

            print(f"🛤 Trajets superposés sauvegardés: {filename}")
            return filename

        except Exception as e:
            print(f"❌ Erreur création trajets superposés: {e}")
            return None

    def create_comprehensive_visualization(self, frame, detections, tracks, trajectories,
                                         complete_trajectories, frame_idx, reid_features_count=0):
        """Crée une visualisation complète avec toutes les informations"""
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(20, 12))
        fig.suptitle(f'VISUALISATION COMPLÈTE - Frame {frame_idx} (ReID: {reid_features_count} features)',
                    fontsize=16, fontweight='bold')

        ax1.imshow(frame)
        ax1.set_title('🎬 Détections + Tracking Bot-SORT avec ReID', fontweight='bold')

        for det in detections:
            x1, y1, x2, y2 = det['bbox']
            rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                   linewidth=2, edgecolor='green', facecolor='none', alpha=0.7)
            ax1.add_patch(rect)

        for track in tracks:
            x1, y1, x2, y2 = track['bbox']
            rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                   linewidth=2, edgecolor='red', facecolor='none')
            ax1.add_patch(rect)

            ax1.plot(track['center'][0], track['center'][1], 'ro', markersize=4)
            ax1.plot(track['foot'][0], track['foot'][1], 'bo', markersize=4)

            ax1.text(x1, y1-10, f"ID:{track['id']}", color='white',
                    fontweight='bold', backgroundcolor='red')

        ax1.legend([patches.Patch(facecolor='none', edgecolor='green', label='Détection'),
                   patches.Patch(facecolor='none', edgecolor='red', label='Track')],
                  ['Détections', 'Tracks'])
        ax1.axis('off')

        if trajectories:
            foot_positions = [[t['pixel_foot_x'], t['pixel_foot_y']] for t in trajectories.values()]
            foot_positions = np.array(foot_positions)

            h, w = frame.shape[:2]
            heatmap, xedges, yedges = np.histogram2d(
                foot_positions[:, 0], foot_positions[:, 1],
                bins=[30, 30], range=[[0, w], [0, h]]
            )
            heatmap = ndimage.gaussian_filter(heatmap, sigma=1.0)

            im2 = ax2.imshow(heatmap.T, origin='lower', aspect='auto',
                           extent=[0, w, 0, h], cmap='hot_r', alpha=0.8)
            ax2.imshow(frame, alpha=0.5)
            ax2.set_title('🔥 Heatmap Instantanée', fontweight='bold')
            plt.colorbar(im2, ax=ax2, label='Densité')
            ax2.axis('off')

        ax3.imshow(frame, alpha=0.3)
        ax3.set_title('🛤 Trajets en Temps Réel', fontweight='bold')

        colors = plt.cm.tab10(np.linspace(0, 1, len(complete_trajectories)))
        for i, (track_id, traj) in enumerate(complete_trajectories.items()):
            if len(traj) < 2:
                continue
            color = colors[i % len(colors)]
            x_coords = [point['pixel_foot_x'] for point in traj]
            y_coords = [point['pixel_foot_y'] for point in traj]
            ax3.plot(x_coords, y_coords, linewidth=2, color=color, alpha=0.8)
            ax3.scatter(x_coords[-1], y_coords[-1], color=color, s=50)
            ax3.text(x_coords[-1], y_coords[-1], f' {track_id}',
                    color=color, fontweight='bold')
        ax3.axis('off')

        ax4.axis('off')
        stats_text = "📊 STATISTIQUES AVANCÉES\n\n"

        if trajectories:
            stats_text += f"• Tracks actifs: {len(tracks)}\n"
            stats_text += f"• Trajectoires: {len(complete_trajectories)}\n"
            stats_text += f"• Features ReID: {reid_features_count}\n"

            velocities = []
            for track_id, traj in complete_trajectories.items():
                if len(traj) >= 2:
                    dx = traj[-1]['pixel_foot_x'] - traj[0]['pixel_foot_x']
                    dy = traj[-1]['pixel_foot_y'] - traj[0]['pixel_foot_y']
                    frames = traj[-1]['frame'] - traj[0]['frame']
                    if frames > 0:
                        speed = np.sqrt(dx**2 + dy**2) / frames
                        velocities.append(speed)

            if velocities:
                stats_text += f"• Vitesse moy: {np.mean(velocities):.1f} px/frame\n"
                stats_text += f"• Vitesse max: {np.max(velocities):.1f} px/frame\n"

            if len(trajectories) > 0:
                area = frame.shape[1] * frame.shape[0]
                density = len(trajectories) / (area / 10000)
                stats_text += f"• Densité: {density:.1f} pers/10kpx\n"

        ax4.text(0.05, 0.95, stats_text, transform=ax4.transAxes, fontsize=11,
                verticalalignment='top', bbox=dict(boxstyle="round,pad=0.3", facecolor="lightgray"))

        plt.tight_layout()
        filename = f"{self.output_dir}comprehensive_frame_{frame_idx:04d}.png"
        plt.savefig(filename, dpi=150, bbox_inches='tight')
        plt.close()

        print(f"💾 Visualisation complète sauvegardée: {filename}")

# ============================================================================
# CLASSES SUPPLEMENTAIRES
# ============================================================================

class InteractionGraphBuilder:
    def __init__(self, distance_threshold=80):
        self.distance_threshold = distance_threshold

    def build_graph(self, trajectories_frame):
        G = nx.Graph()
        track_ids = list(trajectories_frame.keys())

        for tid in track_ids:
            G.add_node(tid)

        for i in range(len(track_ids)):
            for j in range(i+1, len(track_ids)):
                id1 = track_ids[i]
                id2 = track_ids[j]
                p1 = trajectories_frame[id1]
                p2 = trajectories_frame[id2]

                dx = p1["pixel_foot_x"] - p2["pixel_foot_x"]
                dy = p1["pixel_foot_y"] - p2["pixel_foot_y"]
                dist = np.sqrt(dx*dx + dy*dy)

                if dist < self.distance_threshold:
                    G.add_edge(id1, id2, weight=1.0/(dist + 1e-6))

        return G

class GraphMetricsAnalyzer:
    def __init__(self):
        self.timeseries = []

    def compute(self, G, frame_id):
        num_nodes = G.number_of_nodes()
        num_edges = G.number_of_edges()

        metrics = {
            "frame": frame_id,
            "nodes": num_nodes,
            "edges": num_edges,
            "density": nx.density(G) if num_nodes > 1 else 0.0,
            "avg_degree": np.mean([d for _, d in G.degree()]) if num_nodes > 0 else 0.0,
            "clustering": nx.average_clustering(G) if num_nodes > 1 else 0.0
        }

        if num_nodes > 0:
            comps = list(nx.connected_components(G))
            giant = max(comps, key=len)
            metrics["giant_component_frac"] = len(giant) / num_nodes
        else:
            metrics["giant_component_frac"] = 0.0

        self.timeseries.append(metrics)
        return metrics

class RuleBasedAnomalyDetector:
    def __init__(self, density_jump=0.25, cluster_drop=0.2):
        self.prev_metrics = None
        self.density_jump = density_jump
        self.cluster_drop = cluster_drop

    def detect(self, metrics):
        if self.prev_metrics is None:
            self.prev_metrics = metrics
            return False, None

        anomaly = False
        reasons = []

        if metrics["density"] - self.prev_metrics["density"] > self.density_jump:
            anomaly = True
            reasons.append("Regroupement soudain (density ↑)")

        if self.prev_metrics["clustering"] - metrics["clustering"] > self.cluster_drop:
            anomaly = True
            reasons.append("Perte de cohésion (clustering ↓)")

        self.prev_metrics = metrics
        return anomaly, "; ".join(reasons) if reasons else None

class IsolationForestAnomalyDetector:
    def __init__(self, window_size=50, retrain_every=20, contamination=0.05, random_state=42):
        self.window_size = window_size
        self.retrain_every = retrain_every
        self.contamination = contamination
        self.random_state = random_state

        self.buffer = deque(maxlen=window_size)
        self.frame_idx_buffer = deque(maxlen=window_size)
        self.if_model = None
        self.last_retrain = 0

    def _metrics_to_vector(self, m):
        return np.array([m["density"], m["avg_degree"], m["clustering"], m["giant_component_frac"], m["nodes"]], dtype=float)

    def update_and_score(self, metrics, frame_idx):
        vec = self._metrics_to_vector(metrics)
        self.buffer.append(vec)
        self.frame_idx_buffer.append(frame_idx)

        if len(self.buffer) >= max(10, int(self.window_size/5)):
            if (self.if_model is None) or (frame_idx - self.last_retrain >= self.retrain_every):
                X = np.vstack(self.buffer)
                self.if_model = IsolationForest(contamination=self.contamination, random_state=self.random_state)
                self.if_model.fit(X)
                self.last_retrain = frame_idx

        if self.if_model is not None:
            score = self.if_model.decision_function(vec.reshape(1, -1))[0]
            is_anomaly = self.if_model.predict(vec.reshape(1, -1))[0] == -1
            return bool(is_anomaly), float(score)
        else:
            return False, None

class Preprocessor:
    def __init__(self, view_path, target_size=(640, 480)):
        self.view_path = view_path
        self.target_size = target_size
        self.frames = []
        self.original_frames = []

    def load_and_preprocess_frames(self, max_frames=50):
        print("🔄 Chargement des frames...")
        self.frames = []
        self.original_frames = []

        # List all files in the directory and filter for images
        all_files = sorted(os.listdir(self.view_path))
        image_files = [f for f in all_files if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))]

        # Limit frames if max_frames is specified and not None
        if max_frames is not None:
            image_files = image_files[:max_frames]

        for i, file_name in enumerate(image_files):
            frame_path = os.path.join(self.view_path, file_name)
            frame = cv2.imread(frame_path)
            if frame is not None:
                original_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                self.original_frames.append(original_rgb)

                frame = cv2.resize(frame, self.target_size)
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                self.frames.append(frame_rgb)
            else:
                print(f"⚠ Could not load frame: {frame_path}")


        print(f"✅ {len(self.frames)} frames chargées")
        return self.frames

# ============================================================================
# PIPELINE COMPLET AVEC REID PRÉ-ENTRAÎNÉ
# ============================================================================

def run_complete_pipeline_with_reid():
    """Exécute le pipeline complet avec ReID pré-entraîné"""
    print("🎯 PIPELINE COMPLET - Bot-SORT + ReID Pré-entraîné + Heatmap + Trajets")
    print("=" * 80)

    # 1. Initialisation
    print("\n1. 🔧 INITIALISATION AVEC REID")
    calibration = CameraCalibration(calibration_file)
    detector = PersonDetector()
    tracker = BotSortTrackerWithReID(
        track_thresh=0.4,
        track_buffer=30,
        match_thresh=0.7,
        appearance_thresh=0.25,
        motion_model=True,
        reid_model='resnet50'  # ou 'resnet18', 'mobilenet_v2'
    )
    trajectory_extractor = TrajectoryExtractor(calibration)
    visualizer = AdvancedVisualizer(output_visualization_dir)

    # Graph / metrics / anomalies
    graph_builder = InteractionGraphBuilder(distance_threshold=80)
    metrics_analyzer = GraphMetricsAnalyzer()
    rule_detector = RuleBasedAnomalyDetector(density_jump=0.20, cluster_drop=0.15)
    iso_detector = IsolationForestAnomalyDetector(window_size=80, retrain_every=10, contamination=0.05)

    metrics_csv_rows = []

    # 2. Chargement des frames
    print("\n2. 🎬 CHARGEMENT DES FRAMES")
    preprocessor = Preprocessor(view_path)
    frames = preprocessor.load_and_preprocess_frames(max_frames=None)  # Changed to None for all frames

    if not frames:
        print("❌ Aucune frame trouvée")
        return

    print(f"   • Frames chargées: {len(frames)}")
    print(f"   • Calibration: {'✅ Activée' if calibration else '❌ Désactivée'}")

    # 3. Traitement frame par frame
    print("\n3. 🔄 TRAITEMENT DES FRAMES AVEC REID")

    for frame_idx, frame in enumerate(frames, start=1):
        print(f"\n--- Frame {frame_idx}/{len(frames)} ---")

        # Détection
        detections = detector.detect_people(frame)

        # Tracking avec ReID
        tracks = tracker.update(detections, frame)

        # Extraction trajectoires
        trajectories = trajectory_extractor.extract_trajectories(
            tracks, frame.shape[1], frame.shape[0]
        )

        # Visualisation complète
        visualizer.create_comprehensive_visualization(
            frame, detections, tracks, trajectories,
            trajectory_extractor.complete_trajectories,
            frame_idx,
            reid_features_count=len(tracker.appearance_features)
        )

        # Build interaction graph
        if trajectories:
            G = graph_builder.build_graph(trajectories)
        else:
            G = nx.Graph()

        # Compute metrics
        metrics = metrics_analyzer.compute(G, frame_idx)

        # Anomaly detection
        rule_anom, rule_reason = rule_detector.detect(metrics)
        iso_anom, iso_score = iso_detector.update_and_score(metrics, frame_idx)

        # Log metrics
        metrics_row = {
            'frame': frame_idx,
            'nodes': metrics['nodes'],
            'edges': metrics['edges'],
            'density': metrics['density'],
            'avg_degree': metrics['avg_degree'],
            'clustering': metrics['clustering'],
            'giant_component_frac': metrics['giant_component_frac'],
            'rule_anomaly': bool(rule_anom),
            'rule_reason': rule_reason or "",
            'iso_anomaly': bool(iso_anom),
            'iso_score': iso_score if iso_score is not None else ""
        }
        metrics_csv_rows.append(metrics_row)

        # Print summary
        print(f"   • Tracks: {len(tracks)}, Features ReID: {len(tracker.appearance_features)}")
        print(f"   • Métriques: densité={metrics['density']:.3f}, clustering={metrics['clustering']:.3f}")
        if rule_anom:
            print(f"   ⚠ Rule-based ANOMALY: {rule_reason}")
        if iso_anom:
            print(f"   ⚠ IsolationForest ANOMALY: score={iso_score:.4f}")

    # 4. Génération des visualisations finales
    print("\n4. 🎨 GÉNÉRATION DES VISUALISATIONS FINALES")

    # Heatmap de densité globale
    if trajectory_extractor.trajectories_data:
        visualizer.create_density_heatmap(
            trajectory_extractor.trajectories_data,
            frames[0].shape,
            "_final_reid"
        )

    # Trajets superposés
    if trajectory_extractor.complete_trajectories:
        visualizer.create_trajectory_overlay(
            trajectory_extractor.complete_trajectories,
            frames[0],
            "_final_reid"
        )

    # 5. Sauvegarde résultats
    print("\n5. 💾 SAUVEGARDE DES RÉSULTATS")
    trajectory_extractor.save_trajectories_to_csv(output_trajectories_csv)

    # Save metrics CSV
    try:
        with open(output_metrics_csv, 'w', newline='') as f:
            fieldnames = ['frame','nodes','edges','density','avg_degree','clustering','giant_component_frac','rule_anomaly','rule_reason','iso_anomaly','iso_score']
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            for r in metrics_csv_rows:
                writer.writerow(r)
        print(f"✅ Metrics CSV saved: {output_metrics_csv}")
    except Exception as e:
        print(f"❌ Error saving metrics CSV: {e}")

    print(f"\n🎉 PIPELINE AVEC REID TERMINÉ!")
    print(f"📁 Visualisations: {output_visualization_dir}")
    print(f"📊 Trajectories CSV: {output_trajectories_csv}")
    print(f"📈 Metrics CSV: {output_metrics_csv}")
    print(f"🔥 Heatmaps et trajets générés avec ReID!")

    return trajectory_extractor.trajectories_data, trajectory_extractor.complete_trajectories

# ============================================================================
# EXÉCUTION PRINCIPALE
# ============================================================================

if __name__ == "__main__":
    print("🎯 PIPELINE COMPLET - Bot-SORT + ReID Pré-entraîné + Heatmap + Trajets")
    print("=" * 60)

    # Exécution du pipeline avec ReID
    trajectories_data, complete_trajectories = run_complete_pipeline_with_reid()

    print(f"\n📁 ACCÈS RAPIDE:")
    print(f"   • Visualisations: {output_visualization_dir}")
    print(f"   • Données trajectoires: {output_trajectories_csv}")
    print(f"   • Métriques graphes: {output_metrics_csv}")
    print(f"   • Trajectoires complètes: {len(complete_trajectories)}")

🎯 PIPELINE COMPLET - Bot-SORT + ReID Pré-entraîné + Heatmap + Trajets
🎯 PIPELINE COMPLET - Bot-SORT + ReID Pré-entraîné + Heatmap + Trajets

1. 🔧 INITIALISATION AVEC REID
📷 Calibration chargée: 768x576
✅ Homographie précise calculée
✅ Calibration caméra corrigée initialisée
✅ Modèle personnalisé chargé: /content/best.pt
📋 Classes: {0: 'person'}


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


✅ Modèle ReID resnet50 chargé sur cpu
✅ Tracker Bot-SORT avec ReID pré-entraîné initialisé
✅ Extracteur de trajectoires corrigé initialisé

2. 🎬 CHARGEMENT DES FRAMES
🔄 Chargement des frames...
✅ 795 frames chargées
   • Frames chargées: 795
   • Calibration: ✅ Activée

3. 🔄 TRAITEMENT DES FRAMES AVEC REID

--- Frame 1/795 ---
🔍 Détections: 3 personnes
🆕 Track 1 créé (ReID features: 2048)
🆕 Track 2 créé (ReID features: 2048)
🆕 Track 3 créé (ReID features: 2048)
📈 Trajectoires extraites: 3


/tmp/ipython-input-1075263857.py:926: UserWarning: Glyph 127916 (\N{CLAPPER BOARD}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipython-input-1075263857.py:926: UserWarning: Glyph 128293 (\N{FIRE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipython-input-1075263857.py:926: UserWarning: Glyph 128740 (\N{RAILWAY TRACK}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipython-input-1075263857.py:926: UserWarning: Glyph 128202 (\N{BAR CHART}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipython-input-1075263857.py:928: UserWarning: Glyph 127916 (\N{CLAPPER BOARD}) missing from font(s) DejaVu Sans.
  plt.savefig(filename, dpi=150, bbox_inches='tight')
/tmp/ipython-input-1075263857.py:928: UserWarning: Glyph 128293 (\N{FIRE}) missing from font(s) DejaVu Sans.
  plt.savefig(filename, dpi=150, bbox_inches='tight')
/tmp/ipython-input-1075263857.py:928: UserWarning: Glyph 128740 (\N{RAILWAY TRACK}) missing from font(s) DejaVu Sans.
  pl

💾 Visualisation complète sauvegardée: /content/visualizations/comprehensive_frame_0001.png
   • Tracks: 3, Features ReID: 3
   • Métriques: densité=0.000, clustering=0.000

--- Frame 2/795 ---
🔍 Détections: 4 personnes
🆕 Track 4 créé (ReID features: 2048)
📈 Trajectoires extraites: 4
💾 Visualisation complète sauvegardée: /content/visualizations/comprehensive_frame_0002.png
   • Tracks: 4, Features ReID: 4
   • Métriques: densité=0.000, clustering=0.000

--- Frame 3/795 ---
🔍 Détections: 4 personnes
📈 Trajectoires extraites: 4
💾 Visualisation complète sauvegardée: /content/visualizations/comprehensive_frame_0003.png
   • Tracks: 4, Features ReID: 4
   • Métriques: densité=0.000, clustering=0.000

--- Frame 4/795 ---
🔍 Détections: 3 personnes
🆕 Track 5 créé (ReID features: 2048)
📈 Trajectoires extraites: 3
💾 Visualisation complète sauvegardée: /content/visualizations/comprehensive_frame_0004.png
   • Tracks: 3, Features ReID: 5
   • Métriques: densité=0.000, clustering=0.000

--- Frame 5/

...

💾 Visualisation complète sauvegardée: /content/visualizations/comprehensive_frame_0795.png
   • Tracks: 10, Features ReID: 16
   • Métriques: densité=0.133, clustering=0.233

4. 🎨 GÉNÉRATION DES VISUALISATIONS FINALES


/tmp/ipython-input-1075263857.py:765: UserWarning: Glyph 128293 (\N{FIRE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipython-input-1075263857.py:765: UserWarning: Glyph 128202 (\N{BAR CHART}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipython-input-1075263857.py:767: UserWarning: Glyph 128293 (\N{FIRE}) missing from font(s) DejaVu Sans.
  plt.savefig(filename, dpi=150, bbox_inches='tight')
/tmp/ipython-input-1075263857.py:767: UserWarning: Glyph 128202 (\N{BAR CHART}) missing from font(s) DejaVu Sans.
  plt.savefig(filename, dpi=150, bbox_inches='tight')


🔥 Heatmap sauvegardée: /content/visualizations/density_heatmap_final_reid.png


/tmp/ipython-input-1075263857.py:817: UserWarning: Glyph 128740 (\N{RAILWAY TRACK}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipython-input-1075263857.py:817: UserWarning: Glyph 128200 (\N{CHART WITH UPWARDS TREND}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipython-input-1075263857.py:819: UserWarning: Glyph 128740 (\N{RAILWAY TRACK}) missing from font(s) DejaVu Sans.
  plt.savefig(filename, dpi=150, bbox_inches='tight')
/tmp/ipython-input-1075263857.py:819: UserWarning: Glyph 128200 (\N{CHART WITH UPWARDS TREND}) missing from font(s) DejaVu Sans.
  plt.savefig(filename, dpi=150, bbox_inches='tight')


🛤 Trajets superposés sauvegardés: /content/visualizations/trajectory_overlay_final_reid.png

5. 💾 SAUVEGARDE DES RÉSULTATS
✅ Trajectoires corrigées sauvegardées: /content/trajectories_corrected.csv
✅ Metrics CSV saved: /content/graph_metrics.csv

🎉 PIPELINE AVEC REID TERMINÉ!
📁 Visualisations: /content/visualizations/
📊 Trajectories CSV: /content/trajectories_corrected.csv
📈 Metrics CSV: /content/graph_metrics.csv
🔥 Heatmaps et trajets générés avec ReID!

📁 ACCÈS RAPIDE:
   • Visualisations: /content/visualizations/
   • Données trajectoires: /content/trajectories_corrected.csv
   • Métriques graphes: /content/graph_metrics.csv
   • Trajectoires complètes: 163


#SMA

In [ ]:
!pip install mesa==1.2.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.7 MB/s eta 0:00:00


#SMA 1ere version

In [ ]:
!apt-get install ffmpeg -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.


In [ ]:
# ============================================================================
# GÉNÉRATION DE VIDÉO
# ============================================================================

import matplotlib.animation as animation
from PIL import Image
import io

class VideoGenerator:
    """Génère une vidéo de la simulation"""

    def __init__(self, model, steps: int = 200):
        self.model = model
        self.steps = steps
        self.frames = []

    def capture_simulation(self):
        """Capture chaque étape de la simulation pour créer la vidéo"""
        print("🎥 Capture de la simulation en cours...")

        # Exécuter la simulation et capturer chaque étape
        for step in range(self.steps):
            # Avancer d'une étape
            self.model.step()

            # Créer une image de l'état actuel
            fig = self._create_frame()

            # Convertir la figure en image
            buf = io.BytesIO()
            fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
            buf.seek(0)
            self.frames.append(Image.open(buf))

            plt.close(fig)

            if step % 20 == 0:
                active_agents = sum(1 for a in self.model.schedule.agents if not a.evacuated)
                print(f"  Frame {step}/{self.steps} - Agents actifs: {active_agents}")

            # Arrêter si tous les agents sont évacués
            if self.model.evacuated_count >= self.model.schedule.get_agent_count():
                print(f"✅ Tous les agents évacués à l'étape {step}")
                break

        print(f"✅ {len(self.frames)} frames capturées")

    def _create_frame(self):
        """Crée une frame de la simulation"""
        fig, ax = plt.subplots(figsize=(10, 8))

        # Définir les limites
        ax.set_xlim(0, self.model.width)
        ax.set_ylim(0, self.model.height)
        ax.set_aspect('equal')
        ax.set_facecolor('#f5f5f5')

        # Dessiner les obstacles
        for obstacle in self.model.obstacles:
            x, y, w, h = obstacle
            rect = patches.Rectangle((x, y), w, h,
                                   facecolor='gray', alpha=0.7,
                                   edgecolor='black', linewidth=1)
            ax.add_patch(rect)

        # Dessiner les sorties
        for exit_pos in self.model.exits:
            x, y, w = exit_pos
            rect = patches.Rectangle((x - w/2, y - 0.1), w, 0.2,
                                   facecolor='green', alpha=0.7,
                                   edgecolor='darkgreen', linewidth=2)
            ax.add_patch(rect)

        # Dessiner les agents
        for agent in self.model.schedule.agents:
            if hasattr(agent, 'evacuated') and agent.evacuated:
                continue

            x, y = agent.pos

            # Définir la couleur selon l'état
            if agent.state == AgentState.PANIC:
                color = 'red'
                alpha = 0.9
                size = 60
            elif agent.state == AgentState.EVACUATING:
                color = 'orange'
                alpha = 0.8
                size = 50
            elif agent.state == AgentState.GROUPING:
                color = 'green'
                alpha = 0.7
                size = 45
            elif agent.state == AgentState.FOLLOWING:
                color = 'purple'
                alpha = 0.7
                size = 45
            elif agent.state == AgentState.OBSTACLE_AVOIDANCE:
                color = 'brown'
                alpha = 0.7
                size = 45
            elif agent.state == AgentState.DISPERSING:
                color = 'cyan'
                alpha = 0.7
                size = 45
            else:
                color = 'blue'
                alpha = 0.6
                size = 40

            # Dessiner l'agent
            circle = plt.Circle((x, y), 0.2, color=color, alpha=alpha)
            ax.add_patch(circle)

            # Dessiner une flèche pour la direction
            if hasattr(agent, 'velocity') and np.linalg.norm(agent.velocity) > 0:
                dx, dy = agent.velocity
                length = np.linalg.norm([dx, dy])
                if length > 0:
                    # Normaliser et réduire la longueur pour la visibilité
                    dx_norm = dx / length * 0.4
                    dy_norm = dy / length * 0.4
                    ax.arrow(x, y, dx_norm, dy_norm,
                            head_width=0.1, head_length=0.15,
                            fc=color, ec=color, alpha=0.8)

        # Titre avec informations
        active_agents = sum(1 for a in self.model.schedule.agents if not a.evacuated)
        title = f"Step: {self.model.schedule.steps} | Active Agents: {active_agents} | Evacuated: {self.model.evacuated_count}"
        ax.set_title(title, fontsize=14, fontweight='bold')

        # Légende
        legend_elements = [
            patches.Patch(facecolor='blue', alpha=0.6, label='Normal'),
            patches.Patch(facecolor='red', alpha=0.9, label='Panic'),
            patches.Patch(facecolor='orange', alpha=0.8, label='Evacuating'),
            patches.Patch(facecolor='green', alpha=0.7, label='Grouping'),
            patches.Patch(facecolor='green', alpha=0.7, label='Obstacles'),
            patches.Patch(facecolor='green', alpha=0.7, label='Exits'),
        ]
        ax.legend(handles=legend_elements, loc='upper right', fontsize=8)

        # Grille
        ax.grid(True, alpha=0.3, linestyle='--')

        return fig

    def save_video(self, filename: str = "simulation_video.mp4", fps: int = 20):
        """Sauvegarde la vidéo"""
        if not self.frames:
            print("❌ Aucune frame à sauvegarder")
            return

        print(f"🎬 Génération de la vidéo {filename} ({len(self.frames)} frames, {fps} FPS)...")

        # Sauvegarder comme GIF
        gif_filename = filename.replace('.mp4', '.gif')
        self.frames[0].save(gif_filename,
                          save_all=True,
                          append_images=self.frames[1:],
                          duration=1000//fps,
                          loop=0)
        print(f"✅ GIF sauvegardé: {gif_filename}")

        # Essayer de sauvegarder comme MP4 si possible
        try:
            # Utiliser matplotlib pour créer une vidéo MP4
            fig = plt.figure(figsize=(10, 8))
            ax = fig.add_subplot(111)

            # Créer l'animation
            def update(frame_idx):
                ax.clear()
                img = self.frames[frame_idx]
                ax.imshow(img)
                ax.axis('off')
                ax.set_title(f"Frame {frame_idx+1}/{len(self.frames)}")
                return [ax]

            anim = animation.FuncAnimation(fig, update,
                                         frames=len(self.frames),
                                         interval=1000//fps,
                                         blit=False)

            # Sauvegarder
            anim.save(filename, writer='ffmpeg', fps=fps)
            plt.close(fig)
            print(f"✅ Vidéo MP4 sauvegardée: {filename}")

        except Exception as e:
            print(f"⚠ Impossible de sauvegarder en MP4: {e}")
            print(f"📁 GIF disponible: {gif_filename}")

        return gif_filename, filename if 'filename' in locals() else None

# ============================================================================
# MODIFICATION DE LA FONCTION PRINCIPALE POUR INCLURE LA VIDÉO
# ============================================================================

def run_mesa_simulation_batch():
    """Exécute la simulation MESA en mode batch (sans serveur web)"""
    print("="*80)
    print("🚀 SIMULATION MULTI-AGENTS MESA - MODE BATCH")
    print("="*80)

    # 1. Charger les données réelles
    print("\n1. 📥 CHARGEMENT DES DONNÉES DE TRACKING")
    trajectories_path = "/content/trajectories_corrected.csv"

    if os.path.exists(trajectories_path):
        print(f"📁 Fichier trouvé: {trajectories_path}")
        data_loader = RealDataLoader(trajectories_path)
    else:
        print("⚠ Fichier de tracking non trouvé, utilisation de données synthétiques")
        data_loader = RealDataLoader("")

    # 2. Sélection du scénario
    print("\n2. 🎭 SÉLECTION DU SCÉNARIO")
    print("\nScénarios disponibles:")
    print("  1. Normal (reproduction du comportement observé)")
    print("  2. Panique (niveau élevé de stress)")
    print("  3. Sorties bloquées (capacité réduite)")
    print("  4. Haute densité (plus d'agents)")
    print("  5. Évacuation dirigée (avec leaders)")

    choice = input("\nChoisissez un scénario (1-5, Enter pour 1): ").strip() or "1"

    scenarios = {
        "1": ("normal", 0.0),
        "2": ("panic", 0.5),
        "3": ("obstacle_blocked", 0.5),
        "4": ("high_density", 1.0),
        "5": ("leader_emergency", 0.8)
    }

    if choice in scenarios:
        scenario_name, intensity = scenarios[choice]
    else:
        scenario_name, intensity = "normal", 0.0

    print(f"\nScénario sélectionné: {scenario_name} (intensité: {intensity})")

    # 3. Configuration du modèle
    print("\n3. ⚙ CONFIGURATION DU MODÈLE MESA")

    model = CrowdModel(
        real_data_loader=data_loader,
        width=15.0,
        height=12.0,
        time_step=0.1,
        what_if_scenario=scenario_name,
        scenario_intensity=intensity,
        max_agents=30,  # Limiter pour la performance
        verbose=True
    )

    # 4. Demander si l'utilisateur veut générer une vidéo
    print("\n4. 🎥 GÉNÉRATION DE VIDÉO")
    generate_video = input("Voulez-vous générer une vidéo de la simulation? (o/n): ").strip().lower()

    if generate_video == 'o':
        print("🎬 Préparation de la génération vidéo...")

        # 5. Créer et exécuter le générateur de vidéo
        video_gen = VideoGenerator(model, steps=200)
        video_gen.capture_simulation()

        # 6. Sauvegarder la vidéo
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        video_filename = f"simulation_{scenario_name}_{timestamp}.mp4"
        gif_filename = f"simulation_{scenario_name}_{timestamp}.gif"

        gif_path, mp4_path = video_gen.save_video(video_filename, fps=15)

        print(f"\n✅ Vidéo générée avec succès!")
        print(f"📁 GIF: {gif_path}")
        if mp4_path:
            print(f"📁 MP4: {mp4_path}")

        # 7. Continuer avec l'analyse normale
        print("\n5. 📊 ANALYSE DES RÉSULTATS")
        results = model.get_simulation_results()
        analyzer = SimulationAnalyzer(results)
        analyzer.create_summary_report()

        # 8. Visualisation
        print("\n6. 🎨 GÉNÉRATION DES VISUALISATIONS")
        output_dir = create_batch_visualization(results, scenario_name)

        # 9. Sauvegarde des résultats
        print("\n7. 💾 SAUVEGARDE DES RÉSULTATS")
        results_dir = f"mesa_results_{scenario_name}_{timestamp}"
        os.makedirs(results_dir, exist_ok=True)

        # Sauvegarder les données
        pd.DataFrame(results['agent_data']).to_csv(f"{results_dir}/agent_results.csv", index=False)
        results['model_data'].to_csv(f"{results_dir}/model_metrics.csv")

        if results.get('agent_vars_data') is not None:
            results['agent_vars_data'].to_csv(f"{results_dir}/agent_vars.csv")

        with open(f"{results_dir}/global_metrics.json", 'w') as f:
            json.dump(results['global_metrics'], f, indent=2)

        # Copier les vidéos et visualisations
        import shutil
        if gif_path and os.path.exists(gif_path):
            shutil.copy2(gif_path, f"{results_dir}/{os.path.basename(gif_path)}")
        if mp4_path and os.path.exists(mp4_path):
            shutil.copy2(mp4_path, f"{results_dir}/{os.path.basename(mp4_path)}")

        if os.path.exists(output_dir):
            for file in os.listdir(output_dir):
                shutil.copy2(f"{output_dir}/{file}", f"{results_dir}/{file}")

        print(f"\n🎉 SIMULATION TERMINÉE!")
        print(f"📁 Résultats complets dans: {results_dir}")
        print(f"🎥 Vidéos dans: {results_dir}")
        print(f"📈 Visualisations dans: {output_dir}")

        return results, gif_path, mp4_path

    else:
        # Mode sans vidéo (original)
        # 5. Exécution de la simulation
        print("\n5. 🔄 EXÉCUTION DE LA SIMULATION")
        results = model.run_simulation(steps=300)

        # 6. Analyse des résultats
        print("\n6. 📊 ANALYSE DES RÉSULTATS")
        analyzer = SimulationAnalyzer(results)
        analyzer.create_summary_report()

        # 7. Visualisation
        print("\n7. 🎨 GÉNÉRATION DES VISUALISATIONS")
        output_dir = create_batch_visualization(results, scenario_name)

        # 8. Sauvegarde des résultats
        print("\n8. 💾 SAUVEGARDE DES RÉSULTATS")
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        results_dir = f"mesa_results_{scenario_name}_{timestamp}"
        os.makedirs(results_dir, exist_ok=True)

        # Sauvegarder les données
        pd.DataFrame(results['agent_data']).to_csv(f"{results_dir}/agent_results.csv", index=False)
        results['model_data'].to_csv(f"{results_dir}/model_metrics.csv")

        if results.get('agent_vars_data') is not None:
            results['agent_vars_data'].to_csv(f"{results_dir}/agent_vars.csv")

        with open(f"{results_dir}/global_metrics.json", 'w') as f:
            json.dump(results['global_metrics'], f, indent=2)

        # Copier les visualisations
        import shutil
        if os.path.exists(output_dir):
            for file in os.listdir(output_dir):
                shutil.copy2(f"{output_dir}/{file}", f"{results_dir}/{file}")

        print(f"\n🎉 SIMULATION TERMINÉE!")
        print(f"📁 Résultats complets dans: {results_dir}")
        print(f"📈 Visualisations dans: {output_dir}")

        return results

# ============================================================================
# FONCTION SIMPLIFIÉE POUR GÉNÉRER UNE VIDÉO RAPIDE
# ============================================================================

def generate_simulation_video():
    """Fonction simplifiée pour générer une vidéo de simulation"""
    print("🎬 GÉNÉRATEUR DE VIDÉO DE SIMULATION")
    print("="*50)

    # Configuration simple
    trajectories_path = "/content/trajectories_corrected.csv"

    if os.path.exists(trajectories_path):
        print(f"📁 Chargement des données: {trajectories_path}")
        data_loader = RealDataLoader(trajectories_path)
    else:
        print("⚠ Données synthétiques")
        data_loader = RealDataLoader("")

    # Créer le modèle
    model = CrowdModel(
        real_data_loader=data_loader,
        width=15.0,
        height=12.0,
        time_step=0.1,
        what_if_scenario="normal",
        scenario_intensity=0.0,
        max_agents=20,
        verbose=True
    )

    # Demander les paramètres
    steps = input("\nNombre de frames (défaut: 150): ").strip()
    steps = int(steps) if steps else 150

    fps = input("Images par seconde (défaut: 20): ").strip()
    fps = int(fps) if fps else 20

    scenario = input("\nScénario (normal/panic/obstacle_blocked/high_density/leader_emergency, défaut: normal): ").strip()
    scenario = scenario if scenario else "normal"

    # Appliquer le scénario
    model.what_if_scenario = scenario

    # Générer la vidéo
    print(f"\n🎥 Génération de {steps} frames à {fps} FPS...")

    video_gen = VideoGenerator(model, steps=steps)
    video_gen.capture_simulation()

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    video_filename = f"simulation_{scenario}_{timestamp}.mp4"

    gif_path, mp4_path = video_gen.save_video(video_filename, fps=fps)

    print(f"\n✅ Vidéo générée avec succès!")
    if gif_path:
        print(f"📁 GIF: {gif_path}")
    if mp4_path:
        print(f"📁 MP4: {mp4_path}")

    return gif_path, mp4_path

# ============================================================================
# MODIFICATION DU MENU PRINCIPAL
# ============================================================================

def main():
    """Interface principale simplifiée"""
    print("🎯 SIMULATION MULTI-AGENTS MESA - Tracking de Foule")
    print("   Basé sur les données réelles de PETS2009")
    print("\n⚠ NOTE: Le serveur web interactif peut ne pas fonctionner")
    print("   dans certains environnements comme Google Colab.")
    print("   Utilisez le mode batch pour des résultats fiables.")

    while True:
        print("\n" + "="*50)
        print("MENU PRINCIPAL")
        print("="*50)
        print("1. Exécuter une simulation simple (mode batch)")
        print("2. Comparer plusieurs scénarios")
        print("3. Générer une vidéo de simulation")
        print("4. Quitter")

        choice = input("\nChoisissez une option (1-4): ").strip()

        if choice == "1":
            run_mesa_simulation_batch()
        elif choice == "2":
            compare_scenarios_batch()
        elif choice == "3":
            generate_simulation_video()
        elif choice == "4":
            print("\n👋 Au revoir!")
            break
        else:
            print("❌ Choix invalide, veuillez réessayer.")

# ============================================================================
# AJOUTER L'IMPORT DES DÉPENDANCES VIDÉO
# ============================================================================

# Ajouter ces imports au début du fichier si ce n'est pas déjà fait
"""
import matplotlib.animation as animation
from PIL import Image
import io
"""

'\nimport matplotlib.animation as animation\nfrom PIL import Image\nimport io\n'

In [ ]:
# mesa_crowd_simulation_v1_2_0.py
import mesa
import mesa.space
import mesa.time
import numpy as np
import pandas as pd
from enum import Enum
import random
from typing import List, Dict, Optional, Tuple
import networkx as nx
from dataclasses import dataclass
import json
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.animation as animation
from PIL import Image
import io
import os
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# ÉTATS ET TYPES D'AGENTS
# ============================================================================

class AgentState(Enum):
    NORMAL = "normal"
    GROUPING = "grouping"
    DISPERSING = "dispersing"
    PANIC = "panic"
    EVACUATING = "evacuating"
    OBSTACLE_AVOIDANCE = "obstacle_avoidance"
    FOLLOWING = "following"

class AgentType(Enum):
    PEDESTRIAN = "pedestrian"
    LEADER = "leader"
    FOLLOWER = "follower"
    INDEPENDENT = "independent"

# ============================================================================
# DONNÉES RÉELLES
# ============================================================================

class RealDataLoader:
    """Charge et traite les données réelles de tracking"""

    def __init__(self, trajectories_csv: str):
        self.trajectories_csv = trajectories_csv
        self.trajectories_data = None
        self.agent_parameters = {}

    def load_real_trajectories(self):
        """Charge les trajectoires depuis le CSV de tracking"""
        try:
            df = pd.read_csv(self.trajectories_csv)
            print(f"📊 Données réelles chargées: {len(df)} points de trajectoire")

            # Grouper par ID d'agent
            trajectories_by_id = {}
            for _, row in df.iterrows():
                agent_id = int(row['id'])
                if agent_id not in trajectories_by_id:
                    trajectories_by_id[agent_id] = []

                trajectories_by_id[agent_id].append({
                    'frame': int(row['frame']),
                    'world_x': float(row['world_foot_x']),
                    'world_y': float(row['world_foot_y']),
                    'pixel_x': float(row['pixel_foot_x']),
                    'pixel_y': float(row['pixel_foot_y']),
                    'score': float(row['score'])
                })

            # Trier chaque trajectoire par frame
            for agent_id in trajectories_by_id:
                trajectories_by_id[agent_id].sort(key=lambda x: x['frame'])

            self.trajectories_data = trajectories_by_id
            return trajectories_by_id

        except Exception as e:
            print(f"❌ Erreur chargement données: {e}")
            # Créer des données synthétiques pour le test
            return self._create_synthetic_data()

    def _create_synthetic_data(self):
        """Crée des données synthétiques pour le test"""
        print("🔄 Création de données synthétiques pour le test...")
        trajectories_by_id = {}

        for agent_id in range(20):
            trajectories_by_id[agent_id] = []
            x_start = random.uniform(2, 13)
            y_start = random.uniform(2, 10)

            for frame in range(50):
                x = x_start + frame * 0.1
                y = y_start + random.uniform(-0.2, 0.2)

                trajectories_by_id[agent_id].append({
                    'frame': frame,
                    'world_x': x,
                    'world_y': y,
                    'pixel_x': x * 50,
                    'pixel_y': y * 50,
                    'score': random.uniform(0.7, 0.9)
                })

        self.trajectories_data = trajectories_by_id
        return trajectories_by_id

    def extract_agent_parameters(self, agent_id: int, trajectory: List[Dict]):
        """Extrait les paramètres comportementaux d'un agent réel"""
        if len(trajectory) < 5:
            return None

        # Positions et vitesses
        positions = np.array([[p['world_x'], p['world_y']] for p in trajectory])
        frames = np.array([p['frame'] for p in trajectory])

        # Calcul des vitesses (en mètres par pas de temps)
        if len(positions) > 1:
            displacements = np.diff(positions, axis=0)
            time_diff = np.diff(frames) / 10.0  # 10 fps approximatif
            velocities = displacements / time_diff[:, np.newaxis]

            avg_velocity = np.mean(np.linalg.norm(velocities, axis=1))
            max_velocity = np.max(np.linalg.norm(velocities, axis=1))
            avg_direction = np.mean(velocities / (np.linalg.norm(velocities, axis=1, keepdims=True) + 1e-6), axis=0)
        else:
            avg_velocity = 1.0
            max_velocity = 1.5
            avg_direction = np.array([1.0, 0.0])

        # Analyse de la trajectoire
        total_distance = np.sum(np.linalg.norm(np.diff(positions, axis=0), axis=1))
        straightness = np.linalg.norm(positions[-1] - positions[0]) / (total_distance + 1e-6)

        # Variabilité de direction
        if len(positions) > 2:
            directions = np.diff(positions, axis=0)
            direction_changes = np.arccos(np.clip(
                np.sum(directions[1:] * directions[:-1], axis=1) /
                (np.linalg.norm(directions[1:], axis=1) * np.linalg.norm(directions[:-1], axis=1) + 1e-6),
                -1.0, 1.0
            ))
            direction_variability = np.mean(direction_changes)
        else:
            direction_variability = 0.0

        # Déterminer le type d'agent basé sur le comportement
        if straightness > 0.8 and direction_variability < 0.5:
            agent_type = AgentType.LEADER
            sociability = random.uniform(0.7, 0.9)
            following_tendency = 0.1
        elif direction_variability > 1.0 and straightness < 0.4:
            agent_type = AgentType.INDEPENDENT
            sociability = random.uniform(0.2, 0.4)
            following_tendency = random.uniform(0.2, 0.4)
        elif avg_velocity > 1.2 and len(trajectory) > 20:
            agent_type = AgentType.FOLLOWER
            sociability = random.uniform(0.5, 0.7)
            following_tendency = random.uniform(0.6, 0.9)
        else:
            agent_type = AgentType.PEDESTRIAN
            sociability = random.uniform(0.3, 0.6)
            following_tendency = random.uniform(0.3, 0.5)

        # Position initiale (première position)
        initial_position = positions[0]

        return {
            'agent_id': agent_id,
            'initial_position': initial_position,
            'avg_velocity': avg_velocity,
            'max_velocity': max_velocity,
            'avg_direction': avg_direction,
            'straightness': straightness,
            'direction_variability': direction_variability,
            'agent_type': agent_type,
            'sociability': sociability,
            'following_tendency': following_tendency,
            'patience': random.uniform(0.4, 0.8),
            'panic_level': random.uniform(0.1, 0.3),
            'awareness_radius': random.uniform(2.0, 4.0),
            'reaction_time': random.uniform(0.2, 0.4)
        }

# ============================================================================
# GÉNÉRATION DE VIDÉO
# ============================================================================

class VideoGenerator:
    """Génère une vidéo de la simulation"""

    def __init__(self, model, steps: int = 200):
        self.model = model
        self.steps = steps
        self.frames = []

    def capture_simulation(self):
        """Capture chaque étape de la simulation pour créer la vidéo"""
        print("🎥 Capture de la simulation en cours...")

        # Sauvegarder l'état initial
        initial_state = self.model.schedule.steps
        initial_evacuated = self.model.evacuated_count

        # Exécuter la simulation et capturer chaque étape
        for step in range(self.steps):
            # Avancer d'une étape
            self.model.step()

            # Créer une image de l'état actuel
            fig = self._create_frame()

            # Convertir la figure en image
            buf = io.BytesIO()
            fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
            buf.seek(0)
            self.frames.append(Image.open(buf))

            plt.close(fig)

            if step % 20 == 0:
                active_agents = sum(1 for a in self.model.schedule.agents if not a.evacuated)
                print(f"  Frame {step}/{self.steps} - Agents actifs: {active_agents}")

            # Arrêter si tous les agents sont évacués
            if self.model.evacuated_count >= self.model.schedule.get_agent_count():
                print(f"✅ Tous les agents évacués à l'étape {step}")
                break

        print(f"✅ {len(self.frames)} frames capturées")

    def _create_frame(self):
        """Crée une frame de la simulation"""
        fig, ax = plt.subplots(figsize=(12, 9))

        # Définir les limites
        ax.set_xlim(0, self.model.width)
        ax.set_ylim(0, self.model.height)
        ax.set_aspect('equal')
        ax.set_facecolor('#f5f5f5')
        ax.grid(True, alpha=0.3, linestyle='--')

        # Dessiner les obstacles
        for obstacle in self.model.obstacles:
            x, y, w, h = obstacle
            rect = patches.Rectangle((x, y), w, h,
                                   facecolor='#2c3e50', alpha=0.8,
                                   edgecolor='#34495e', linewidth=2)
            ax.add_patch(rect)

        # Dessiner les sorties
        for exit_pos in self.model.exits:
            x, y, w = exit_pos
            rect = patches.Rectangle((x - w/2, y - 0.1), w, 0.2,
                                   facecolor='#27ae60', alpha=0.8,
                                   edgecolor='#229954', linewidth=2)
            ax.add_patch(rect)

        # Dessiner les trajectoires (dernières 20 positions)
        for agent in self.model.schedule.agents:
            if hasattr(agent, 'evacuated') and agent.evacuated:
                continue

            if hasattr(agent, 'trajectory') and len(agent.trajectory) > 1:
                # Prendre les 20 dernières positions
                recent_traj = agent.trajectory[-20:]
                x_vals = [pos[0] for pos in recent_traj]
                y_vals = [pos[1] for pos in recent_traj]
                ax.plot(x_vals, y_vals, color='gray', alpha=0.3, linewidth=1)

        # Dessiner les agents
        for agent in self.model.schedule.agents:
            if hasattr(agent, 'evacuated') and agent.evacuated:
                continue

            x, y = agent.pos

            # Définir la couleur selon l'état
            if agent.state == AgentState.PANIC:
                color = '#e74c3c'  # Rouge
                alpha = 0.9
                size = 80
                edgecolor = '#c0392b'
                edgewidth = 2
            elif agent.state == AgentState.EVACUATING:
                color = '#e67e22'  # Orange
                alpha = 0.8
                size = 70
                edgecolor = '#d35400'
                edgewidth = 2
            elif agent.state == AgentState.GROUPING:
                color = '#2ecc71'  # Vert
                alpha = 0.8
                size = 65
                edgecolor = '#27ae60'
                edgewidth = 1
            elif agent.state == AgentState.FOLLOWING:
                color = '#9b59b6'  # Violet
                alpha = 0.8
                size = 65
                edgecolor = '#8e44ad'
                edgewidth = 1
            elif agent.state == AgentState.OBSTACLE_AVOIDANCE:
                color = '#795548'  # Marron
                alpha = 0.8
                size = 65
                edgecolor = '#5d4037'
                edgewidth = 1
            elif agent.state == AgentState.DISPERSING:
                color = '#00bcd4'  # Cyan
                alpha = 0.8
                size = 65
                edgecolor = '#0097a7'
                edgewidth = 1
            else:
                color = '#3498db'  # Bleu
                alpha = 0.7
                size = 60
                edgecolor = '#2980b9'
                edgewidth = 1

            # Dessiner l'agent
            circle = plt.Circle((x, y), 0.25, color=color, alpha=alpha,
                               edgecolor=edgecolor, linewidth=edgewidth)
            ax.add_patch(circle)

            # Ajouter un petit point au centre
            ax.plot(x, y, 'o', color='white', markersize=2, alpha=0.8)

            # Dessiner une flèche pour la direction
            if hasattr(agent, 'velocity') and np.linalg.norm(agent.velocity) > 0:
                dx, dy = agent.velocity
                length = np.linalg.norm([dx, dy])
                if length > 0:
                    # Normaliser et réduire la longueur pour la visibilité
                    scale = 0.5
                    dx_norm = dx / length * scale
                    dy_norm = dy / length * scale
                    ax.arrow(x, y, dx_norm, dy_norm,
                            head_width=0.15, head_length=0.2,
                            fc=edgecolor, ec=edgecolor, alpha=0.9, linewidth=1.5)

        # Titre avec informations
        active_agents = sum(1 for a in self.model.schedule.agents if not a.evacuated)
        total_agents = self.model.schedule.get_agent_count()

        # Calculer le temps écoulé (approximatif)
        time_elapsed = self.model.schedule.steps * self.model.time_step

        title = f"Simulation de Foule - Étape: {self.model.schedule.steps} | "
        title += f"Temps: {time_elapsed:.1f}s | "
        title += f"Actifs: {active_agents}/{total_agents} | "
        title += f"Évacués: {self.model.evacuated_count}"

        ax.set_title(title, fontsize=14, fontweight='bold', pad=20)

        # Labels des axes
        ax.set_xlabel('Position X (mètres)', fontsize=11)
        ax.set_ylabel('Position Y (mètres)', fontsize=11)

        # Légende
        legend_elements = [
            patches.Patch(facecolor='#3498db', alpha=0.7, label='Normal'),
            patches.Patch(facecolor='#e74c3c', alpha=0.9, label='Panique'),
            patches.Patch(facecolor='#e67e22', alpha=0.8, label='Évacuation'),
            patches.Patch(facecolor='#2ecc71', alpha=0.8, label='Regroupement'),
            patches.Patch(facecolor='#9b59b6', alpha=0.8, label='Suivi'),
            patches.Patch(facecolor='#2c3e50', alpha=0.8, label='Obstacles'),
            patches.Patch(facecolor='#27ae60', alpha=0.8, label='Sorties'),
        ]

        ax.legend(handles=legend_elements, loc='upper right', fontsize=9,
                 framealpha=0.9, edgecolor='black')

        # Ajouter un texte avec les statistiques
        stats_text = f"Scénario: {self.model.what_if_scenario}\n"
        stats_text += f"Intensité: {self.model.scenario_intensity:.1f}\n"

        if hasattr(self.model, 'global_metrics') and len(self.model.global_metrics) > 0:
            last_metrics = self.model.global_metrics[-1]
            if 'average_speed' in last_metrics:
                stats_text += f"Vitesse moyenne: {last_metrics['average_speed']:.2f} m/s\n"
            if 'average_stress' in last_metrics:
                stats_text += f"Stress moyen: {last_metrics['average_stress']:.3f}"

        ax.text(0.02, 0.98, stats_text, transform=ax.transAxes,
               fontsize=9, verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='gray'))

        plt.tight_layout()

        return fig

    def save_video(self, filename: str = "simulation_video.mp4", fps: int = 20):
        """Sauvegarde la vidéo"""
        if not self.frames:
            print("❌ Aucune frame à sauvegarder")
            return None, None

        print(f"🎬 Génération de la vidéo {filename} ({len(self.frames)} frames, {fps} FPS)...")

        # Sauvegarder comme GIF
        gif_filename = filename.replace('.mp4', '.gif')
        try:
            self.frames[0].save(gif_filename,
                              save_all=True,
                              append_images=self.frames[1:],
                              duration=1000//fps,
                              loop=0,
                              optimize=True,
                              quality=95)
            print(f"✅ GIF sauvegardé: {gif_filename}")
        except Exception as e:
            print(f"❌ Erreur lors de la sauvegarde du GIF: {e}")
            gif_filename = None

        # Essayer de sauvegarder comme MP4 si possible
        mp4_filename = None
        try:
            # Créer une figure pour l'animation
            fig, ax = plt.subplots(figsize=(12, 9))
            ax.axis('off')

            # Fonction d'initialisation
            def init():
                ax.imshow(self.frames[0])
                return [ax]

            # Fonction de mise à jour
            def update(frame_idx):
                ax.clear()
                ax.axis('off')
                img = self.frames[frame_idx]
                ax.imshow(img)

                # Ajouter le numéro de frame
                ax.text(0.02, 0.02, f"Frame: {frame_idx+1}/{len(self.frames)}",
                       transform=ax.transAxes, fontsize=12,
                       bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

                return [ax]

            # Créer l'animation
            anim = animation.FuncAnimation(fig, update,
                                         init_func=init,
                                         frames=len(self.frames),
                                         interval=1000//fps,
                                         blit=False)

            # Sauvegarder en MP4
            mp4_filename = filename
            anim.save(mp4_filename, writer='ffmpeg', fps=fps,
                     extra_args=['-vcodec', 'libx264', '-pix_fmt', 'yuv420p'])
            plt.close(fig)
            print(f"✅ Vidéo MP4 sauvegardée: {mp4_filename}")

        except Exception as e:
            print(f"⚠ Impossible de sauvegarder en MP4: {e}")
            print("ℹ Assurez-vous que ffmpeg est installé:")
            print("  - Sur Ubuntu/Debian: sudo apt-get install ffmpeg")
            print("  - Sur Colab: !apt-get install ffmpeg -y")

        return gif_filename, mp4_filename

# ============================================================================
# AGENT MESA
# ============================================================================

class PersonAgent(mesa.Agent):
    """Agent représentant une personne avec comportement réaliste"""

    def __init__(self, unique_id: int, model, parameters: Dict):
        super().__init__(unique_id, model)

        # Paramètres individuels
        self.agent_type = parameters['agent_type']
        self.sociability = parameters['sociability']
        self.following_tendency = parameters['following_tendency']
        self.patience = parameters['patience']
        self.panic_level = parameters['panic_level']
        self.awareness_radius = parameters['awareness_radius']
        self.reaction_time = parameters['reaction_time']
        self.desired_speed = parameters['avg_velocity']
        self.max_speed = parameters['max_velocity']

        # État dynamique
        self.state = AgentState.NORMAL
        self.stress = 0.0
        self.fatigue = 0.0
        self.target_exit = None
        self.leader = None
        self.followers = []

        # Historique
        self.trajectory = []
        self.velocity_history = []
        self.state_history = []

        # Statistiques
        self.distance_traveled = 0.0
        self.collisions_count = 0
        self.state_changes = 0
        self.evacuated = False

        # Initialiser la position
        initial_pos = parameters['initial_position']
        self.model.space.place_agent(self, initial_pos)
        self.trajectory.append(initial_pos)

        # Initialiser la vitesse
        initial_velocity = parameters['avg_direction'] * self.desired_speed
        self.velocity = initial_velocity
        self.velocity_history.append(initial_velocity)

        print(f"👤 Agent {unique_id} créé ({self.agent_type.value}) à {initial_pos}")

    def step(self):
        """Étape de simulation pour l'agent"""
        if self.evacuated:
            return

        # Mettre à jour l'état
        self._update_state()

        # Calculer les forces sociales
        social_force = self._calculate_social_force()
        obstacle_force = self._calculate_obstacle_force()
        exit_force = self._calculate_exit_force()

        # Combiner les forces
        total_force = (
            1.5 * social_force +
            2.0 * obstacle_force +
            1.0 * exit_force
        )

        # Mettre à jour le mouvement
        self._update_movement(total_force)

        # Mettre à jour l'historique
        self.trajectory.append(self.pos)
        self.velocity_history.append(self.velocity)
        self.state_history.append(self.state)

        # Vérifier les sorties
        self._check_exit()

    def _update_state(self):
        """Met à jour l'état comportemental de l'agent"""
        # Récupérer les voisins
        neighbors = self.model.space.get_neighbors(
            self.pos,
            self.awareness_radius,
            include_center=False
        )

        # Calculer la densité locale
        local_density = len(neighbors) / (np.pi * self.awareness_radius**2)

        # Mettre à jour le stress
        density_stress = min(1.0, local_density / 2.0)
        self.stress = (
            0.6 * density_stress +
            0.3 * self.panic_level +
            0.1 * self.fatigue
        )

        # Déterminer le nouvel état
        old_state = self.state

        if self.stress > 0.7:
            self.state = AgentState.PANIC
            self.panic_level = min(1.0, self.panic_level + 0.05)
        elif local_density > 1.5 and self.sociability > 0.6:
            self.state = AgentState.GROUPING
        elif self.target_exit is not None:
            self.state = AgentState.EVACUATING
        elif local_density < 0.3:
            self.state = AgentState.DISPERSING
        else:
            self.state = AgentState.NORMAL

        if old_state != self.state:
            self.state_changes += 1
            self.model.state_changes.append({
                'step': self.model.schedule.steps,
                'agent_id': self.unique_id,
                'old_state': old_state.value,
                'new_state': self.state.value
            })

    def _calculate_social_force(self) -> np.ndarray:
        """Calcule la force sociale (modèle de Helbing)"""
        total_force = np.zeros(2)
        neighbors = self.model.space.get_neighbors(
            self.pos,
            self.awareness_radius,
            include_center=False
        )

        for neighbor in neighbors:
            if hasattr(neighbor, 'evacuated') and neighbor.evacuated:
                continue

            # Vecteur entre les agents
            diff = np.array(self.pos) - np.array(neighbor.pos)
            distance = np.linalg.norm(diff)

            if distance > 0:
                direction = diff / distance

                # Force de répulsion sociale
                A = 2.0  # Amplitude
                B = 0.3  # Portée
                social_force = A * np.exp(-distance / B) * direction

                # Force de compression physique
                if distance < 0.5:  # Distance de contact
                    compression = max(0, 0.5 - distance)
                    body_force = 10.0 * compression * direction
                    social_force += body_force
                    self.collisions_count += 0.1

                total_force += social_force

        return total_force

    def _calculate_obstacle_force(self) -> np.ndarray:
        """Calcule la force de répulsion des obstacles"""
        total_force = np.zeros(2)

        for obstacle in self.model.obstacles:
            # Trouver le point le plus proche sur l'obstacle
            ox, oy, ow, oh = obstacle

            closest_x = max(ox, min(self.pos[0], ox + ow))
            closest_y = max(oy, min(self.pos[1], oy + oh))

            diff = np.array([self.pos[0] - closest_x, self.pos[1] - closest_y])
            distance = np.linalg.norm(diff)

            if distance > 0:
                direction = diff / distance
                # Force de répulsion (inverse carré)
                force_strength = min(5.0, 1.0 / (distance**2 + 0.1))
                total_force += force_strength * direction

        return total_force

    def _calculate_exit_force(self) -> np.ndarray:
        """Calcule la force d'attraction vers les sorties"""
        if self.target_exit is None:
            # Trouver la sortie la plus proche
            exits = self.model.exits
            if not exits:
                return np.zeros(2)

            min_dist = float('inf')
            for exit in exits:
                dist = np.linalg.norm(np.array(self.pos) - np.array([exit[0], exit[1]]))
                if dist < min_dist:
                    min_dist = dist
                    self.target_exit = exit

        if self.target_exit:
            exit_pos = np.array([self.target_exit[0], self.target_exit[1]])
            direction = exit_pos - np.array(self.pos)
            distance = np.linalg.norm(direction)

            if distance > 0:
                # Force plus forte quand plus proche
                strength = min(2.0, 5.0 / (distance + 1.0))
                return strength * (direction / distance)

        return np.zeros(2)

    def _update_movement(self, total_force: np.ndarray):
        """Met à jour la position et la vitesse de l'agent"""
        # Direction souhaitée
        if self.state == AgentState.PANIC:
            # Direction aléatoire dans la panique
            angle = random.uniform(0, 2*np.pi)
            desired_direction = np.array([np.cos(angle), np.sin(angle)])
        elif self.state == AgentState.EVACUATING and self.target_exit:
            # Vers la sortie
            exit_pos = np.array([self.target_exit[0], self.target_exit[1]])
            direction = exit_pos - np.array(self.pos)
            if np.linalg.norm(direction) > 0:
                desired_direction = direction / np.linalg.norm(direction)
        else:
            # Direction basée sur la force sociale
            if np.linalg.norm(total_force) > 0:
                desired_direction = total_force / np.linalg.norm(total_force)
            else:
                if np.linalg.norm(self.velocity) > 0:
                    desired_direction = self.velocity / np.linalg.norm(self.velocity)
                else:
                    desired_direction = np.array([1.0, 0.0])

        # Vitesse souhaitée
        desired_speed = self.desired_speed
        if self.state == AgentState.PANIC:
            desired_speed *= 1.5
        elif self.state == AgentState.EVACUATING:
            desired_speed *= 1.2

        # Équation d'évolution
        desired_velocity = desired_speed * desired_direction
        acceleration = (
            (desired_velocity - self.velocity) / self.reaction_time +
            total_force
        )

        # Limiter l'accélération
        max_acceleration = 5.0
        acc_norm = np.linalg.norm(acceleration)
        if acc_norm > max_acceleration:
            acceleration = acceleration / acc_norm * max_acceleration

        # Mettre à jour la vitesse
        self.velocity += acceleration * self.model.time_step

        # Limiter la vitesse
        max_speed = self.max_speed * (1.0 + 0.5 * self.panic_level)
        speed = np.linalg.norm(self.velocity)
        if speed > max_speed:
            self.velocity = self.velocity / speed * max_speed

        # Nouvelle position
        new_pos = np.array(self.pos) + self.velocity * self.model.time_step

        # Vérifier les collisions avec les obstacles
        collision = self._check_obstacle_collision(new_pos)
        if collision:
            # Rebondir
            self.velocity *= -0.5
            self.collisions_count += 1
        else:
            # Mettre à jour la position
            self.model.space.move_agent(self, tuple(new_pos))
            self.distance_traveled += speed * self.model.time_step

    def _check_obstacle_collision(self, position: np.ndarray) -> bool:
        """Vérifie si une position entre en collision avec un obstacle"""
        for obstacle in self.model.obstacles:
            ox, oy, ow, oh = obstacle
            if (ox <= position[0] <= ox + ow and
                oy <= position[1] <= oy + oh):
                return True
        return False

    def _check_exit(self):
        """Vérifie si l'agent a atteint une sortie"""
        if self.target_exit is None:
            return

        exit_pos = np.array([self.target_exit[0], self.target_exit[1]])
        distance = np.linalg.norm(np.array(self.pos) - exit_pos)

        if distance < 0.5:  # À portée de la sortie
            self.evacuated = True
            self.model.evacuated_count += 1
            if self.model.verbose:
                print(f"🚪 Agent {self.unique_id} évacué!")

# ============================================================================
# MODÈLE MESA
# ============================================================================

class CrowdModel(mesa.Model):
    """Modèle de foule multi-agents avec MESA"""

    def __init__(self,
                 real_data_loader: RealDataLoader,
                 width: float = 15.0,
                 height: float = 12.0,
                 time_step: float = 0.1,
                 what_if_scenario: str = "normal",
                 scenario_intensity: float = 0.5,
                 max_agents: int = 50,
                 verbose: bool = True):

        super().__init__()

        # Paramètres de simulation
        self.width = width
        self.height = height
        self.time_step = time_step
        self.what_if_scenario = what_if_scenario
        self.scenario_intensity = scenario_intensity
        self.max_agents = max_agents
        self.verbose = verbose

        # Espace continu
        self.space = mesa.space.ContinuousSpace(width, height, True)

        # Planificateur (RandomActivation)
        self.schedule = mesa.time.RandomActivation(self)

        # Obstacles (x, y, largeur, hauteur)
        self.obstacles = [
            (0, 0, 15, 0.2),      # Mur bas
            (0, 11.8, 15, 0.2),   # Mur haut
            (0, 0, 0.2, 12),      # Mur gauche
            (14.8, 0, 0.2, 12),   # Mur droit
            (5, 4, 2, 1),         # Obstacle central
            (10, 7, 1, 3)         # Colonne
        ]

        # Sorties (x, y, largeur)
        self.exits = [
            (7.5, 0, 1.5),    # Sortie bas
            (7.5, 12, 1.5),   # Sortie haut
            (0, 6, 1.0),      # Sortie gauche
            (15, 6, 1.0)      # Sortie droite
        ]

        # Métriques
        self.evacuated_count = 0
        self.state_changes = []
        self.global_metrics = []

        # Charger les données réelles
        self.real_data_loader = real_data_loader
        trajectories = real_data_loader.load_real_trajectories()

        # Créer les agents à partir des données réelles
        self._create_agents_from_real_data(trajectories)

        # Appliquer le scénario what-if
        self._apply_what_if_scenario()

        # Collecteur de données
        self.datacollector = mesa.DataCollector(
            model_reporters={
                "Active Agents": lambda m: sum(1 for a in m.schedule.agents if not a.evacuated),
                "Evacuated Agents": "evacuated_count",
                "Average Speed": lambda m: np.mean([
                    np.linalg.norm(a.velocity) for a in m.schedule.agents if not a.evacuated
                ]) if any(not a.evacuated for a in m.schedule.agents) else 0,
                "Average Stress": lambda m: np.mean([
                    a.stress for a in m.schedule.agents if not a.evacuated
                ]) if any(not a.evacuated for a in m.schedule.agents) else 0,
                "Average Panic": lambda m: np.mean([
                    a.panic_level for a in m.schedule.agents if not a.evacuated
                ]) if any(not a.evacuated for a in m.schedule.agents) else 0,
            },
            agent_reporters={
                "State": lambda a: a.state.value,
                "Panic Level": "panic_level",
                "Speed": lambda a: np.linalg.norm(a.velocity),
                "Distance Traveled": "distance_traveled"
            }
        )

        if self.verbose:
            print(f"✅ Modèle MESA initialisé avec {self.schedule.get_agent_count()} agents")
            print(f"🎭 Scénario: {what_if_scenario} (intensité: {scenario_intensity})")

    def _create_agents_from_real_data(self, trajectories: Dict):
        """Crée des agents à partir des trajectoires réelles"""
        agent_id = 0

        for real_agent_id, traj in trajectories.items():
            # Extraire les paramètres de l'agent réel
            params = self.real_data_loader.extract_agent_parameters(real_agent_id, traj)

            if params is None:
                continue

            # Créer l'agent MESA
            agent = PersonAgent(agent_id, self, params)
            self.schedule.add(agent)
            self.space.place_agent(agent, tuple(params['initial_position']))

            agent_id += 1

            # Limiter le nombre d'agents pour la performance
            if agent_id >= self.max_agents:
                if self.verbose:
                    print(f"⚠ Limite d'agents atteinte: {self.max_agents}")
                break

        if self.verbose:
            print(f"👥 {agent_id} agents créés à partir de données réelles")

    def _apply_what_if_scenario(self):
        """Applique un scénario what-if"""
        if self.what_if_scenario == "panic":
            self._apply_panic_scenario()
        elif self.what_if_scenario == "obstacle_blocked":
            self._apply_obstacle_blocked_scenario()
        elif self.what_if_scenario == "high_density":
            self._apply_high_density_scenario()
        elif self.what_if_scenario == "leader_emergency":
            self._apply_leader_emergency_scenario()

    def _apply_panic_scenario(self):
        """Scénario de panique"""
        for agent in self.schedule.agents:
            agent.panic_level = min(1.0, agent.panic_level + self.scenario_intensity)
            if self.scenario_intensity > 0.7:
                agent.state = AgentState.PANIC

    def _apply_obstacle_blocked_scenario(self):
        """Scénario de sorties bloquées"""
        # Bloquer certaines sorties
        exit_count = len(self.exits)
        blocked_count = int(exit_count * self.scenario_intensity)
        self.exits = self.exits[blocked_count:]

    def _apply_high_density_scenario(self):
        """Scénario de haute densité"""
        # Ajouter des agents synthétiques
        current_count = self.schedule.get_agent_count()
        new_agents = int(current_count * self.scenario_intensity)

        for i in range(new_agents):
            agent_id = current_count + i

            # Position aléatoire
            pos = (random.uniform(1, 14), random.uniform(1, 11))

            # Paramètres synthétiques
            params = {
                'agent_id': agent_id,
                'initial_position': pos,
                'avg_velocity': random.uniform(0.8, 1.2),
                'max_velocity': random.uniform(1.2, 1.8),
                'avg_direction': np.array([random.uniform(-1, 1), random.uniform(-1, 1)]),
                'straightness': random.uniform(0.3, 0.7),
                'direction_variability': random.uniform(0.5, 1.5),
                'agent_type': AgentType.PEDESTRIAN,
                'sociability': random.uniform(0.3, 0.6),
                'following_tendency': random.uniform(0.3, 0.5),
                'patience': random.uniform(0.4, 0.8),
                'panic_level': random.uniform(0.1, 0.3),
                'awareness_radius': random.uniform(2.0, 4.0),
                'reaction_time': random.uniform(0.2, 0.4)
            }

            # Normaliser la direction
            if np.linalg.norm(params['avg_direction']) > 0:
                params['avg_direction'] = params['avg_direction'] / np.linalg.norm(params['avg_direction'])

            # Créer l'agent
            agent = PersonAgent(agent_id, self, params)
            self.schedule.add(agent)
            self.space.place_agent(agent, pos)

    def _apply_leader_emergency_scenario(self):
        """Scénario d'évacuation avec leaders"""
        leaders = [a for a in self.schedule.agents if a.agent_type == AgentType.LEADER]

        for leader in leaders:
            leader.state = AgentState.EVACUATING

            # Faire suivre les agents proches
            for agent in self.schedule.agents:
                if agent.unique_id != leader.unique_id:
                    dist = np.linalg.norm(np.array(agent.pos) - np.array(leader.pos))
                    if dist < 3.0:
                        agent.leader = leader
                        agent.following_tendency = 1.0
                        agent.state = AgentState.FOLLOWING

    def step(self):
        """Exécute une étape de simulation"""
        self.schedule.step()

        # Collecter les données
        self.datacollector.collect(self)

        # Calculer les métriques globales
        self._compute_global_metrics()

        # Vérifier la fin de la simulation
        active_agents = sum(1 for a in self.schedule.agents if not a.evacuated)
        if active_agents == 0 and self.verbose:
            print("✅ Tous les agents sont évacués!")

    def _compute_global_metrics(self):
        """Calcule les métriques globales"""
        active_agents = [a for a in self.schedule.agents if not a.evacuated]

        if not active_agents:
            return

        metrics = {
            'step': self.schedule.steps,
            'active_agents': len(active_agents),
            'average_speed': np.mean([np.linalg.norm(a.velocity) for a in active_agents]),
            'average_stress': np.mean([a.stress for a in active_agents]),
            'average_panic': np.mean([a.panic_level for a in active_agents]),
            'state_distribution': {}
        }

        # Distribution des états
        for state in AgentState:
            count = sum(1 for a in active_agents if a.state == state)
            metrics['state_distribution'][state.value] = count

        self.global_metrics.append(metrics)

    def run_simulation(self, steps: int = 500):
        """Exécute la simulation pour un nombre d'étapes donné"""
        if self.verbose:
            print(f"\n🚀 Démarrage simulation MESA ({steps} steps)")

        for step in range(steps):
            if self.verbose and step % 50 == 0:
                active = sum(1 for a in self.schedule.agents if not a.evacuated)
                print(f"  Step {step}/{steps} - Agents actifs: {active}")

            self.step()

            # Arrêter si tous les agents sont évacués
            if self.evacuated_count >= self.schedule.get_agent_count():
                if self.verbose:
                    print(f"✅ Simulation terminée à l'étape {step}")
                break

        if self.verbose:
            print(f"\n📊 Simulation terminée:")
            print(f"   • Étapes: {self.schedule.steps}")
            print(f"   • Agents évacués: {self.evacuated_count}/{self.schedule.get_agent_count()}")
            print(f"   • Taux d'évacuation: {(self.evacuated_count/self.schedule.get_agent_count())*100:.1f}%")

        return self.get_simulation_results()

    def get_simulation_results(self):
        """Récupère les résultats de la simulation"""
        # Données des agents
        agent_data = []
        for agent in self.schedule.agents:
            agent_data.append({
                'id': agent.unique_id,
                'type': agent.agent_type.value,
                'final_state': agent.state.value,
                'panic_level': agent.panic_level,
                'distance_traveled': agent.distance_traveled,
                'collisions': agent.collisions_count,
                'state_changes': agent.state_changes,
                'evacuated': agent.evacuated,
                'final_position': agent.pos if hasattr(agent, 'pos') else None
            })

        # Données du modèle
        model_df = self.datacollector.get_model_vars_dataframe()
        agent_df = self.datacollector.get_agent_vars_dataframe()

        return {
            'agent_data': agent_data,
            'model_data': model_df,
            'agent_vars_data': agent_df,
            'global_metrics': self.global_metrics,
            'state_changes': self.state_changes
        }

# ============================================================================
# VISUALISATION MESA - VERSION COMPATIBLE 1.2.0
# ============================================================================

def agent_portrayal(agent):
    """Portrait de l'agent pour la visualisation MESA"""
    portrayal = {
        "Shape": "circle",
        "Filled": "true",
        "r": 0.3,
        "Layer": 0,
        "Color": "blue"
    }

    # Couleur selon l'état
    if agent.state == AgentState.PANIC:
        portrayal["Color"] = "red"
    elif agent.state == AgentState.EVACUATING:
        portrayal["Color"] = "orange"
    elif agent.state == AgentState.GROUPING:
        portrayal["Color"] = "green"
    elif agent.state == AgentState.FOLLOWING:
        portrayal["Color"] = "purple"
    elif agent.state == AgentState.OBSTACLE_AVOIDANCE:
        portrayal["Color"] = "brown"
    elif agent.state == AgentState.DISPERSING:
        portrayal["Color"] = "cyan"

    # Taille selon le type
    if agent.agent_type == AgentType.LEADER:
        portrayal["r"] = 0.4
        portrayal["Layer"] = 1

    # Si évacué, ne pas afficher
    if hasattr(agent, 'evacuated') and agent.evacuated:
        portrayal["Color"] = "gray"
        portrayal["r"] = 0.1

    return portrayal

def create_mesa_visualization(model_params):
    """Crée une visualisation interactive MESA pour version 1.2.0"""

    # Dans MESA 1.2.0, on utilise directement CanvasGrid
    from mesa.visualization import CanvasGrid, ChartModule, TextElement, ModularServer

    # Carte des agents
    grid = CanvasGrid(
        agent_portrayal,
        int(model_params['width']),
        int(model_params['height']),
        int(model_params['width'] * 40),
        int(model_params['height'] * 40)
    )

    # Graphiques
    chart = ChartModule([
        {"Label": "Active Agents", "Color": "blue"},
        {"Label": "Evacuated Agents", "Color": "green"},
        {"Label": "Average Speed", "Color": "red"},
        {"Label": "Average Stress", "Color": "orange"}
    ])

    # Texte
    class ModelText(TextElement):
        def render(self, model):
            return f"Agents actifs: {sum(1 for a in model.schedule.agents if not a.evacuated)}<br>" \
                   f"Agents évacués: {model.evacuated_count}<br>" \
                   f"Étape: {model.schedule.steps}"

    model_text = ModelText()

    # Serveur - CORRECTION ICI: Utiliser ModularServer directement
    server = ModularServer(
        CrowdModel,
        [grid, model_text, chart],
        "Simulation de Foule Multi-Agents",
        model_params
    )

    return server

# ============================================================================
# VISUALISATION SANS SERVEUR (BATCH)
# ============================================================================

def create_batch_visualization(simulation_results, scenario_name=""):
    """Crée des visualisations matplotlib sans serveur web"""

    analyzer = SimulationAnalyzer(simulation_results)
    analyzer.create_summary_report()

    output_dir = f"batch_visualization_{scenario_name}_{datetime.now().strftime('%H%M%S')}"
    analyzer.plot_simulation_results(output_dir)

    return output_dir

# ============================================================================
# ANALYSE ET VISUALISATION AVANCÉE
# ============================================================================

class SimulationAnalyzer:
    """Analyse les résultats de simulation"""

    def __init__(self, simulation_results):
        self.results = simulation_results
        self.agent_data = simulation_results['agent_data']
        self.model_data = simulation_results['model_data']
        self.global_metrics = simulation_results['global_metrics']

    def create_summary_report(self):
        """Crée un rapport de synthèse"""
        print("\n" + "="*60)
        print("📊 RAPPORT DE SYNTHÈSE DE LA SIMULATION")
        print("="*60)

        # Statistiques générales
        total_agents = len(self.agent_data)
        evacuated_agents = sum(1 for a in self.agent_data if a['evacuated'])
        avg_panic = np.mean([a['panic_level'] for a in self.agent_data])
        avg_distance = np.mean([a['distance_traveled'] for a in self.agent_data])
        total_collisions = sum(a['collisions'] for a in self.agent_data)

        print(f"\n📈 STATISTIQUES GÉNÉRALES:")
        print(f"   • Agents totaux: {total_agents}")
        print(f"   • Agents évacués: {evacuated_agents} ({evacuated_agents/total_agents*100:.1f}%)")
        print(f"   • Panique moyenne: {avg_panic:.3f}")
        print(f"   • Distance moyenne parcourue: {avg_distance:.1f} m")
        print(f"   • Collisions totales: {total_collisions}")

        # Distribution des états finaux
        print(f"\n🎭 DISTRIBUTION DES ÉTATS FINAUX:")
        states = {}
        for agent in self.agent_data:
            state = agent['final_state']
            states[state] = states.get(state, 0) + 1

        for state, count in states.items():
            percentage = count / total_agents * 100
            print(f"   • {state}: {count} agents ({percentage:.1f}%)")

        # Métriques temporelles
        if len(self.global_metrics) > 0:
            print(f"\n⏱ MÉTRIQUES TEMPORELLES:")
            print(f"   • Étapes de simulation: {len(self.global_metrics)}")
            print(f"   • Vitesse moyenne finale: {self.global_metrics[-1]['average_speed']:.2f} m/s")
            print(f"   • Stress moyen final: {self.global_metrics[-1]['average_stress']:.3f}")

    def plot_simulation_results(self, output_dir="simulation_plots"):
        """Génère des visualisations des résultats"""
        import os
        os.makedirs(output_dir, exist_ok=True)

        # 1. Évolution des métriques globales
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))

        if len(self.global_metrics) > 0:
            steps = [m['step'] for m in self.global_metrics]

            # Agents actifs
            axes[0, 0].plot(steps, [m['active_agents'] for m in self.global_metrics], 'b-', linewidth=2)
            axes[0, 0].set_title('Agents Actifs au Cours du Temps')
            axes[0, 0].set_xlabel('Étape')
            axes[0, 0].set_ylabel('Nombre d\'agents')
            axes[0, 0].grid(True, alpha=0.3)

            # Vitesse moyenne
            axes[0, 1].plot(steps, [m['average_speed'] for m in self.global_metrics], 'r-', linewidth=2)
            axes[0, 1].set_title('Vitesse Moyenne au Cours du Temps')
            axes[0, 1].set_xlabel('Étape')
            axes[0, 1].set_ylabel('Vitesse (m/s)')
            axes[0, 1].grid(True, alpha=0.3)

            # Stress moyen
            axes[1, 0].plot(steps, [m['average_stress'] for m in self.global_metrics], 'orange', linewidth=2)
            axes[1, 0].set_title('Stress Moyen au Cours du Temps')
            axes[1, 0].set_xlabel('Étape')
            axes[1, 0].set_ylabel('Stress (0-1)')
            axes[1, 0].grid(True, alpha=0.3)

            # Distribution des états (dernière étape)
            last_state_dist = self.global_metrics[-1]['state_distribution']
            states = list(last_state_dist.keys())
            counts = list(last_state_dist.values())

            colors = ['blue', 'green', 'cyan', 'red', 'orange', 'purple', 'brown']
            axes[1, 1].bar(states, counts, color=colors[:len(states)])
            axes[1, 1].set_title('Distribution des États (Fin de Simulation)')
            axes[1, 1].set_xlabel('État')
            axes[1, 1].set_ylabel('Nombre d\'agents')
            axes[1, 1].tick_params(axis='x', rotation=45)

        plt.tight_layout()
        plt.savefig(f"{output_dir}/global_metrics.png", dpi=150, bbox_inches='tight')

        # 2. Positions finales des agents
        fig2, ax2 = plt.subplots(figsize=(10, 8))

        # Dessiner les obstacles
        for obs in [(0, 0, 15, 0.2), (0, 11.8, 15, 0.2), (0, 0, 0.2, 12),
                   (14.8, 0, 0.2, 12), (5, 4, 2, 1), (10, 7, 1, 3)]:
            x, y, w, h = obs
            rect = patches.Rectangle((x, y), w, h, facecolor='gray', alpha=0.5, edgecolor='black')
            ax2.add_patch(rect)

        # Dessiner les sorties
        for exit in [(7.5, 0, 1.5), (7.5, 12, 1.5), (0, 6, 1.0), (15, 6, 1.0)]:
            x, y, w = exit
            rect = patches.Rectangle((x - w/2, y - 0.1), w, 0.2,
                                    facecolor='green', alpha=0.7, edgecolor='darkgreen')
            ax2.add_patch(rect)

        # Positions finales des agents
        for agent in self.agent_data:
            if agent['final_position']:
                x, y = agent['final_position']
                color = 'red' if agent['panic_level'] > 0.5 else 'blue'
                size = 50 if agent['type'] == 'leader' else 30
                ax2.scatter(x, y, color=color, s=size, alpha=0.7)

        ax2.set_xlim(0, 15)
        ax2.set_ylim(0, 12)
        ax2.set_aspect('equal')
        ax2.set_title('Positions Finales des Agents')
        ax2.set_xlabel('X (mètres)')
        ax2.set_ylabel('Y (mètres)')

        plt.tight_layout()
        plt.savefig(f"{output_dir}/final_positions.png", dpi=150, bbox_inches='tight')

        # 3. Distribution des métriques individuelles
        fig3, axes3 = plt.subplots(2, 2, figsize=(12, 10))

        # Distribution de la panique
        panic_levels = [a['panic_level'] for a in self.agent_data]
        axes3[0, 0].hist(panic_levels, bins=20, color='red', alpha=0.7, edgecolor='black')
        axes3[0, 0].set_title('Distribution des Niveaux de Panique')
        axes3[0, 0].set_xlabel('Niveau de panique')
        axes3[0, 0].set_ylabel('Nombre d\'agents')

        # Distribution des distances parcourues
        distances = [a['distance_traveled'] for a in self.agent_data]
        axes3[0, 1].hist(distances, bins=20, color='blue', alpha=0.7, edgecolor='black')
        axes3[0, 1].set_title('Distribution des Distances Parcourues')
        axes3[0, 1].set_xlabel('Distance (mètres)')
        axes3[0, 1].set_ylabel('Nombre d\'agents')

        # Distribution des collisions
        collisions = [a['collisions'] for a in self.agent_data]
        if len(collisions) > 0:
            axes3[1, 0].hist(collisions, bins=range(0, int(max(collisions)) + 2),
                             color='orange', alpha=0.7, edgecolor='black')
            axes3[1, 0].set_title('Distribution des Collisions')
            axes3[1, 0].set_xlabel('Nombre de collisions')
            axes3[1, 0].set_ylabel('Nombre d\'agents')

        # Distribution des changements d'état
        state_changes = [a['state_changes'] for a in self.agent_data]
        if len(state_changes) > 0:
            axes3[1, 1].hist(state_changes, bins=range(0, int(max(state_changes)) + 2),
                             color='green', alpha=0.7, edgecolor='black')
            axes3[1, 1].set_title('Distribution des Changements d\'État')
            axes3[1, 1].set_xlabel('Nombre de changements d\'état')
            axes3[1, 1].set_ylabel('Nombre d\'agents')

        plt.tight_layout()
        plt.savefig(f"{output_dir}/distributions.png", dpi=150, bbox_inches='tight')

        plt.close('all')
        print(f"📈 Visualisations sauvegardées dans: {output_dir}")

# ============================================================================
# EXÉCUTION PRINCIPALE
# ============================================================================

def run_mesa_simulation_batch():
    """Exécute la simulation MESA en mode batch (sans serveur web)"""
    print("="*80)
    print("🚀 SIMULATION MULTI-AGENTS MESA - MODE BATCH")
    print("="*80)

    # 1. Charger les données réelles
    print("\n1. 📥 CHARGEMENT DES DONNÉES DE TRACKING")
    trajectories_path = "/content/trajectories_corrected.csv"

    if os.path.exists(trajectories_path):
        print(f"📁 Fichier trouvé: {trajectories_path}")
        data_loader = RealDataLoader(trajectories_path)
    else:
        print("⚠ Fichier de tracking non trouvé, utilisation de données synthétiques")
        data_loader = RealDataLoader("")

    # 2. Sélection du scénario
    print("\n2. 🎭 SÉLECTION DU SCÉNARIO")
    print("\nScénarios disponibles:")
    print("  1. Normal (reproduction du comportement observé)")
    print("  2. Panique (niveau élevé de stress)")
    print("  3. Sorties bloquées (capacité réduite)")
    print("  4. Haute densité (plus d'agents)")
    print("  5. Évacuation dirigée (avec leaders)")

    choice = input("\nChoisissez un scénario (1-5, Enter pour 1): ").strip() or "1"

    scenarios = {
        "1": ("normal", 0.0),
        "2": ("panic", 0.5),
        "3": ("obstacle_blocked", 0.5),
        "4": ("high_density", 1.0),
        "5": ("leader_emergency", 0.8)
    }

    if choice in scenarios:
        scenario_name, intensity = scenarios[choice]
    else:
        scenario_name, intensity = "normal", 0.0

    print(f"\nScénario sélectionné: {scenario_name} (intensité: {intensity})")

    # 3. Configuration du modèle
    print("\n3. ⚙ CONFIGURATION DU MODÈLE MESA")

    model = CrowdModel(
        real_data_loader=data_loader,
        width=15.0,
        height=12.0,
        time_step=0.1,
        what_if_scenario=scenario_name,
        scenario_intensity=intensity,
        max_agents=30,  # Limiter pour la performance
        verbose=True
    )

    # 4. Demander si l'utilisateur veut générer une vidéo
    print("\n4. 🎥 GÉNÉRATION DE VIDÉO")
    generate_video = input("Voulez-vous générer une vidéo de la simulation? (o/n): ").strip().lower()

    if generate_video == 'o':
        print("🎬 Préparation de la génération vidéo...")

        # 5. Créer et exécuter le générateur de vidéo
        video_gen = VideoGenerator(model, steps=200)
        video_gen.capture_simulation()

        # 6. Sauvegarder la vidéo
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        video_filename = f"simulation_{scenario_name}_{timestamp}.mp4"
        gif_filename = f"simulation_{scenario_name}_{timestamp}.gif"

        gif_path, mp4_path = video_gen.save_video(video_filename, fps=15)

        print(f"\n✅ Vidéo générée avec succès!")
        if gif_path:
            print(f"📁 GIF: {gif_path}")
        if mp4_path:
            print(f"📁 MP4: {mp4_path}")

        # 7. Continuer avec l'analyse normale
        print("\n5. 📊 ANALYSE DES RÉSULTATS")
        results = model.get_simulation_results()
        analyzer = SimulationAnalyzer(results)
        analyzer.create_summary_report()

        # 8. Visualisation
        print("\n6. 🎨 GÉNÉRATION DES VISUALISATIONS")
        output_dir = create_batch_visualization(results, scenario_name)

        # 9. Sauvegarde des résultats
        print("\n7. 💾 SAUVEGARDE DES RÉSULTATS")
        results_dir = f"mesa_results_{scenario_name}_{timestamp}"
        os.makedirs(results_dir, exist_ok=True)

        # Sauvegarder les données
        pd.DataFrame(results['agent_data']).to_csv(f"{results_dir}/agent_results.csv", index=False)
        results['model_data'].to_csv(f"{results_dir}/model_metrics.csv")

        if results.get('agent_vars_data') is not None:
            results['agent_vars_data'].to_csv(f"{results_dir}/agent_vars.csv")

        with open(f"{results_dir}/global_metrics.json", 'w') as f:
            json.dump(results['global_metrics'], f, indent=2)

        # Copier les vidéos et visualisations
        import shutil
        if gif_path and os.path.exists(gif_path):
            shutil.copy2(gif_path, f"{results_dir}/{os.path.basename(gif_path)}")
        if mp4_path and os.path.exists(mp4_path):
            shutil.copy2(mp4_path, f"{results_dir}/{os.path.basename(mp4_path)}")

        if os.path.exists(output_dir):
            for file in os.listdir(output_dir):
                shutil.copy2(f"{output_dir}/{file}", f"{results_dir}/{file}")

        print(f"\n🎉 SIMULATION TERMINÉE!")
        print(f"📁 Résultats complets dans: {results_dir}")
        if gif_path or mp4_path:
            print(f"🎥 Vidéos dans: {results_dir}")
        print(f"📈 Visualisations dans: {output_dir}")

        return results, gif_path, mp4_path

    else:
        # Mode sans vidéo (original)
        # 5. Exécution de la simulation
        print("\n5. 🔄 EXÉCUTION DE LA SIMULATION")
        results = model.run_simulation(steps=300)

        # 6. Analyse des résultats
        print("\n6. 📊 ANALYSE DES RÉSULTATS")
        analyzer = SimulationAnalyzer(results)
        analyzer.create_summary_report()

        # 7. Visualisation
        print("\n7. 🎨 GÉNÉRATION DES VISUALISATIONS")
        output_dir = create_batch_visualization(results, scenario_name)

        # 8. Sauvegarde des résultats
        print("\n8. 💾 SAUVEGARDE DES RÉSULTATS")
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        results_dir = f"mesa_results_{scenario_name}_{timestamp}"
        os.makedirs(results_dir, exist_ok=True)

        # Sauvegarder les données
        pd.DataFrame(results['agent_data']).to_csv(f"{results_dir}/agent_results.csv", index=False)
        results['model_data'].to_csv(f"{results_dir}/model_metrics.csv")

        if results.get('agent_vars_data') is not None:
            results['agent_vars_data'].to_csv(f"{results_dir}/agent_vars.csv")

        with open(f"{results_dir}/global_metrics.json", 'w') as f:
            json.dump(results['global_metrics'], f, indent=2)

        # Copier les visualisations
        import shutil
        if os.path.exists(output_dir):
            for file in os.listdir(output_dir):
                shutil.copy2(f"{output_dir}/{file}", f"{results_dir}/{file}")

        print(f"\n🎉 SIMULATION TERMINÉE!")
        print(f"📁 Résultats complets dans: {results_dir}")
        print(f"📈 Visualisations dans: {output_dir}")

        return results

# ============================================================================
# FONCTION SIMPLIFIÉE POUR GÉNÉRER UNE VIDÉO RAPIDE
# ============================================================================

def generate_simulation_video():
    """Fonction simplifiée pour générer une vidéo de simulation"""
    print("🎬 GÉNÉRATEUR DE VIDÉO DE SIMULATION")
    print("="*50)

    # Configuration simple
    trajectories_path = "/content/trajectories_corrected.csv"

    if os.path.exists(trajectories_path):
        print(f"📁 Chargement des données: {trajectories_path}")
        data_loader = RealDataLoader(trajectories_path)
    else:
        print("⚠ Données synthétiques")
        data_loader = RealDataLoader("")

    # Créer le modèle
    model = CrowdModel(
        real_data_loader=data_loader,
        width=15.0,
        height=12.0,
        time_step=0.1,
        what_if_scenario="normal",
        scenario_intensity=0.0,
        max_agents=20,
        verbose=True
    )

    # Demander les paramètres
    steps = input("\nNombre de frames (défaut: 150): ").strip()
    steps = int(steps) if steps else 150

    fps = input("Images par seconde (défaut: 20): ").strip()
    fps = int(fps) if fps else 20

    scenario = input("\nScénario (normal/panic/obstacle_blocked/high_density/leader_emergency, défaut: normal): ").strip()
    scenario = scenario if scenario else "normal"

    # Appliquer le scénario
    model.what_if_scenario = scenario

    # Générer la vidéo
    print(f"\n🎥 Génération de {steps} frames à {fps} FPS...")

    video_gen = VideoGenerator(model, steps=steps)
    video_gen.capture_simulation()

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    video_filename = f"simulation_{scenario}_{timestamp}.mp4"

    gif_path, mp4_path = video_gen.save_video(video_filename, fps=fps)

    print(f"\n✅ Vidéo générée avec succès!")
    if gif_path:
        print(f"📁 GIF: {gif_path}")
    if mp4_path:
        print(f"📁 MP4: {mp4_path}")

    return gif_path, mp4_path

# ============================================================================
# COMPARAISON DE SCÉNARIOS
# ============================================================================

def compare_scenarios_batch():
    """Compare plusieurs scénarios what-if en mode batch"""
    print("\n" + "="*80)
    print("📊 COMPARAISON DE SCÉNARIOS WHAT-IF")
    print("="*80)

    # Scénarios à comparer
    scenarios = [
        ("normal", 0.0),
        ("panic", 0.3),
        ("panic", 0.7),
        ("obstacle_blocked", 0.5),
        ("high_density", 1.5),
        ("leader_emergency", 0.8)
    ]

    results = {}

    # Charger les données réelles une fois
    trajectories_path = "/content/trajectories_corrected.csv"
    if os.path.exists(trajectories_path):
        data_loader = RealDataLoader(trajectories_path)
    else:
        print("⚠ Fichier de tracking non trouvé, utilisation de données synthétiques")
        data_loader = RealDataLoader("")

    for scenario_name, intensity in scenarios:
        print(f"\n{'='*50}")
        print(f"Scénario: {scenario_name.upper()} (intensité: {intensity})")
        print(f"{'='*50}")

        # Créer et exécuter le modèle
        model = CrowdModel(
            real_data_loader=data_loader,
            width=15.0,
            height=12.0,
            time_step=0.1,
            what_if_scenario=scenario_name,
            scenario_intensity=intensity,
            max_agents=20,  # Limité pour la performance
            verbose=False
        )

        # Exécuter la simulation
        sim_results = model.run_simulation(steps=200)

        # Extraire les métriques clés
        agent_data = sim_results['agent_data']
        total_agents = len(agent_data)
        evacuated = sum(1 for a in agent_data if a['evacuated'])
        avg_panic = np.mean([a['panic_level'] for a in agent_data])
        avg_distance = np.mean([a['distance_traveled'] for a in agent_data])
        total_collisions = sum(a['collisions'] for a in agent_data)

        results[scenario_name] = {
            'evacuation_rate': evacuated / total_agents if total_agents > 0 else 0,
            'avg_panic': avg_panic,
            'avg_distance': avg_distance,
            'total_collisions': total_collisions,
            'avg_speed': np.mean([m['average_speed'] for m in sim_results['global_metrics']]) if sim_results['global_metrics'] else 0,
            'max_stress': np.max([m['average_stress'] for m in sim_results['global_metrics']]) if sim_results['global_metrics'] else 0
        }

        print(f"  Taux d'évacuation: {results[scenario_name]['evacuation_rate']*100:.1f}%")
        print(f"  Panique moyenne: {results[scenario_name]['avg_panic']:.3f}")
        print(f"  Collisions: {results[scenario_name]['total_collisions']}")

    # Visualisation comparative
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle('Comparaison des Scénarios What-If', fontsize=16, fontweight='bold')

    scenarios_list = list(results.keys())

    # Taux d'évacuation
    evacuation_rates = [results[s]['evacuation_rate'] for s in scenarios_list]
    axes[0, 0].bar(scenarios_list, evacuation_rates, color='green', alpha=0.7)
    axes[0, 0].set_title('Taux d\'Évacuation')
    axes[0, 0].set_ylabel('Taux')
    axes[0, 0].tick_params(axis='x', rotation=45)
    axes[0, 0].set_ylim(0, 1)

    # Panique moyenne
    panic_levels = [results[s]['avg_panic'] for s in scenarios_list]
    axes[0, 1].bar(scenarios_list, panic_levels, color='red', alpha=0.7)
    axes[0, 1].set_title('Panique Moyenne')
    axes[0, 1].set_ylabel('Niveau (0-1)')
    axes[0, 1].tick_params(axis='x', rotation=45)
    axes[0, 1].set_ylim(0, 1)

    # Collisions
    collisions = [results[s]['total_collisions'] for s in scenarios_list]
    axes[0, 2].bar(scenarios_list, collisions, color='orange', alpha=0.7)
    axes[0, 2].set_title('Nombre de Collisions')
    axes[0, 2].set_ylabel('Collisions')
    axes[0, 2].tick_params(axis='x', rotation=45)

    # Distance moyenne
    distances = [results[s]['avg_distance'] for s in scenarios_list]
    axes[1, 0].bar(scenarios_list, distances, color='blue', alpha=0.7)
    axes[1, 0].set_title('Distance Moyenne Parcourue')
    axes[1, 0].set_ylabel('Distance (m)')
    axes[1, 0].tick_params(axis='x', rotation=45)

    # Vitesse moyenne
    speeds = [results[s]['avg_speed'] for s in scenarios_list]
    axes[1, 1].bar(scenarios_list, speeds, color='purple', alpha=0.7)
    axes[1, 1].set_title('Vitesse Moyenne')
    axes[1, 1].set_ylabel('Vitesse (m/s)')
    axes[1, 1].tick_params(axis='x', rotation=45)

    # Stress maximal
    stresses = [results[s]['max_stress'] for s in scenarios_list]
    axes[1, 2].bar(scenarios_list, stresses, color='brown', alpha=0.7)
    axes[1, 2].set_title('Stress Maximal')
    axes[1, 2].set_ylabel('Stress (0-1)')
    axes[1, 2].tick_params(axis='x', rotation=45)
    axes[1, 2].set_ylim(0, 1)

    plt.tight_layout()
    plt.savefig('scenario_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f"\n📈 Comparaison sauvegardée: scenario_comparison.png")

    # Sauvegarde des résultats de comparaison
    comparison_df = pd.DataFrame.from_dict(results, orient='index')
    comparison_df.to_csv('scenario_comparison_results.csv')
    print(f"📊 Données de comparaison sauvegardées: scenario_comparison_results.csv")

    return results

# ============================================================================
# INTERFACE SIMPLIFIÉE
# ============================================================================

def main():
    """Interface principale simplifiée"""
    print("🎯 SIMULATION MULTI-AGENTS MESA - Tracking de Foule")
    print("   Basé sur les données réelles de PETS2009")
    print("\n⚠ NOTE: Le serveur web interactif peut ne pas fonctionner")
    print("   dans certains environnements comme Google Colab.")
    print("   Utilisez le mode batch pour des résultats fiables.")

    while True:
        print("\n" + "="*50)
        print("MENU PRINCIPAL")
        print("="*50)
        print("1. Exécuter une simulation simple (mode batch)")
        print("2. Comparer plusieurs scénarios")
        print("3. Générer une vidéo de simulation")
        print("4. Quitter")

        choice = input("\nChoisissez une option (1-4): ").strip()

        if choice == "1":
            run_mesa_simulation_batch()
        elif choice == "2":
            compare_scenarios_batch()
        elif choice == "3":
            generate_simulation_video()
        elif choice == "4":
            print("\n👋 Au revoir!")
            break
        else:
            print("❌ Choix invalide, veuillez réessayer.")

# ============================================================================
# POINT D'ENTRÉE POUR GOOGLE COLAB
# ============================================================================

if __name__ == "__main__":
    # Démarrer directement en mode batch pour Colab
    # (le serveur web ne fonctionne pas bien dans Colab)

    try:
        # Vérifier si nous sommes dans Colab
        import google.colab
        IN_COLAB = True
        print("🌐 Environnement Google Colab détecté")
        print("🔧 Utilisation du mode batch (serveur web non supporté)")

        # Vérifier si ffmpeg est installé
        try:
            import subprocess
            subprocess.run(['ffmpeg', '-version'], capture_output=True)
            print("✅ ffmpeg est installé (MP4 possible)")
        except:
            print("⚠ ffmpeg n'est pas installé, seulement GIF possible")
            print("💡 Pour installer ffmpeg dans Colab: !apt-get install ffmpeg -y")

        # Exécuter directement en mode batch
        main()

    except ImportError:
        IN_COLAB = False
        print("💻 Environnement local détecté")
        print("⚡ Vous pouvez utiliser toutes les fonctionnalités")

        # Dans un environnement local, proposer toutes les options
        print("\nActions disponibles:")
        print("1. Exécuter une simulation simple (mode batch - recommandé)")
        print("2. Comparer plusieurs scénarios")
        print("3. Générer une vidéo de simulation")
        print("4. Essayer le serveur web interactif (peut ne pas fonctionner)")

        action = input("\nChoisissez une action (1-4): ").strip()

        if action == "1":
            run_mesa_simulation_batch()
        elif action == "2":
            compare_scenarios_batch()
        elif action == "3":
            generate_simulation_video()
        elif action == "4":
            # Essayer le serveur web
            print("\n⚠ ATTENTION: Le serveur web peut nécessiter des dépendances supplémentaires.")
            print("   Il est recommandé d'utiliser le mode batch pour des résultats fiables.")

            confirm = input("\nVoulez-vous quand même essayer le serveur web? (o/n): ").strip().lower()
            if confirm == 'o':
                try:
                    # Essayer d'importer les modules de visualisation
                    from mesa.visualization import CanvasGrid, ChartModule, TextElement, ModularServer

                    # Charger les données
                    trajectories_path = "/content/trajectories_corrected.csv"
                    if os.path.exists(trajectories_path):
                        data_loader = RealDataLoader(trajectories_path)
                    else:
                        print("⚠ Fichier non trouvé, utilisation de données synthétiques")
                        data_loader = RealDataLoader("")

                    # Configuration
                    model_params = {
                        "real_data_loader": data_loader,
                        "width": 15.0,
                        "height": 12.0,
                        "time_step": 0.1,
                        "what_if_scenario": "normal",
                        "scenario_intensity": 0.0,
                        "max_agents": 15,
                        "verbose": False
                    }

                    # Créer le serveur
                    server = create_mesa_visualization(model_params)
                    server.port = 8521

                    print(f"\n🌐 Serveur web démarré!")
                    print(f"➡ Ouvrez votre navigateur à: http://localhost:8521/")
                    print(f"➡ Pour arrêter le serveur: Ctrl+C")

                    server.launch()

                except Exception as e:
                    print(f"❌ Erreur lors du lancement du serveur web: {e}")
                    print("🔧 Retour au mode batch...")
                    run_mesa_simulation_batch()
            else:
                run_mesa_simulation_batch()
        else:
            print("❌ Choix invalide, exécution par défaut")
            run_mesa_simulation_batch()

🌐 Environnement Google Colab détecté
🔧 Utilisation du mode batch (serveur web non supporté)
✅ ffmpeg est installé (MP4 possible)
🎯 SIMULATION MULTI-AGENTS MESA - Tracking de Foule
   Basé sur les données réelles de PETS2009

⚠ NOTE: Le serveur web interactif peut ne pas fonctionner
   dans certains environnements comme Google Colab.
   Utilisez le mode batch pour des résultats fiables.

MENU PRINCIPAL
1. Exécuter une simulation simple (mode batch)
2. Comparer plusieurs scénarios
3. Générer une vidéo de simulation
4. Quitter

Choisissez une option (1-4): 3
🎬 GÉNÉRATEUR DE VIDÉO DE SIMULATION
📁 Chargement des données: /content/trajectories_corrected.csv
📊 Données réelles chargées: 591 points de trajectoire
👤 Agent 0 créé (leader) à [     9.7948       30.24]
👤 Agent 1 créé (follower) à [    0.10789       20.75]
👤 Agent 2 créé (independent) à [   0.043271      33.432]
👤 Agent 3 créé (leader) à [     14.319      18.758]
👤 Agent 4 créé (follower) à [     18.053      9.6202]
👤 Agent 5 créé (f

#SMA version finale

In [ ]:
# mesa_crowd_simulation_v2_0_0.py
import mesa
import mesa.space
import mesa.time
import numpy as np
import pandas as pd
from enum import Enum
import random
from typing import List, Dict, Optional, Tuple
import networkx as nx
from dataclasses import dataclass
import json
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.animation as animation
from PIL import Image
import io
import os
import warnings
import math
from scipy.spatial import KDTree
from scipy.spatial.distance import cdist
from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error, mean_absolute_error
import sys
warnings.filterwarnings('ignore')

# ============================================================================
# CONSTANTES ET CONFIGURATION
# ============================================================================

VALIDATION_THRESHOLD = 0.7  # Seuil minimum d'acceptation de validation
CALIBRATION_ITERATIONS = 50  # Nombre d'itérations pour calibration
HELBING_PARAMS_RANGES = {
    'A_social': (1.0, 3.0),      # Amplitude force sociale
    'B_social': (0.2, 0.5),      # Portée force sociale
    'k_body': (8.0, 12.0),       # Force corps à corps
    'lambda_': (0.3, 0.7),       # Facteur d'anisotropie
    'tau': (0.3, 0.6)           # Temps de relaxation
}


# ÉTATS ET TYPES D'AGENTS

In [ ]:
# ============================================================================
# ÉTATS ET TYPES D'AGENTS
# ============================================================================

class AgentState(Enum):
    NORMAL = "normal"
    GROUPING = "grouping"
    DISPERSING = "dispersing"
    PANIC = "panic"
    EVACUATING = "evacuating"
    OBSTACLE_AVOIDANCE = "obstacle_avoidance"
    FOLLOWING = "following"
    STOPPED = "stopped"

class AgentType(Enum):
    PEDESTRIAN = "pedestrian"
    LEADER = "leader"
    FOLLOWER = "follower"
    INDEPENDENT = "independent"

# DONNÉES RÉELLES - VERSION RENFORCÉE

In [ ]:
# ============================================================================
# DONNÉES RÉELLES - VERSION RENFORCÉE
# ============================================================================

class RealDataLoader:
    """Charge et traite les données réelles de tracking SANS FALLBACK SYNTHÉTIQUE"""

    def __init__(self, trajectories_csv: str, fps: float = 10.0):
        self.trajectories_csv = trajectories_csv
        self.trajectories_data = None
        self.agent_parameters = {}
        self.collective_metrics = {}
        self.fps = fps  # Images par seconde

        if not os.path.exists(trajectories_csv):
            raise FileNotFoundError(
                f"❌ Fichier de données réelles non trouvé: {trajectories_csv}\n"
                f"   Veuillez fournir des données réelles de tracking pour une simulation valide."
            )

    def normalize_trajectories(self, trajectories):
        """
        Normalise les trajectoires pour qu'elles rentrent dans l'espace de simulation.
        On conserve les proportions relatives.
        """
        # Collecter toutes les positions
        all_x = []
        all_y = []
        for agent_id, traj in trajectories.items():
            for point in traj:
                all_x.append(point['world_x'])
                all_y.append(point['world_y'])

        all_x = np.array(all_x)
        all_y = np.array(all_y)

        # Calculer les bornes actuelles
        x_min, x_max = np.min(all_x), np.max(all_x)
        y_min, y_max = np.min(all_y), np.max(all_y)

        print(f"   Plage des données brutes: X[{x_min:.2f}, {x_max:.2f}], Y[{y_min:.2f}, {y_max:.2f}]")

        # Définir les bornes cibles (avec une marge de 1 mètre)
        target_x_min, target_x_max = 1.0, 14.0
        target_y_min, target_y_max = 1.0, 11.0

        # Fonction de normalisation linéaire
        def normalize(value, min_val, max_val, target_min, target_max):
            if max_val - min_val == 0:
                return target_min
            return (value - min_val) / (max_val - min_val) * (target_max - target_min) + target_min

        # Appliquer la normalisation
        normalized_trajectories = {}
        for agent_id, traj in trajectories.items():
            normalized_trajectories[agent_id] = []
            for point in traj:
                norm_x = normalize(point['world_x'], x_min, x_max, target_x_min, target_x_max)
                norm_y = normalize(point['world_y'], y_min, y_max, target_y_min, target_y_max)
                normalized_point = point.copy()
                normalized_point['world_x'] = norm_x
                normalized_point['world_y'] = norm_y
                normalized_trajectories[agent_id].append(normalized_point)

        # Vérifier les nouvelles plages
        norm_x_vals = [p['world_x'] for traj in normalized_trajectories.values() for p in traj]
        norm_y_vals = [p['world_y'] for traj in normalized_trajectories.values() for p in traj]
        print(f"   Plage après normalisation: X[{min(norm_x_vals):.2f}, {max(norm_x_vals):.2f}], Y[{min(norm_y_vals):.2f}, {max(norm_y_vals):.2f}]")

        return normalized_trajectories

    def load_real_trajectories(self) -> Dict:
        """Charge les trajectoires depuis le CSV de tracking"""
        try:
            df = pd.read_csv(self.trajectories_csv)
            print(f"📊 Données réelles chargées: {len(df)} points de trajectoire")

            # Valider les colonnes requises
            required_columns = ['id', 'frame', 'world_foot_x', 'world_foot_y']
            missing_cols = [col for col in required_columns if col not in df.columns]
            if missing_cols:
                raise ValueError(f"Colonnes manquantes: {missing_cols}")

            # Grouper par ID d'agent
            trajectories_by_id = {}
            for _, row in df.iterrows():
                agent_id = int(row['id'])
                if agent_id not in trajectories_by_id:
                    trajectories_by_id[agent_id] = []

                trajectories_by_id[agent_id].append({
                    'frame': int(row['frame']),
                    'world_x': float(row['world_foot_x']),
                    'world_y': float(row['world_foot_y']),
                    'pixel_x': float(row.get('pixel_foot_x', row['world_foot_x'] * 50)),
                    'pixel_y': float(row.get('pixel_foot_y', row['world_foot_y'] * 50)),
                    'score': float(row.get('score', 0.8))
                })

            # Trier chaque trajectoire par frame
            for agent_id in trajectories_by_id:
                trajectories_by_id[agent_id].sort(key=lambda x: x['frame'])

            # Filtrer les trajectoires trop courtes
            min_length = 10
            trajectories_by_id = {k: v for k, v in trajectories_by_id.items() if len(v) >= min_length}

            # Normaliser les trajectoires
            print("📐 Normalisation des trajectoires...")
            trajectories_by_id = self.normalize_trajectories(trajectories_by_id)

            self.trajectories_data = trajectories_by_id

            # Calculer les métriques collectives
            self._compute_collective_metrics(trajectories_by_id)

            print(f"✅ {len(trajectories_by_id)} agents réels chargés et normalisés")
            return trajectories_by_id

        except Exception as e:
            print(f"❌ Erreur critique lors du chargement: {e}")
            raise

    # ... (le reste de la classe reste inchangé)

    def _compute_collective_metrics(self, trajectories: Dict):
        """Calcule les métriques collectives à partir des données réelles"""
        print("📈 Calcul des métriques collectives...")

        all_positions = []
        all_velocities = []

        for agent_id, traj in trajectories.items():
            positions = np.array([[p['world_x'], p['world_y']] for p in traj])
            all_positions.append(positions)

            if len(traj) > 1:
                frames = np.array([p['frame'] for p in traj])
                time_diff = np.diff(frames) / 10.0  # 10 fps
                velocities = np.diff(positions, axis=0) / time_diff[:, np.newaxis]
                all_velocities.extend(velocities)

        # Métriques de densité
        if all_positions:
            all_positions_flat = np.vstack([pos[0] for pos in all_positions])  # Positions initiales
            self.collective_metrics['initial_density'] = self._compute_density_metrics(all_positions_flat)

        # Métriques de vitesse
        if all_velocities:
            velocities_array = np.array(all_velocities)
            speeds = np.linalg.norm(velocities_array, axis=1)
            self.collective_metrics['avg_speed'] = np.mean(speeds)
            self.collective_metrics['speed_std'] = np.std(speeds)
            self.collective_metrics['speed_distribution'] = {
                'min': np.min(speeds),
                'max': np.max(speeds),
                'percentiles': np.percentile(speeds, [25, 50, 75])
            }

        # Métriques de dispersion
        if len(all_positions) > 1:
            final_positions = np.vstack([pos[-1] for pos in all_positions])
            self.collective_metrics['dispersion'] = np.std(final_positions, axis=0)

        print("✅ Métriques collectives calculées")

    def _compute_density_metrics(self, positions: np.ndarray) -> Dict:
        """Calcule les métriques de densité spatiale"""
        if len(positions) < 2:
            return {'avg_nearest_neighbor': 0, 'density_map': None}

        # Distance au plus proche voisin
        tree = KDTree(positions)
        distances, _ = tree.query(positions, k=2)  # k=2 car le premier est l'agent lui-même
        avg_nearest_dist = np.mean(distances[:, 1])

        # Estimation de densité sur grille
        x_min, y_min = np.min(positions, axis=0)
        x_max, y_max = np.max(positions, axis=0)

        grid_size = 1.0  # mètres
        x_bins = int((x_max - x_min) / grid_size) + 1
        y_bins = int((y_max - y_min) / grid_size) + 1

        density_map = np.zeros((x_bins, y_bins))

        for x, y in positions:
            i = int((x - x_min) / grid_size)
            j = int((y - y_min) / grid_size)
            if 0 <= i < x_bins and 0 <= j < y_bins:
                density_map[i, j] += 1

        return {
            'avg_nearest_neighbor': float(avg_nearest_dist),
            'density_map': density_map.tolist(),
            'grid_size': grid_size,
            'bounds': [float(x_min), float(x_max), float(y_min), float(y_max)]
        }

    def extract_agent_parameters(self, agent_id: int, trajectory: List[Dict]) -> Optional[Dict]:
        """Extrait les paramètres comportementaux d'un agent réel"""
        if len(trajectory) < 5:
            return None

        # Positions et vitesses
        positions = np.array([[p['world_x'], p['world_y']] for p in trajectory])
        frames = np.array([p['frame'] for p in trajectory])

        # Calcul des vitesses
        if len(positions) > 1:
            displacements = np.diff(positions, axis=0)
            time_diff = np.diff(frames) / 10.0  # 10 fps
            velocities = displacements / time_diff[:, np.newaxis]

            avg_velocity = np.mean(np.linalg.norm(velocities, axis=1))
            max_velocity = np.max(np.linalg.norm(velocities, axis=1))
            velocity_std = np.std(np.linalg.norm(velocities, axis=1))

            # Direction moyenne (normalisée)
            if len(velocities) > 0:
                avg_direction = np.mean(velocities, axis=0)
                dir_norm = np.linalg.norm(avg_direction)
                if dir_norm > 0:
                    avg_direction = avg_direction / dir_norm
                else:
                    avg_direction = np.array([1.0, 0.0])
            else:
                avg_direction = np.array([1.0, 0.0])
        else:
            avg_velocity = 1.0
            max_velocity = 1.5
            velocity_std = 0.2
            avg_direction = np.array([1.0, 0.0])

        # Analyse de la trajectoire
        total_distance = np.sum(np.linalg.norm(np.diff(positions, axis=0), axis=1))
        straightness = 0.0
        if total_distance > 0:
            straightness = np.linalg.norm(positions[-1] - positions[0]) / total_distance

        # Variabilité de direction
        direction_variability = 0.0
        if len(positions) > 2:
            directions = np.diff(positions, axis=0)
            if len(directions) > 1:
                # Produit scalaire normalisé
                dots = np.sum(directions[1:] * directions[:-1], axis=1)
                norms = np.linalg.norm(directions[1:], axis=1) * np.linalg.norm(directions[:-1], axis=1)
                with np.errstate(invalid='ignore'):
                    angles = np.arccos(np.clip(dots / (norms + 1e-6), -1.0, 1.0))
                direction_variability = np.mean(np.nan_to_num(angles))

        # Analyse des arrêts
        stop_count = 0
        if len(velocities) > 0:
            speeds = np.linalg.norm(velocities, axis=1)
            stop_count = np.sum(speeds < 0.1)  # Vitesse < 0.1 m/s

        # Déterminer le type d'agent basé sur le comportement
        agent_type = self._classify_agent_type(
            straightness, direction_variability, avg_velocity,
            velocity_std, len(trajectory)
        )

        # Paramètres sociaux basés sur le type
        sociability, following_tendency = self._get_social_parameters(agent_type)

        # Initialiser le graphe de proximité (vide, sera construit dynamiquement)
        proximity_graph = nx.Graph()

        return {
            'agent_id': agent_id,
            'initial_position': positions[0],
            'avg_velocity': float(avg_velocity),
            'max_velocity': float(max_velocity),
            'velocity_std': float(velocity_std),
            'avg_direction': avg_direction.tolist(),
            'straightness': float(straightness),
            'direction_variability': float(direction_variability),
            'stop_count': int(stop_count),
            'agent_type': agent_type,
            'sociability': sociability,
            'following_tendency': following_tendency,
            'patience': random.uniform(0.4, 0.8),
            'panic_level': random.uniform(0.1, 0.3),
            'awareness_radius': random.uniform(2.0, 4.0),
            'reaction_time': random.uniform(0.2, 0.4),
            'proximity_graph': proximity_graph,
            'collective_metrics': self.collective_metrics
        }

    def _classify_agent_type(self, straightness: float, direction_variability: float,
                           avg_velocity: float, velocity_std: float, traj_length: int) -> AgentType:
        """Classifie le type d'agent basé sur les métriques comportementales"""

        if straightness > 0.8 and direction_variability < 0.3:
            # Trajectoire droite et stable = Leader
            return AgentType.LEADER
        elif direction_variability > 1.0 and straightness < 0.4 and velocity_std > 0.5:
            # Erratique et imprévisible = Indépendant
            return AgentType.INDEPENDENT
        elif avg_velocity > 1.2 and velocity_std < 0.3 and traj_length > 20:
            # Rapide et constant = Follower
            return AgentType.FOLLOWER
        else:
            # Comportement standard = Piéton
            return AgentType.PEDESTRIAN

    def _get_social_parameters(self, agent_type: AgentType) -> Tuple[float, float]:
        """Retourne les paramètres sociaux selon le type d'agent"""
        if agent_type == AgentType.LEADER:
            return random.uniform(0.7, 0.9), random.uniform(0.1, 0.3)
        elif agent_type == AgentType.INDEPENDENT:
            return random.uniform(0.2, 0.4), random.uniform(0.2, 0.4)
        elif agent_type == AgentType.FOLLOWER:
            return random.uniform(0.5, 0.7), random.uniform(0.6, 0.9)
        else:  # PEDESTRIAN
            return random.uniform(0.3, 0.6), random.uniform(0.3, 0.5)

# MÉTRIQUES DE VALIDATION

In [ ]:
# ============================================================================
# MÉTRIQUES DE VALIDATION
# ============================================================================

class ValidationMetrics:
    """Calcule les métriques de validation entre simulation et réalité"""

    def __init__(self, real_trajectories: Dict, simulated_trajectories: Dict, fps: float = 10.0):
        self.real_trajectories = real_trajectories
        self.sim_trajectories = simulated_trajectories
        self.fps = fps  # Images par seconde des données réelles
        self.metrics = {}

    def compute_all_metrics(self) -> Dict:
        """Calcule toutes les métriques de validation"""
        print("🔬 Calcul des métriques de validation...")

        self.metrics = {
            'position_errors': self.compute_position_errors(),
            'velocity_errors': self.compute_velocity_errors(),
            'trajectory_similarities': self.compute_trajectory_similarities(),
            'collective_metrics_comparison': self.compare_collective_metrics()
        }

        # Score de validation global
        self.metrics['validation_score'] = self.compute_validation_score()

        print(f"✅ Validation terminée - Score: {self.metrics['validation_score']:.3f}")
        return self.metrics

    def compute_position_errors(self) -> Dict:
        """Calcule les erreurs de position"""
        errors = {'mse': [], 'mae': [], 'rmse': []}

        for agent_id in self.real_trajectories:
            if agent_id in self.sim_trajectories:
                real_pos = np.array([[p['world_x'], p['world_y']]
                                   for p in self.real_trajectories[agent_id]])
                sim_pos = np.array(self.sim_trajectories[agent_id])

                # Aligner les longueurs
                min_len = min(len(real_pos), len(sim_pos))
                if min_len > 0:
                    real_pos_aligned = real_pos[:min_len]
                    sim_pos_aligned = sim_pos[:min_len]

                    # Calcul des erreurs
                    mse = mean_squared_error(real_pos_aligned, sim_pos_aligned)
                    mae = mean_absolute_error(real_pos_aligned, sim_pos_aligned)
                    rmse = np.sqrt(mse)

                    errors['mse'].append(mse)
                    errors['mae'].append(mae)
                    errors['rmse'].append(rmse)

        # Statistiques globales
        result = {}
        for error_type in ['mse', 'mae', 'rmse']:
            if errors[error_type]:
                result[f'{error_type}_mean'] = np.mean(errors[error_type])
                result[f'{error_type}_std'] = np.std(errors[error_type])
                result[f'{error_type}_median'] = np.median(errors[error_type])

        return result

    def compute_velocity_errors(self) -> Dict:
        """Calcule les erreurs de vitesse"""
        real_speeds = []
        sim_speeds = []

        for agent_id in self.real_trajectories:
            if agent_id in self.sim_trajectories and len(self.real_trajectories[agent_id]) > 1:
                # Vitesses réelles
                real_traj = self.real_trajectories[agent_id]
                real_pos = np.array([[p['world_x'], p['world_y']] for p in real_traj])
                real_frames = np.array([p['frame'] for p in real_traj])

                if len(real_pos) > 1:
                    real_displacements = np.diff(real_pos, axis=0)
                    real_time_diff = np.diff(real_frames) / self.fps
                    real_velocities = real_displacements / real_time_diff[:, np.newaxis]
                    real_speeds.extend(np.linalg.norm(real_velocities, axis=1))

                # Vitesses simulées
                sim_pos = np.array(self.sim_trajectories[agent_id])
                if len(sim_pos) > 1:
                    # Estimation avec pas de temps constant (0.1s)
                    sim_displacements = np.diff(sim_pos, axis=0)
                    sim_speeds.extend(np.linalg.norm(sim_displacements / 0.1, axis=1))

        result = {}
        if real_speeds and sim_speeds:
            min_len = min(len(real_speeds), len(sim_speeds))
            real_speeds_aligned = real_speeds[:min_len]
            sim_speeds_aligned = sim_speeds[:min_len]

            # Correlation
            try:
                correlation, _ = pearsonr(real_speeds_aligned, sim_speeds_aligned)
                result['speed_correlation'] = correlation
            except:
                result['speed_correlation'] = 0.0

            # Erreurs
            result['speed_mse'] = mean_squared_error(real_speeds_aligned, sim_speeds_aligned)
            result['speed_mae'] = mean_absolute_error(real_speeds_aligned, sim_speeds_aligned)

        return result

    def compute_trajectory_similarities(self) -> Dict:
        """Calcule les similarités de trajectoire (DTW simplifié)"""
        similarities = []

        for agent_id in self.real_trajectories:
            if agent_id in self.sim_trajectories:
                real_pos = np.array([[p['world_x'], p['world_y']]
                                   for p in self.real_trajectories[agent_id]])
                sim_pos = np.array(self.sim_trajectories[agent_id])

                min_len = min(len(real_pos), len(sim_pos))
                if min_len > 0:
                    # Distance euclidienne moyenne
                    distances = np.linalg.norm(real_pos[:min_len] - sim_pos[:min_len], axis=1)
                    avg_distance = np.mean(distances)

                    # Similarité (inverse de la distance, normalisée)
                    max_expected_distance = 5.0  # mètres
                    similarity = max(0, 1 - (avg_distance / max_expected_distance))
                    similarities.append(similarity)

        result = {}
        if similarities:
            result['avg_similarity'] = np.mean(similarities)
            result['similarity_std'] = np.std(similarities)
            result['similarity_min'] = np.min(similarities)
            result['similarity_max'] = np.max(similarities)

        return result

    def compare_collective_metrics(self, sim_collective_metrics: Optional[Dict] = None) -> Dict:
        """Compare les métriques collectives"""
        # Pour simplifier, nous comparons des métriques basiques
        # En production, il faudrait calculer les mêmes métriques sur les deux jeux de données

        result = {
            'note': "Les métriques collectives nécessitent le calcul sur les données simulées",
            'requires_additional_data': True
        }

        return result

    def compute_validation_score(self) -> float:
        """Calcule un score de validation global (0-1)"""
        score_components = []

        # Score basé sur les erreurs de position
        if 'position_errors' in self.metrics:
            pos_errors = self.metrics['position_errors']
            if 'rmse_mean' in pos_errors:
                # Plus RMSE est petit, meilleur est le score
                rmse_score = max(0, 1 - (pos_errors['rmse_mean'] / 2.0))  # 2m d'erreur max
                score_components.append(rmse_score * 0.4)  # 40% du score

        # Score basé sur la corrélation des vitesses
        if 'velocity_errors' in self.metrics:
            vel_errors = self.metrics['velocity_errors']
            if 'speed_correlation' in vel_errors:
                corr_score = max(0, vel_errors['speed_correlation'])
                score_components.append(corr_score * 0.3)  # 30% du score

        # Score basé sur la similarité des trajectoires
        if 'trajectory_similarities' in self.metrics:
            traj_sim = self.metrics['trajectory_similarities']
            if 'avg_similarity' in traj_sim:
                score_components.append(traj_sim['avg_similarity'] * 0.3)  # 30% du score

        if score_components:
            return np.mean(score_components)
        return 0.0

    def is_valid(self, threshold: float = VALIDATION_THRESHOLD) -> bool:
        """Détermine si la simulation est valide selon un seuil"""
        if 'validation_score' not in self.metrics:
            self.compute_all_metrics()

        return self.metrics['validation_score'] >= threshold

    def generate_validation_report(self) -> str:
        """Génère un rapport de validation détaillé"""
        if not self.metrics:
            self.compute_all_metrics()

        report = []
        report.append("=" * 60)
        report.append("📊 RAPPORT DE VALIDATION DE LA SIMULATION")
        report.append("=" * 60)
        report.append(f"\nScore de validation: {self.metrics['validation_score']:.3f}")
        report.append(f"Statut: {'✅ VALIDE' if self.is_valid() else '❌ INVALIDE'}")

        # Détails des erreurs
        if 'position_errors' in self.metrics:
            report.append("\n🔍 Erreurs de Position:")
            for key, value in self.metrics['position_errors'].items():
                report.append(f"  {key}: {value:.3f}")

        if 'velocity_errors' in self.metrics:
            report.append("\n⚡ Erreurs de Vitesse:")
            for key, value in self.metrics['velocity_errors'].items():
                report.append(f"  {key}: {value:.3f}")

        if 'trajectory_similarities' in self.metrics:
            report.append("\n🛤 Similarités de Trajectoire:")
            for key, value in self.metrics['trajectory_similarities'].items():
                report.append(f"  {key}: {value:.3f}")

        report.append("\n" + "=" * 60)
        return "\n".join(report)

# CALIBRATION HELBING

In [ ]:
# ============================================================================
# CALIBRATION HELBING
# ============================================================================

class HelbingCalibrator:
    """Calibre les paramètres du modèle de Helbing"""

    def __init__(self, real_data_loader: RealDataLoader):
        self.real_data = real_data_loader
        self.best_params = {
            'A_social': 2.0,      # Amplitude force sociale
            'B_social': 0.3,      # Portée force sociale
            'k_body': 10.0,       # Force corps à corps
            'lambda_': 0.5,       # Facteur d'anisotropie
            'tau': 0.5           # Temps de relaxation
        }
        self.validation_history = []

    def calibrate(self, iterations: int = CALIBRATION_ITERATIONS) -> Dict:
        """Calibre les paramètres par recherche adaptative"""
        print("🎯 Démarrage de la calibration Helbing...")

        best_score = -float('inf')

        for i in range(iterations):
            # Générer des paramètres candidats
            candidate_params = self._generate_candidate_params()

            # Évaluer les paramètres
            score = self._evaluate_parameters(candidate_params)

            # Mettre à jour les meilleurs paramètres
            if score > best_score:
                best_score = score
                self.best_params = candidate_params.copy()

                if i % 10 == 0:
                    print(f"  Itération {i}: Score = {score:.3f}")

            self.validation_history.append({
                'iteration': i,
                'score': score,
                'params': candidate_params.copy()
            })

        print(f"✅ Calibration terminée - Meilleur score: {best_score:.3f}")
        return self.best_params

    def _generate_candidate_params(self) -> Dict:
        """Génère des paramètres candidats intelligemment"""
        # Exploration vs exploitation
        if random.random() < 0.3:  # 30% exploration
            # Exploration aléatoire
            params = {}
            for param_name, (min_val, max_val) in HELBING_PARAMS_RANGES.items():
                params[param_name] = random.uniform(min_val, max_val)
        else:
            # Exploitation autour des meilleurs paramètres
            params = self.best_params.copy()
            for param_name in params:
                if param_name in HELBING_PARAMS_RANGES:
                    min_val, max_val = HELBING_PARAMS_RANGES[param_name]
                    # Perturbation gaussienne
                    perturbation = random.gauss(0, 0.1 * (max_val - min_val))
                    params[param_name] = np.clip(
                        params[param_name] + perturbation,
                        min_val, max_val
                    )

        return params

    def _evaluate_parameters(self, params: Dict) -> float:
        """Évalue la qualité des paramètres par simulation rapide"""
        # Créer un modèle réduit pour l'évaluation
        from copy import deepcopy

        # Simulation réduite pour évaluation rapide
        # Dans la réalité, on ferait une simulation complète mais courte
        try:
            # Score basé sur la cohérence physique
            score = self._compute_physics_score(params)

            # Ajouter un peu de bruit pour éviter les optima locaux
            score += random.uniform(-0.01, 0.01)

            return score
        except:
            return -float('inf')  # Pénalité pour les paramètres invalides

    def _compute_physics_score(self, params: Dict) -> float:
        """Calcule un score basé sur la cohérence physique"""
        score = 1.0

        # Vérifier les relations physiques attendues
        # 1. A_social devrait être positif et raisonnable
        if params['A_social'] <= 0:
            score -= 0.5

        # 2. B_social devrait être > 0
        if params['B_social'] <= 0:
            score -= 0.5

        # 3. Rapport A/B devrait être dans une plage raisonnable
        if params['B_social'] > 0:
            ratio = params['A_social'] / params['B_social']
            if ratio < 2 or ratio > 20:
                score -= 0.3

        # 4. Temps de relaxation positif
        if params['tau'] <= 0:
            score -= 0.5

        return max(0, score)

    def plot_calibration_history(self):
        """Visualise l'historique de calibration"""
        if not self.validation_history:
            print("❌ Aucune donnée d'historique disponible")
            return

        iterations = [h['iteration'] for h in self.validation_history]
        scores = [h['score'] for h in self.validation_history]

        plt.figure(figsize=(10, 6))
        plt.plot(iterations, scores, 'b-', alpha=0.7, linewidth=2)
        plt.fill_between(iterations, scores, alpha=0.2)

        # Meilleur score
        best_idx = np.argmax(scores)
        plt.scatter(iterations[best_idx], scores[best_idx],
                   color='red', s=100, zorder=5,
                   label=f'Meilleur score: {scores[best_idx]:.3f}')

        plt.xlabel('Itération de Calibration')
        plt.ylabel('Score de Validation')
        plt.title('Historique de Calibration des Paramètres Helbing')
        plt.legend()
        plt.grid(True, alpha=0.3)

        # Annoter les meilleurs paramètres
        best_params = self.validation_history[best_idx]['params']
        params_text = "\n".join([f"{k}: {v:.3f}" for k, v in best_params.items()])

        plt.figtext(0.02, 0.02, f"Meilleurs paramètres:\n{params_text}",
                   fontsize=9, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

        plt.tight_layout()
        plt.savefig('calibration_history.png', dpi=150)
        plt.show()

# MÉTRIQUES COLLECTIVES

In [ ]:
# ============================================================================
# MÉTRIQUES COLLECTIVES
# ============================================================================

class CollectiveMetricsCalculator:
    """Calcule les métriques collectives en temps réel"""

    def __init__(self):
        self.metrics_history = []

    def compute_metrics(self, agents: List, step: int) -> Dict:
        """Calcule les métriques collectives pour un pas de temps donné"""
        if not agents:
            return {}

        positions = np.array([agent.pos for agent in agents])

        metrics = {
            'step': step,
            'agent_count': len(agents),
            'density': self._compute_density(positions),
            'clustering': self._compute_clustering(positions),
            'polarization': self._compute_polarization(agents),
            'entropy': self._compute_entropy(positions),
            'avg_speed': self._compute_average_speed(agents),
            'avg_stress': self._compute_average_stress(agents)
        }

        self.metrics_history.append(metrics)
        return metrics

    def _compute_density(self, positions: np.ndarray) -> float:
        """Calcule la densité spatiale"""
        if len(positions) < 2:
            return 0.0

        # Aire convexe ou estimation par plus proches voisins
        try:
            from scipy.spatial import ConvexHull
            if len(positions) >= 3:
                hull = ConvexHull(positions)
                area = hull.volume if hasattr(hull, 'volume') else hull.area
                density = len(positions) / max(area, 0.1)
            else:
                # Estimation par distance moyenne
                distances = cdist(positions, positions)
                np.fill_diagonal(distances, np.inf)
                avg_distance = np.min(distances, axis=1).mean()
                density = 1.0 / (avg_distance ** 2 + 0.1)
        except:
            # Fallback simple
            bounds = np.ptp(positions, axis=0)
            area = bounds[0] * bounds[1] if bounds[0] > 0 and bounds[1] > 0 else 1.0
            density = len(positions) / area

        return float(density)

    def _compute_clustering(self, positions: np.ndarray, radius: float = 2.0) -> float:
        """Calcule le coefficient de clustering"""
        if len(positions) < 3:
            return 0.0

        # Construction du graphe de proximité
        G = nx.Graph()
        for i, pos in enumerate(positions):
            G.add_node(i, pos=pos)

        # Ajouter les arêtes pour les paires proches
        for i in range(len(positions)):
            for j in range(i + 1, len(positions)):
                distance = np.linalg.norm(positions[i] - positions[j])
                if distance < radius:
                    G.add_edge(i, j, weight=distance)

        # Coefficient de clustering moyen
        if G.number_of_edges() > 0:
            try:
                clustering = nx.average_clustering(G)
                return float(clustering)
            except:
                return 0.0
        return 0.0

    def _compute_polarization(self, agents: List) -> float:
        """Calcule la polarisation des directions"""
        if not agents:
            return 0.0

        velocities = []
        for agent in agents:
            if hasattr(agent, 'velocity') and np.linalg.norm(agent.velocity) > 0:
                velocities.append(agent.velocity / np.linalg.norm(agent.velocity))

        if len(velocities) < 2:
            return 0.0

        velocities_array = np.array(velocities)
        # Polarisation = norme de la moyenne des vecteurs unitaires
        polarization = np.linalg.norm(np.mean(velocities_array, axis=0))
        return float(polarization)

    def _compute_entropy(self, positions: np.ndarray, grid_size: float = 0.5) -> float:
        """Calcule l'entropie spatiale (désordre)"""
        if len(positions) < 2:
            return 0.0

        # Discrétisation spatiale
        x_min, y_min = np.min(positions, axis=0)
        x_max, y_max = np.max(positions, axis=0)

        if x_max - x_min < grid_size or y_max - y_min < grid_size:
            return 0.0

        x_bins = int((x_max - x_min) / grid_size) + 1
        y_bins = int((y_max - y_min) / grid_size) + 1

        # Histogramme 2D
        hist = np.zeros((x_bins, y_bins))
        for x, y in positions:
            i = min(int((x - x_min) / grid_size), x_bins - 1)
            j = min(int((y - y_min) / grid_size), y_bins - 1)
            hist[i, j] += 1

        # Normalisation
        hist_flat = hist.flatten()
        hist_flat = hist_flat[hist_flat > 0]
        if len(hist_flat) < 2:
            return 0.0

        prob = hist_flat / np.sum(hist_flat)

        # Entropie de Shannon
        entropy = -np.sum(prob * np.log2(prob))

        # Normalisation par rapport à l'entropie maximale
        max_entropy = np.log2(len(prob))
        if max_entropy > 0:
            entropy_normalized = entropy / max_entropy
        else:
            entropy_normalized = 0.0

        return float(entropy_normalized)

    def _compute_average_speed(self, agents: List) -> float:
        """Calcule la vitesse moyenne"""
        speeds = []
        for agent in agents:
            if hasattr(agent, 'velocity'):
                speed = np.linalg.norm(agent.velocity)
                speeds.append(speed)

        return float(np.mean(speeds)) if speeds else 0.0

    def _compute_average_stress(self, agents: List) -> float:
        """Calcule le stress moyen"""
        stresses = []
        for agent in agents:
            if hasattr(agent, 'stress'):
                stresses.append(agent.stress)

        return float(np.mean(stresses)) if stresses else 0.0

    def plot_metrics_evolution(self, output_file: str = "collective_metrics.png"):
        """Visualise l'évolution des métriques collectives"""
        if len(self.metrics_history) < 2:
            print("❌ Pas assez de données pour la visualisation")
            return

        steps = [m['step'] for m in self.metrics_history]

        fig, axes = plt.subplots(3, 2, figsize=(15, 12))

        # Densité
        axes[0, 0].plot(steps, [m['density'] for m in self.metrics_history],
                       'b-', linewidth=2)
        axes[0, 0].set_title('Évolution de la Densité')
        axes[0, 0].set_ylabel('Densité (agents/m²)')
        axes[0, 0].grid(True, alpha=0.3)

        # Clustering
        axes[0, 1].plot(steps, [m['clustering'] for m in self.metrics_history],
                       'g-', linewidth=2)
        axes[0, 1].set_title('Évolution du Clustering')
        axes[0, 1].set_ylabel('Coefficient de Clustering')
        axes[0, 1].grid(True, alpha=0.3)

        # Polarisation
        axes[1, 0].plot(steps, [m['polarization'] for m in self.metrics_history],
                       'r-', linewidth=2)
        axes[1, 0].set_title('Évolution de la Polarisation')
        axes[1, 0].set_ylabel('Polarisation')
        axes[1, 0].grid(True, alpha=0.3)

        # Entropie
        axes[1, 1].plot(steps, [m['entropy'] for m in self.metrics_history],
                       'purple', linewidth=2)
        axes[1, 1].set_title('Évolution de l\'Entropie Spatiale')
        axes[1, 1].set_ylabel('Entropie (normalisée)')
        axes[1, 1].grid(True, alpha=0.3)

        # Vitesse moyenne
        axes[2, 0].plot(steps, [m['avg_speed'] for m in self.metrics_history],
                       'orange', linewidth=2)
        axes[2, 0].set_title('Évolution de la Vitesse Moyenne')
        axes[2, 0].set_ylabel('Vitesse (m/s)')
        axes[2, 0].grid(True, alpha=0.3)

        # Stress moyen
        axes[2, 1].plot(steps, [m['avg_stress'] for m in self.metrics_history],
                       'brown', linewidth=2)
        axes[2, 1].set_title('Évolution du Stress Moyen')
        axes[2, 1].set_ylabel('Stress (0-1)')
        axes[2, 1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig(output_file, dpi=150, bbox_inches='tight')
        plt.close()

        print(f"📈 Graphique des métriques sauvegardé: {output_file}")

# AGENT MESA - VERSION AMÉLIORÉE

In [ ]:
# ============================================================================
# AGENT MESA - VERSION AMÉLIORÉE
# ============================================================================

class PersonAgent(mesa.Agent):
    """Agent représentant une personne avec comportement réaliste et graphe de proximité"""

    def __init__(self, unique_id: int, model, parameters: Dict, helbing_params: Dict):
        super().__init__(unique_id, model)

        # Paramètres individuels (issus des données réelles)
        self.agent_type = parameters['agent_type']
        self.sociability = parameters['sociability']
        self.following_tendency = parameters['following_tendency']
        self.patience = parameters['patience']
        self.panic_level = parameters['panic_level']
        self.awareness_radius = parameters['awareness_radius']
        self.reaction_time = parameters['reaction_time']
        self.desired_speed = parameters['avg_velocity']
        self.max_speed = parameters['max_velocity']

        # Paramètres Helbing calibrés
        self.A_social = helbing_params['A_social']
        self.B_social = helbing_params['B_social']
        self.k_body = helbing_params['k_body']
        self.lambda_ = helbing_params['lambda_']
        self.tau = helbing_params['tau']

        # État dynamique
        self.state = AgentState.NORMAL
        self.stress = 0.0
        self.fatigue = 0.0
        self.target_exit = None
        self.leader = None
        self.followers = []

        # Graphe de proximité dynamique
        self.proximity_graph = nx.Graph()
        self.neighbors_history = []

        # Historique
        self.trajectory = []
        self.velocity_history = []
        self.state_history = []
        self.neighbor_count_history = []

        # Statistiques
        self.distance_traveled = 0.0
        self.collisions_count = 0
        self.state_changes = 0
        self.evacuated = False
        self.evacuation_time = None

        # Initialiser la position
        initial_pos = parameters['initial_position']
        self.model.space.place_agent(self, tuple(initial_pos))
        self.trajectory.append(tuple(initial_pos))

        # Initialiser la vitesse
        initial_velocity = np.array(parameters['avg_direction']) * self.desired_speed
        self.velocity = initial_velocity
        self.velocity_history.append(initial_velocity.copy())

        # Métriques collectives héritées
        self.collective_metrics = parameters.get('collective_metrics', {})

        if model.verbose:
            print(f"👤 Agent {unique_id} créé ({self.agent_type.value}) à {initial_pos}")

    def step(self):
        """Étape de simulation pour l'agent"""
        if self.evacuated:
            return

        # Mettre à jour le graphe de proximité
        self._update_proximity_graph()

        # Mettre à jour l'état
        self._update_state()

        # Calculer les forces sociales avec paramètres calibrés
        social_force = self._calculate_social_force()
        obstacle_force = self._calculate_obstacle_force()
        exit_force = self._calculate_exit_force()

        # Combiner les forces avec pondérations
        total_force = (
            self.A_social * social_force +
            2.0 * obstacle_force +
            self.lambda_ * exit_force
        )

        # Mettre à jour le mouvement
        self._update_movement(total_force)

        # Mettre à jour l'historique
        self.trajectory.append(self.pos)
        self.velocity_history.append(self.velocity.copy())
        self.state_history.append(self.state)
        self.neighbor_count_history.append(len(self.proximity_graph.nodes()) - 1 if self.proximity_graph else 0)

        # Vérifier les sorties
        self._check_exit()

    def _update_proximity_graph(self):
        """Met à jour le graphe de proximité dynamique"""
        # Réinitialiser le graphe
        self.proximity_graph.clear()
        self.proximity_graph.add_node(self.unique_id,
                                    pos=self.pos,
                                    state=self.state.value,
                                    agent_type=self.agent_type.value)

        # Trouver les voisins
        neighbors = self.model.space.get_neighbors(
            self.pos,
            self.awareness_radius,
            include_center=False
        )

        for neighbor in neighbors:
            if hasattr(neighbor, 'evacuated') and neighbor.evacuated:
                continue

            distance = np.linalg.norm(np.array(self.pos) - np.array(neighbor.pos))

            # Ajouter au graphe si dans le rayon de proximité
            if distance < self.awareness_radius:
                self.proximity_graph.add_node(neighbor.unique_id,
                                            pos=neighbor.pos,
                                            state=neighbor.state.value,
                                            agent_type=neighbor.agent_type.value)

                self.proximity_graph.add_edge(self.unique_id, neighbor.unique_id,
                                            weight=distance,
                                            type='proximity')

                # Influence sociale basée sur le graphe
                if distance < 1.0:  # Très proche
                    # Contagion de panique
                    if neighbor.state == AgentState.PANIC and self.state != AgentState.PANIC:
                        self.panic_level = min(1.0, self.panic_level + 0.05)

                    # Synchronisation de direction
                    if random.random() < self.sociability:
                        # Influence légère sur la direction
                        neighbor_dir = neighbor.velocity / np.linalg.norm(neighbor.velocity) \
                                     if np.linalg.norm(neighbor.velocity) > 0 else np.array([1.0, 0.0])
                        self_dir = self.velocity / np.linalg.norm(self.velocity) \
                                 if np.linalg.norm(self.velocity) > 0 else np.array([1.0, 0.0])

                        # Mélange des directions
                        alpha = 0.1  # Force d'influence
                        new_dir = (1 - alpha) * self_dir + alpha * neighbor_dir
                        if np.linalg.norm(new_dir) > 0:
                            self.velocity = self.velocity / np.linalg.norm(self.velocity) * np.linalg.norm(self.velocity)

    def _update_state(self):
        """Met à jour l'état comportemental de l'agent"""
        # Calculer la densité locale à partir du graphe
        local_density = len(self.proximity_graph.nodes()) / (np.pi * self.awareness_radius**2)

        # Mettre à jour le stress
        density_stress = min(1.0, local_density / 2.0)
        self.stress = (
            0.6 * density_stress +
            0.3 * self.panic_level +
            0.1 * self.fatigue
        )

        # Déterminer le nouvel état
        old_state = self.state

        if self.stress > 0.7:
            self.state = AgentState.PANIC
            self.panic_level = min(1.0, self.panic_level + 0.05)
        elif local_density > 1.5 and self.sociability > 0.6:
            # Vérifier si des voisins sont dans le même état
            neighbor_states = [self.proximity_graph.nodes[n].get('state')
                             for n in self.proximity_graph.nodes() if n != self.unique_id]
            if neighbor_states and 'grouping' in neighbor_states:
                self.state = AgentState.GROUPING
        elif self.target_exit is not None:
            self.state = AgentState.EVACUATING
        elif local_density < 0.3:
            self.state = AgentState.DISPERSING
        elif np.linalg.norm(self.velocity) < 0.1:
            self.state = AgentState.STOPPED
        else:
            self.state = AgentState.NORMAL

        if old_state != self.state:
            self.state_changes += 1
            self.model.state_changes.append({
                'step': self.model.schedule.steps,
                'agent_id': self.unique_id,
                'old_state': old_state.value,
                'new_state': self.state.value,
                'stress': self.stress,
                'density': local_density
            })

    def _calculate_social_force(self) -> np.ndarray:
        """Calcule la force sociale (modèle de Helbing calibré)"""
        total_force = np.zeros(2)
        neighbors = list(self.proximity_graph.nodes())

        for neighbor_id in neighbors:
            if neighbor_id == self.unique_id:
                continue

            neighbor = self.model.get_agent_by_id(neighbor_id)
            if not neighbor or (hasattr(neighbor, 'evacuated') and neighbor.evacuated):
                continue

            # Vecteur entre les agents
            diff = np.array(self.pos) - np.array(neighbor.pos)
            distance = np.linalg.norm(diff)

            if distance > 0:
                direction = diff / distance

                # Force de répulsion sociale (Helbing)
                social_force = self.A_social * np.exp(-distance / self.B_social) * direction

                # Force de compression physique (corps à corps)
                if distance < 0.5:  # Distance de contact
                    compression = max(0, 0.5 - distance)
                    body_force = self.k_body * compression * direction
                    social_force += body_force
                    self.collisions_count += 0.1

                # Anisotropie (l'agent est plus sensible devant)
                if np.linalg.norm(self.velocity) > 0:
                    forward = self.velocity / np.linalg.norm(self.velocity)
                    cos_angle = np.dot(direction, forward)
                    anisotropy = self.lambda_ + (1 - self.lambda_) * (1 + cos_angle) / 2
                    social_force *= anisotropy

                total_force += social_force

        return total_force

    def _calculate_obstacle_force(self) -> np.ndarray:
        """Calcule la force de répulsion des obstacles"""
        total_force = np.zeros(2)

        for obstacle in self.model.obstacles:
            ox, oy, ow, oh = obstacle

            # Trouver le point le plus proche sur l'obstacle
            closest_x = max(ox, min(self.pos[0], ox + ow))
            closest_y = max(oy, min(self.pos[1], oy + oh))

            diff = np.array([self.pos[0] - closest_x, self.pos[1] - closest_y])
            distance = np.linalg.norm(diff)

            if distance > 0:
                direction = diff / distance
                # Force de répulsion (inverse carré)
                force_strength = min(5.0, 1.0 / (distance**2 + 0.1))
                total_force += force_strength * direction

        return total_force

    def _calculate_exit_force(self) -> np.ndarray:
        """Calcule la force d'attraction vers les sorties"""
        if self.target_exit is None:
            # Trouver la sortie la plus proche
            exits = self.model.exits
            if not exits:
                return np.zeros(2)

            min_dist = float('inf')
            for exit in exits:
                dist = np.linalg.norm(np.array(self.pos) - np.array([exit[0], exit[1]]))
                if dist < min_dist:
                    min_dist = dist
                    self.target_exit = exit

        if self.target_exit:
            exit_pos = np.array([self.target_exit[0], self.target_exit[1]])
            direction = exit_pos - np.array(self.pos)
            distance = np.linalg.norm(direction)

            if distance > 0:
                # Force plus forte quand plus proche
                strength = min(2.0, 5.0 / (distance + 1.0))
                return strength * (direction / distance)

        return np.zeros(2)

    def _update_movement(self, total_force: np.ndarray):
        """Met à jour la position et la vitesse de l'agent avec le modèle de Helbing"""
        # Direction souhaitée
        if self.state == AgentState.PANIC:
            # Direction aléatoire dans la panique
            angle = random.uniform(0, 2 * np.pi)
            desired_direction = np.array([np.cos(angle), np.sin(angle)])
        elif self.state == AgentState.EVACUATING and self.target_exit:
            # Vers la sortie
            exit_pos = np.array([self.target_exit[0], self.target_exit[1]])
            direction = exit_pos - np.array(self.pos)
            if np.linalg.norm(direction) > 0:
                desired_direction = direction / np.linalg.norm(direction)
        else:
            # Direction basée sur la force sociale
            if np.linalg.norm(total_force) > 0:
                desired_direction = total_force / np.linalg.norm(total_force)
            else:
                if np.linalg.norm(self.velocity) > 0:
                    desired_direction = self.velocity / np.linalg.norm(self.velocity)
                else:
                    desired_direction = np.array([1.0, 0.0])

        # Vitesse souhaitée
        desired_speed = self.desired_speed
        if self.state == AgentState.PANIC:
            desired_speed *= 1.5
        elif self.state == AgentState.EVACUATING:
            desired_speed *= 1.2
        elif self.state == AgentState.STOPPED:
            desired_speed = 0.0

        # Équation d'évolution de Helbing
        desired_velocity = desired_speed * desired_direction

        # Terme de relaxation vers la vitesse souhaitée
        relaxation_term = (desired_velocity - self.velocity) / self.tau

        # Accélération totale
        acceleration = relaxation_term + total_force

        # Limiter l'accélération
        max_acceleration = 5.0
        acc_norm = np.linalg.norm(acceleration)
        if acc_norm > max_acceleration:
            acceleration = acceleration / acc_norm * max_acceleration

        # Mettre à jour la vitesse
        self.velocity += acceleration * self.model.time_step

        # Limiter la vitesse
        max_speed = self.max_speed * (1.0 + 0.5 * self.panic_level)
        speed = np.linalg.norm(self.velocity)
        if speed > max_speed:
            self.velocity = self.velocity / speed * max_speed

        # Nouvelle position
        new_pos = np.array(self.pos) + self.velocity * self.model.time_step

        # Vérifier les collisions avec les obstacles
        collision = self._check_obstacle_collision(new_pos)
        if collision:
            # Rebondir avec amortissement
            self.velocity *= -0.5
            self.collisions_count += 1
        else:
            # Mettre à jour la position
            self.model.space.move_agent(self, tuple(new_pos))
            self.distance_traveled += speed * self.model.time_step

    def _check_obstacle_collision(self, position: np.ndarray) -> bool:
        """Vérifie si une position entre en collision avec un obstacle"""
        for obstacle in self.model.obstacles:
            ox, oy, ow, oh = obstacle
            if (ox <= position[0] <= ox + ow and
                oy <= position[1] <= oy + oh):
                return True
        return False

    def _check_exit(self):
        """Vérifie si l'agent a atteint une sortie"""
        if self.target_exit is None:
            return

        exit_pos = np.array([self.target_exit[0], self.target_exit[1]])
        distance = np.linalg.norm(np.array(self.pos) - exit_pos)

        if distance < 0.5:  # À portée de la sortie
            self.evacuated = True
            self.model.evacuated_count += 1
            self.evacuation_time = self.model.schedule.steps * self.model.time_step

            if self.model.verbose:
                print(f"🚪 Agent {self.unique_id} évacué en {self.evacuation_time:.1f}s!")

# MODÈLE MESA - VERSION AMÉLIORÉE

In [ ]:
# ============================================================================
# MODÈLE MESA - VERSION AMÉLIORÉE
# ============================================================================

class CrowdModel(mesa.Model):
    """Modèle de foule multi-agents avec validation, calibration et métriques collectives"""

    def __init__(self,
                 real_data_loader: RealDataLoader,
                 width: float = 15.0,
                 height: float = 12.0,
                 time_step: float = 0.1,
                 what_if_scenario: str = "normal",
                 scenario_intensity: float = 0.5,
                 max_agents: int = 50,
                 verbose: bool = True,
                 validation_mode: bool = False):

        super().__init__()

        # Paramètres de simulation
        self.width = width
        self.height = height
        self.time_step = time_step
        self.what_if_scenario = what_if_scenario
        self.scenario_intensity = scenario_intensity
        self.max_agents = max_agents
        self.verbose = verbose
        self.validation_mode = validation_mode

        # Espace continu
        self.space = mesa.space.ContinuousSpace(width, height, True)

        # Planificateur
        self.schedule = mesa.time.RandomActivation(self)

        # Obstacles (x, y, largeur, hauteur)
        self.obstacles = [
            (0, 0, 15, 0.2),      # Mur bas
            (0, 11.8, 15, 0.2),   # Mur haut
            (0, 0, 0.2, 12),      # Mur gauche
            (14.8, 0, 0.2, 12),   # Mur droit
            (5, 4, 2, 1),         # Obstacle central
            (10, 7, 1, 3)         # Colonne
        ]

        # Sorties (x, y, largeur)
        self.exits = [
            (7.5, 0, 1.5),    # Sortie bas
            (7.5, 12, 1.5),   # Sortie haut
            (0, 6, 1.0),      # Sortie gauche
            (15, 6, 1.0)      # Sortie droite
        ]

        # Métriques
        self.evacuated_count = 0
        self.state_changes = []
        self.global_metrics = []
        self.collective_metrics_history = []

        # Systèmes de validation et calibration
        self.real_data_loader = real_data_loader
        self.collective_calculator = CollectiveMetricsCalculator()
        self.validation_metrics = None
        self.is_validated = False

        # Paramètres Helbing (par défaut, seront calibrés)
        self.helbing_params = {
            'A_social': 2.0,
            'B_social': 0.3,
            'k_body': 10.0,
            'lambda_': 0.5,
            'tau': 0.5
        }

        # Calibration si demandée
        if not validation_mode:
            self._calibrate_helbing()

        # Charger les données réelles
        trajectories = real_data_loader.load_real_trajectories()

        # Créer les agents à partir des données réelles
        self._create_agents_from_real_data(trajectories)

        # Appliquer le scénario what-if
        self._apply_what_if_scenario()

        # Collecteur de données
        self.datacollector = mesa.DataCollector(
            model_reporters={
                "Active Agents": lambda m: sum(1 for a in m.schedule.agents if not a.evacuated),
                "Evacuated Agents": "evacuated_count",
                "Average Speed": lambda m: np.mean([
                    np.linalg.norm(a.velocity) for a in m.schedule.agents if not a.evacuated
                ]) if any(not a.evacuated for a in m.schedule.agents) else 0,
                "Average Stress": lambda m: np.mean([
                    a.stress for a in m.schedule.agents if not a.evacuated
                ]) if any(not a.evacuated for a in m.schedule.agents) else 0,
                "Average Panic": lambda m: np.mean([
                    a.panic_level for a in m.schedule.agents if not a.evacuated
                ]) if any(not a.evacuated for a in m.schedule.agents) else 0,
                "Clustering Coefficient": lambda m: np.mean([
                    nx.average_clustering(a.proximity_graph)
                    for a in m.schedule.agents if not a.evacuated and a.proximity_graph
                ]) if any(not a.evacuated for a in m.schedule.agents) else 0,
            },
            agent_reporters={
                "State": lambda a: a.state.value,
                "Panic Level": "panic_level",
                "Speed": lambda a: np.linalg.norm(a.velocity),
                "Distance Traveled": "distance_traveled",
                "Neighbor Count": lambda a: len(a.proximity_graph.nodes()) - 1 if a.proximity_graph else 0
            }
        )

        if self.verbose:
            print(f"✅ Modèle MESA initialisé avec {self.schedule.get_agent_count()} agents")
            print(f"🎭 Scénario: {what_if_scenario} (intensité: {scenario_intensity})")
            print(f"🎯 Paramètres Helbing: {self.helbing_params}")

    def _calibrate_helbing(self):
        """Calibre les paramètres de Helbing"""
        if self.validation_mode:
            # En mode validation, on utilise des paramètres fixes pour comparaison
            print("⚙ Mode validation - Pas de calibration")
            return

        print("🎯 Calibration des paramètres Helbing...")
        calibrator = HelbingCalibrator(self.real_data_loader)
        self.helbing_params = calibrator.calibrate(iterations=CALIBRATION_ITERATIONS)

        # Visualiser l'historique de calibration
        calibrator.plot_calibration_history()

    def _create_agents_from_real_data(self, trajectories: Dict):
        """Crée des agents à partir des trajectoires réelles"""
        agent_id = 0

        for real_agent_id, traj in trajectories.items():
            # Extraire les paramètres de l'agent réel
            params = self.real_data_loader.extract_agent_parameters(real_agent_id, traj)

            if params is None:
                continue

            # Créer l'agent MESA avec paramètres calibrés
            agent = PersonAgent(agent_id, self, params, self.helbing_params)
            self.schedule.add(agent)

            agent_id += 1

            # Limiter le nombre d'agents pour la performance
            if agent_id >= self.max_agents:
                if self.verbose:
                    print(f"⚠ Limite d'agents atteinte: {self.max_agents}")
                break

        if self.verbose:
            print(f"👥 {agent_id} agents créés à partir de données réelles")

    def _apply_what_if_scenario(self):
        if self.what_if_scenario == "panic":
            self._apply_panic_scenario()
        elif self.what_if_scenario == "obstacle_blocked":
            self._apply_obstacle_blocked_scenario()
        elif self.what_if_scenario == "high_density":
            self._apply_high_density_scenario()
        elif self.what_if_scenario == "leader_emergency":
            self._apply_leader_emergency_scenario()

    def _apply_panic_scenario(self):
        """Scénario de panique"""
        for agent in self.schedule.agents:
            agent.panic_level = min(1.0, agent.panic_level + self.scenario_intensity)
            if self.scenario_intensity > 0.7:
                agent.state = AgentState.PANIC

    def _apply_obstacle_blocked_scenario(self):
        """Scénario de sorties bloquées"""
        # Bloquer certaines sorties
        exit_count = len(self.exits)
        blocked_count = int(exit_count * self.scenario_intensity)
        self.exits = self.exits[blocked_count:]

    def _apply_high_density_scenario(self):
        """Scénario de haute densité"""
        # Ajouter des agents synthétiques
        current_count = self.schedule.get_agent_count()
        new_agents = int(current_count * self.scenario_intensity)

        for i in range(new_agents):
            agent_id = current_count + i

            # Position aléatoire
            pos = (random.uniform(1, 14), random.uniform(1, 11))

            # Paramètres synthétiques
            params = {
                'agent_id': agent_id,
                'initial_position': pos,
                'avg_velocity': random.uniform(0.8, 1.2),
                'max_velocity': random.uniform(1.2, 1.8),
                'avg_direction': np.array([random.uniform(-1, 1), random.uniform(-1, 1)]),
                'straightness': random.uniform(0.3, 0.7),
                'direction_variability': random.uniform(0.5, 1.5),
                'agent_type': AgentType.PEDESTRIAN,
                'sociability': random.uniform(0.3, 0.6),
                'following_tendency': random.uniform(0.3, 0.5),
                'patience': random.uniform(0.4, 0.8),
                'panic_level': random.uniform(0.1, 0.3),
                'awareness_radius': random.uniform(2.0, 4.0),
                'reaction_time': random.uniform(0.2, 0.4)
            }

            # Normaliser la direction
            if np.linalg.norm(params['avg_direction']) > 0:
                params['avg_direction'] = params['avg_direction'] / np.linalg.norm(params['avg_direction'])

            # Ajouter les métriques collectives
            params['collective_metrics'] = self.real_data_loader.collective_metrics

            # Créer l'agent
            agent = PersonAgent(agent_id, self, params, self.helbing_params)
            self.schedule.add(agent)
            self.space.place_agent(agent, pos)

    def _apply_leader_emergency_scenario(self):
        """Scénario d'évacuation avec leaders"""
        leaders = [a for a in self.schedule.agents if a.agent_type == AgentType.LEADER]

        for leader in leaders:
            leader.state = AgentState.EVACUATING

            # Faire suivre les agents proches
            for agent in self.schedule.agents:
                if agent.unique_id != leader.unique_id:
                    dist = np.linalg.norm(np.array(agent.pos) - np.array(leader.pos))
                    if dist < 3.0:
                        agent.leader = leader
                        agent.following_tendency = 1.0
                        agent.state = AgentState.FOLLOWING
    def get_agent_by_id(self, agent_id: int):
        """Retourne un agent par son ID"""
        for agent in self.schedule.agents:
            if agent.unique_id == agent_id:
                return agent
        return None

    def validate_simulation(self, max_steps: int = 100) -> Dict:
        """Valide la simulation contre les données réelles"""
        print("🔬 Démarrage de la validation...")

        # Exécuter la simulation pour validation
        for step in range(max_steps):
            self.step()

            if step % 20 == 0 and self.verbose:
                active = sum(1 for a in self.schedule.agents if not a.evacuated)
                print(f"  Validation step {step}/{max_steps} - Agents actifs: {active}")

        # Collecter les trajectoires simulées
        simulated_trajectories = {}
        for agent in self.schedule.agents:
            if hasattr(agent, 'trajectory'):
                simulated_trajectories[agent.unique_id] = agent.trajectory

        # Calculer les métriques de validation
        self.validation_metrics = ValidationMetrics(
            self.real_data_loader.trajectories_data,
            simulated_trajectories,
            fps=self.real_data_loader.fps
        )

        validation_results = self.validation_metrics.compute_all_metrics()
        self.is_validated = self.validation_metrics.is_valid()

        # Générer le rapport
        report = self.validation_metrics.generate_validation_report()
        print(report)

        if self.is_validated:
            print("✅ Simulation validée avec succès!")
        else:
            print("⚠ Simulation non validée - Considérer recalibration")

        return validation_results

    def step(self):
        """Exécute une étape de simulation"""
        self.schedule.step()

        # Collecter les données
        self.datacollector.collect(self)

        # Calculer les métriques collectives
        active_agents = [a for a in self.schedule.agents if not a.evacuated]
        collective_metrics = self.collective_calculator.compute_metrics(
            active_agents, self.schedule.steps
        )

        if collective_metrics:
            self.collective_metrics_history.append(collective_metrics)

        # Calculer les métriques globales
        self._compute_global_metrics()

        # Vérifier la fin de la simulation
        active_agents_count = len(active_agents)
        if active_agents_count == 0 and self.verbose:
            print("✅ Tous les agents sont évacués!")

    def _compute_global_metrics(self):
        """Calcule les métriques globales"""
        active_agents = [a for a in self.schedule.agents if not a.evacuated]

        if not active_agents:
            return

        # Métriques de base
        metrics = {
            'step': self.schedule.steps,
            'time': self.schedule.steps * self.time_step,
            'active_agents': len(active_agents),
            'average_speed': np.mean([np.linalg.norm(a.velocity) for a in active_agents]),
            'average_stress': np.mean([a.stress for a in active_agents]),
            'average_panic': np.mean([a.panic_level for a in active_agents]),
            'state_distribution': {},
            'collisions_total': sum([a.collisions_count for a in active_agents])
        }

        # Distribution des états
        for state in AgentState:
            count = sum(1 for a in active_agents if a.state == state)
            metrics['state_distribution'][state.value] = count

        # Métriques de graphe (si disponibles)
        if hasattr(active_agents[0], 'proximity_graph'):
            try:
                all_graphs = [a.proximity_graph for a in active_agents if a.proximity_graph]
                if all_graphs:
                    # Construire un graphe global
                    global_graph = nx.Graph()
                    for graph in all_graphs:
                        global_graph = nx.compose(global_graph, graph)

                    if global_graph.number_of_nodes() > 0:
                        metrics['global_clustering'] = nx.average_clustering(global_graph)
                        metrics['connected_components'] = nx.number_connected_components(global_graph)
                        metrics['avg_degree'] = np.mean([d for _, d in global_graph.degree()])
            except:
                pass

        self.global_metrics.append(metrics)

    def run_simulation(self, steps: int = 500, validate_first: bool = True):
        """Exécute la simulation complète avec validation optionnelle"""
        if self.verbose:
            print(f"\n🚀 Démarrage simulation MESA ({steps} steps)")

        # Phase de validation si demandée
        if validate_first and not self.validation_mode:
            print("\n📋 PHASE DE VALIDATION")
            validation_results = self.validate_simulation(max_steps=min(100, steps//2))

            if not self.is_validated:
                print("⚠ Attention: Simulation non validée, résultats à interpréter avec prudence")

            # Réinitialiser pour la simulation complète
            self.evacuated_count = 0
            for agent in self.schedule.agents:
                agent.evacuated = False
                agent.trajectory = [agent.trajectory[0]] if agent.trajectory else []
                agent.velocity_history = [agent.velocity_history[0]] if agent.velocity_history else []

        # Simulation principale
        print("\n🎯 SIMULATION PRINCIPALE")
        for step in range(steps):
            if self.verbose and step % 50 == 0:
                active = sum(1 for a in self.schedule.agents if not a.evacuated)
                print(f"  Step {step}/{steps} - Agents actifs: {active}")

            self.step()

            # Arrêter si tous les agents sont évacués
            if self.evacuated_count >= self.schedule.get_agent_count():
                if self.verbose:
                    print(f"✅ Simulation terminée à l'étape {step}")
                break

        # Générer les visualisations collectives
        if self.collective_metrics_history:
            self.collective_calculator.plot_metrics_evolution()

        if self.verbose:
            print(f"\n📊 Simulation terminée:")
            print(f"   • Étapes: {self.schedule.steps}")
            print(f"   • Temps total: {self.schedule.steps * self.time_step:.1f}s")
            print(f"   • Agents évacués: {self.evacuated_count}/{self.schedule.get_agent_count()}")
            print(f"   • Taux d'évacuation: {(self.evacuated_count/self.schedule.get_agent_count())*100:.1f}%")

        return self.get_simulation_results()

    def get_simulation_results(self):
        """Récupère les résultats de la simulation"""
        # Données des agents
        agent_data = []
        for agent in self.schedule.agents:
            agent_data.append({
                'id': agent.unique_id,
                'type': agent.agent_type.value,
                'final_state': agent.state.value,
                'panic_level': agent.panic_level,
                'stress': agent.stress,
                'distance_traveled': agent.distance_traveled,
                'collisions': agent.collisions_count,
                'state_changes': agent.state_changes,
                'evacuated': agent.evacuated,
                'evacuation_time': agent.evacuation_time,
                'final_position': agent.pos if hasattr(agent, 'pos') else None,
                'neighbor_count_history': agent.neighbor_count_history if hasattr(agent, 'neighbor_count_history') else []
            })

        # Données du modèle
        model_df = self.datacollector.get_model_vars_dataframe()
        agent_df = self.datacollector.get_agent_vars_dataframe()

        return {
            'agent_data': agent_data,
            'model_data': model_df,
            'agent_vars_data': agent_df,
            'global_metrics': self.global_metrics,
            'collective_metrics': self.collective_metrics_history,
            'state_changes': self.state_changes,
            'validation_metrics': self.validation_metrics.metrics if self.validation_metrics else None,
            'is_validated': self.is_validated,
            'helbing_params': self.helbing_params
        }

# GÉNÉRATION DE VIDÉO - VERSION AMÉLIORÉE

In [ ]:
# ============================================================================
# GÉNÉRATION DE VIDÉO - VERSION AMÉLIORÉE
# ============================================================================

class VideoGenerator:
    """Génère une vidéo de la simulation avec indicateurs de scénario"""

    def __init__(self, model, steps: int = 200):
        self.model = model
        self.steps = steps
        self.frames = []

    def capture_simulation(self):
        """Capture chaque étape de la simulation pour créer la vidéo"""
        print("🎥 Capture de la simulation en cours...")

        # Sauvegarder l'état initial
        initial_state = self.model.schedule.steps

        # Exécuter la simulation et capturer chaque étape
        for step in range(self.steps):
            # Avancer d'une étape
            self.model.step()

            # Créer une image de l'état actuel
            fig = self._create_frame(step)

            # Convertir la figure en image
            buf = io.BytesIO()
            fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
            buf.seek(0)
            self.frames.append(Image.open(buf))

            plt.close(fig)

            if step % 20 == 0:
                active_agents = sum(1 for a in self.model.schedule.agents if not a.evacuated)
                print(f"  Frame {step}/{self.steps} - Agents actifs: {active_agents}")

            # Arrêter si tous les agents sont évacués
            if self.model.evacuated_count >= self.model.schedule.get_agent_count():
                print(f"✅ Tous les agents évacués à l'étape {step}")
                break

        print(f"✅ {len(self.frames)} frames capturées")

    def _create_frame(self, step: int):
        """Crée une frame de la simulation avec indicateurs de scénario"""
        fig, ax = plt.subplots(figsize=(14, 10))

        # Définir les limites
        ax.set_xlim(0, self.model.width)
        ax.set_ylim(0, self.model.height)
        ax.set_aspect('equal')
        ax.set_facecolor('#f5f5f5')
        ax.grid(True, alpha=0.3, linestyle='--')

        # Titre avec scénario en évidence
        scenario_colors = {
            'normal': 'blue',
            'panic': 'red',
            'obstacle_blocked': 'orange',
            'high_density': 'purple',
            'leader_emergency': 'green'
        }

        scenario_color = scenario_colors.get(self.model.what_if_scenario, 'black')
        title = f"SCÉNARIO: {self.model.what_if_scenario.upper()}"
        ax.text(0.5, 1.02, title, transform=ax.transAxes,
               fontsize=16, fontweight='bold', color=scenario_color,
               ha='center', va='bottom',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor=scenario_color))

        # Dessiner les obstacles
        for obstacle in self.model.obstacles:
            x, y, w, h = obstacle
            rect = patches.Rectangle((x, y), w, h,
                                   facecolor='#2c3e50', alpha=0.8,
                                   edgecolor='#34495e', linewidth=2)
            ax.add_patch(rect)

        # Dessiner les sorties (avec indication si bloquées)
        for i, exit_pos in enumerate(self.model.exits):
            x, y, w = exit_pos
            color = '#27ae60'  # Vert normal
            alpha = 0.8

            # Si scénario "obstacle_blocked", marquer certaines sorties
            if self.model.what_if_scenario == "obstacle_blocked" and i < 2:
                color = '#e74c3c'  # Rouge pour bloqué
                alpha = 0.5

            rect = patches.Rectangle((x - w/2, y - 0.1), w, 0.2,
                                   facecolor=color, alpha=alpha,
                                   edgecolor='darkgreen' if color == '#27ae60' else 'darkred',
                                   linewidth=2)
            ax.add_patch(rect)

            # Texte pour sortie bloquée
            if color == '#e74c3c':
                ax.text(x, y, 'BLOQUÉ', fontsize=8, fontweight='bold',
                       color='white', ha='center', va='center')

        # Dessiner les trajectoires (dernières positions)
        for agent in self.model.schedule.agents:
            if hasattr(agent, 'evacuated') and agent.evacuated:
                continue

            if hasattr(agent, 'trajectory') and len(agent.trajectory) > 1:
                # Prendre les dernières positions
                recent_traj = agent.trajectory[-10:]
                x_vals = [pos[0] for pos in recent_traj]
                y_vals = [pos[1] for pos in recent_traj]

                # Couleur selon l'état
                if agent.state == AgentState.PANIC:
                    traj_color = '#e74c3c'
                    alpha = 0.5
                elif agent.state == AgentState.EVACUATING:
                    traj_color = '#e67e22'
                    alpha = 0.4
                elif agent.state == AgentState.FOLLOWING:
                    traj_color = '#9b59b6'
                    alpha = 0.4
                else:
                    traj_color = 'gray'
                    alpha = 0.3

                ax.plot(x_vals, y_vals, color=traj_color, alpha=alpha, linewidth=1.5)

        # Dessiner les agents
        panic_count = 0
        following_count = 0

        for agent in self.model.schedule.agents:
            if hasattr(agent, 'evacuated') and agent.evacuated:
                continue

            x, y = agent.pos

            # Compter les états spéciaux
            if agent.state == AgentState.PANIC:
                panic_count += 1
            elif agent.state == AgentState.FOLLOWING:
                following_count += 1

            # Définir la couleur selon l'état
            color_map = {
                AgentState.PANIC: ('#e74c3c', 0.9, 80, '#c0392b', 2),  # Rouge
                AgentState.EVACUATING: ('#e67e22', 0.8, 70, '#d35400', 2),  # Orange
                AgentState.GROUPING: ('#2ecc71', 0.8, 65, '#27ae60', 1),  # Vert
                AgentState.FOLLOWING: ('#9b59b6', 0.8, 65, '#8e44ad', 1),  # Violet
                AgentState.OBSTACLE_AVOIDANCE: ('#795548', 0.8, 65, '#5d4037', 1),  # Marron
                AgentState.DISPERSING: ('#00bcd4', 0.8, 65, '#0097a7', 1),  # Cyan
                AgentState.STOPPED: ('#95a5a6', 0.6, 60, '#7f8c8d', 1),  # Gris
                AgentState.NORMAL: ('#3498db', 0.7, 60, '#2980b9', 1)   # Bleu
            }

            color, alpha, size, edgecolor, edgewidth = color_map.get(
                agent.state, ('#3498db', 0.7, 60, '#2980b9', 1)
            )

            # Dessiner l'agent
            circle = plt.Circle((x, y), 0.25, color=color, alpha=alpha,
                               edgecolor=edgecolor, linewidth=edgewidth)
            ax.add_patch(circle)

            # Marqueur spécial pour les leaders
            if agent.agent_type == AgentType.LEADER:
                ax.plot(x, y, '*', color='gold', markersize=10, alpha=0.9)

            # Dessiner une flèche pour la direction
            if hasattr(agent, 'velocity') and np.linalg.norm(agent.velocity) > 0.1:
                dx, dy = agent.velocity
                length = np.linalg.norm([dx, dy])

                # Normaliser et réduire la longueur pour la visibilité
                scale = 0.5
                dx_norm = dx / length * scale
                dy_norm = dy / length * scale

                arrow_color = 'white' if agent.state == AgentState.PANIC else edgecolor
                ax.arrow(x, y, dx_norm, dy_norm,
                        head_width=0.15, head_length=0.2,
                        fc=arrow_color, ec=arrow_color, alpha=0.9, linewidth=1.5)

        # Panneau d'informations
        active_agents = sum(1 for a in self.model.schedule.agents if not a.evacuated)
        total_agents = self.model.schedule.get_agent_count()
        time_elapsed = self.model.schedule.steps * self.model.time_step

        # Boîte d'information principale
        info_text = f"Étape: {self.model.schedule.steps}\n"
        info_text += f"Temps: {time_elapsed:.1f}s\n"
        info_text += f"Actifs: {active_agents}/{total_agents}\n"
        info_text += f"Évacués: {self.model.evacuated_count}\n"

        if panic_count > 0:
            info_text += f"PANIQUE: {panic_count}\n"

        if following_count > 0 and self.model.what_if_scenario == "leader_emergency":
            info_text += f"Suiveurs: {following_count}"

        ax.text(0.02, 0.98, info_text, transform=ax.transAxes,
               fontsize=10, fontweight='bold', verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='gray'))

        # Boîte de paramètres
        param_text = f"Intensité: {self.model.scenario_intensity:.1f}\n"

        if hasattr(self.model, 'helbing_params'):
            param_text += f"A: {self.model.helbing_params['A_social']:.2f}\n"
            param_text += f"B: {self.model.helbing_params['B_social']:.2f}\n"

        if self.model.global_metrics and len(self.model.global_metrics) > 0:
            last_metrics = self.model.global_metrics[-1]
            param_text += f"Vitesse: {last_metrics['average_speed']:.2f} m/s\n"
            param_text += f"Stress: {last_metrics['average_stress']:.3f}"

        ax.text(0.98, 0.98, param_text, transform=ax.transAxes,
               fontsize=9, verticalalignment='top', horizontalalignment='right',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='gray'))

        # Légende améliorée
        legend_elements = [
            patches.Patch(facecolor='#3498db', alpha=0.7, label='Normal'),
            patches.Patch(facecolor='#e74c3c', alpha=0.9, label='Panique'),
            patches.Patch(facecolor='#e67e22', alpha=0.8, label='Évacuation'),
            patches.Patch(facecolor='#2ecc71', alpha=0.8, label='Regroupement'),
            patches.Patch(facecolor='#9b59b6', alpha=0.8, label='Suivi'),
            patches.Patch(facecolor='gold', alpha=0.9, label='Leader'),
            patches.Patch(facecolor='#2c3e50', alpha=0.8, label='Obstacles'),
            patches.Patch(facecolor='#27ae60', alpha=0.8, label='Sorties'),
        ]

        ax.legend(handles=legend_elements, loc='upper left', fontsize=9,
                 framealpha=0.9, edgecolor='black')

        # Indicateur de validation
        if hasattr(self.model, 'is_validated') and self.model.is_validated:
            ax.text(0.5, 0.02, "✅ SIMULATION VALIDÉE", transform=ax.transAxes,
                   fontsize=12, fontweight='bold', color='green', ha='center',
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))

        plt.tight_layout(rect=[0, 0.03, 1, 0.97])
        return fig

    def save_video(self, filename: str = "simulation_video.mp4", fps: int = 20):
        """Sauvegarde la vidéo"""
        if not self.frames:
            print("❌ Aucune frame à sauvegarder")
            return None, None

        print(f"🎬 Génération de la vidéo {filename} ({len(self.frames)} frames, {fps} FPS)...")

        # Sauvegarder comme GIF
        gif_filename = filename.replace('.mp4', '.gif')
        try:
            self.frames[0].save(gif_filename,
                              save_all=True,
                              append_images=self.frames[1:],
                              duration=1000//fps,
                              loop=0,
                              optimize=True,
                              quality=95)
            print(f"✅ GIF sauvegardé: {gif_filename}")
        except Exception as e:
            print(f"❌ Erreur lors de la sauvegarde du GIF: {e}")
            gif_filename = None

        # Essayer de sauvegarder comme MP4
        mp4_filename = None
        try:
            import subprocess
            # Vérifier si ffmpeg est disponible
            result = subprocess.run(['ffmpeg', '-version'], capture_output=True)
            if result.returncode == 0:
                # Créer une animation MP4
                fig, ax = plt.subplots(figsize=(14, 10))
                ax.axis('off')

                def update(frame_idx):
                    ax.clear()
                    ax.axis('off')
                    img = self.frames[frame_idx]
                    ax.imshow(img)

                    # Ajouter le numéro de frame
                    ax.text(0.02, 0.02, f"Frame: {frame_idx+1}/{len(self.frames)}",
                           transform=ax.transAxes, fontsize=12,
                           bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

                    return [ax]

                anim = animation.FuncAnimation(fig, update,
                                             frames=len(self.frames),
                                             interval=1000//fps,
                                             blit=False)

                mp4_filename = filename
                anim.save(mp4_filename, writer='ffmpeg', fps=fps,
                         extra_args=['-vcodec', 'libx264', '-pix_fmt', 'yuv420p'])
                plt.close(fig)
                print(f"✅ Vidéo MP4 sauvegardée: {mp4_filename}")
            else:
                print("⚠ ffmpeg n'est pas installé, impossible de créer MP4")
        except Exception as e:
            print(f"⚠ Impossible de sauvegarder en MP4: {e}")

        return gif_filename, mp4_filename


# ANALYSE ET VISUALISATION AVANCÉE

In [ ]:
# ============================================================================
# ANALYSE ET VISUALISATION AVANCÉE
# ============================================================================

class SimulationAnalyzer:
    """Analyse les résultats de simulation avec métriques avancées"""

    def __init__(self, simulation_results):
        self.results = simulation_results
        self.agent_data = simulation_results['agent_data']
        self.model_data = simulation_results['model_data']
        self.global_metrics = simulation_results['global_metrics']
        self.collective_metrics = simulation_results.get('collective_metrics', [])
        self.validation_metrics = simulation_results.get('validation_metrics')
        self.helbing_params = simulation_results.get('helbing_params', {})

    def create_comprehensive_report(self) -> str:
        """Crée un rapport de synthèse complet et le retourne sous forme de chaîne"""
        report_lines = []
        report_lines.append("=" * 70)
        report_lines.append("📊 RAPPORT COMPLET DE SIMULATION")
        report_lines.append("=" * 70)

        # Validation
        if self.validation_metrics:
            report_lines.append(f"\n🔬 VALIDATION:")
            report_lines.append(f"   • Score de validation: {self.validation_metrics.get('validation_score', 0):.3f}")
            report_lines.append(f"   • Statut: {'✅ VALIDE' if self.validation_metrics.get('validation_score', 0) >= VALIDATION_THRESHOLD else '⚠ LIMITE'}")

            if 'position_errors' in self.validation_metrics:
                errors = self.validation_metrics['position_errors']
                if 'rmse_mean' in errors:
                    report_lines.append(f"   • Erreur position (RMSE): {errors['rmse_mean']:.3f} m")

        # Paramètres Helbing
        if self.helbing_params:
            report_lines.append(f"\n⚙ PARAMÈTRES HELBING:")
            for param, value in self.helbing_params.items():
                report_lines.append(f"   • {param}: {value:.3f}")

        # Statistiques générales
        total_agents = len(self.agent_data)
        evacuated_agents = sum(1 for a in self.agent_data if a['evacuated'])
        avg_panic = np.mean([a['panic_level'] for a in self.agent_data])
        avg_distance = np.mean([a['distance_traveled'] for a in self.agent_data])
        total_collisions = sum(a['collisions'] for a in self.agent_data)
        avg_state_changes = np.mean([a['state_changes'] for a in self.agent_data])

        report_lines.append(f"\n📈 STATISTIQUES GÉNÉRALES:")
        report_lines.append(f"   • Agents totaux: {total_agents}")
        report_lines.append(f"   • Agents évacués: {evacuated_agents} ({evacuated_agents/total_agents*100:.1f}%)")
        report_lines.append(f"   • Panique moyenne: {avg_panic:.3f}")
        report_lines.append(f"   • Distance moyenne parcourue: {avg_distance:.1f} m")
        report_lines.append(f"   • Collisions totales: {total_collisions}")
        report_lines.append(f"   • Changements d'état moyens: {avg_state_changes:.1f}")

        # Temps d'évacuation
        evacuation_times = [a['evacuation_time'] for a in self.agent_data
                          if a['evacuated'] and a['evacuation_time'] is not None]
        if evacuation_times:
            report_lines.append(f"   • Temps d'évacuation moyen: {np.mean(evacuation_times):.1f} s")
            report_lines.append(f"   • Temps d'évacuation max: {np.max(evacuation_times):.1f} s")

        # Distribution des états finaux
        report_lines.append(f"\n🎭 DISTRIBUTION DES ÉTATS FINAUX:")
        states = {}
        for agent in self.agent_data:
            state = agent['final_state']
            states[state] = states.get(state, 0) + 1

        for state, count in sorted(states.items(), key=lambda x: x[1], reverse=True):
            percentage = count / total_agents * 100
            report_lines.append(f"   • {state}: {count} agents ({percentage:.1f}%)")

        # Distribution des types
        report_lines.append(f"\n👥 DISTRIBUTION DES TYPES:")
        types = {}
        for agent in self.agent_data:
            agent_type = agent['type']
            types[agent_type] = types.get(agent_type, 0) + 1

        for agent_type, count in types.items():
            percentage = count / total_agents * 100
            report_lines.append(f"   • {agent_type}: {count} agents ({percentage:.1f}%)")

        # Métriques collectives
        if self.collective_metrics and len(self.collective_metrics) > 0:
            report_lines.append(f"\n🌐 MÉTRIQUES COLLECTIVES (finales):")
            last_metrics = self.collective_metrics[-1]

            metrics_to_show = ['density', 'clustering', 'polarization', 'entropy', 'avg_speed', 'avg_stress']
            for metric in metrics_to_show:
                if metric in last_metrics:
                    value = last_metrics[metric]
                    report_lines.append(f"   • {metric}: {value:.3f}")

        report_lines.append("="*70)
        return "\n".join(report_lines)

    def plot_comprehensive_results(self, output_dir="simulation_analysis"):
        """Génère des visualisations complètes des résultats"""
        import os
        os.makedirs(output_dir, exist_ok=True)

        # 1. Évolution temporelle complète
        self._plot_temporal_evolution(output_dir)

        # 2. Analyse spatiale
        self._plot_spatial_analysis(output_dir)

        # 3. Analyse des agents
        self._plot_agent_analysis(output_dir)

        # 4. Validation (si disponible)
        if self.validation_metrics:
            self._plot_validation_results(output_dir)

        print(f"📈 Visualisations sauvegardées dans: {output_dir}")

    def _plot_temporal_evolution(self, output_dir: str):
        """Génère les graphiques d'évolution temporelle"""
        if not self.global_metrics or len(self.global_metrics) < 2:
            return

        steps = [m['step'] for m in self.global_metrics]

        fig, axes = plt.subplots(3, 3, figsize=(18, 15))
        fig.suptitle('Évolution Temporelle des Métriques', fontsize=16, fontweight='bold')

        # Agents actifs
        axes[0, 0].plot(steps, [m['active_agents'] for m in self.global_metrics],
                       'b-', linewidth=2)
        axes[0, 0].set_title('Agents Actifs')
        axes[0, 0].set_xlabel('Étape')
        axes[0, 0].set_ylabel('Nombre')
        axes[0, 0].grid(True, alpha=0.3)

        # Vitesse moyenne
        axes[0, 1].plot(steps, [m['average_speed'] for m in self.global_metrics],
                       'r-', linewidth=2)
        axes[0, 1].set_title('Vitesse Moyenne')
        axes[0, 1].set_xlabel('Étape')
        axes[0, 1].set_ylabel('Vitesse (m/s)')
        axes[0, 1].grid(True, alpha=0.3)

        # Stress moyen
        axes[0, 2].plot(steps, [m['average_stress'] for m in self.global_metrics],
                       'orange', linewidth=2)
        axes[0, 2].set_title('Stress Moyen')
        axes[0, 2].set_xlabel('Étape')
        axes[0, 2].set_ylabel('Stress (0-1)')
        axes[0, 2].grid(True, alpha=0.3)

        # Distribution des états (heatmap)
        if len(self.global_metrics) > 0:
            all_states = set()
            for m in self.global_metrics:
                all_states.update(m['state_distribution'].keys())

            state_matrix = []
            for state in sorted(all_states):
                state_counts = [m['state_distribution'].get(state, 0) for m in self.global_metrics]
                state_matrix.append(state_counts)

            if state_matrix:
                im = axes[1, 0].imshow(state_matrix, aspect='auto', cmap='viridis')
                axes[1, 0].set_title('Distribution des États')
                axes[1, 0].set_xlabel('Étape')
                axes[1, 0].set_ylabel('État')
                axes[1, 0].set_yticks(range(len(sorted(all_states))))
                axes[1, 0].set_yticklabels(sorted(all_states))
                plt.colorbar(im, ax=axes[1, 0])

        # Collisions cumulées
        if 'collisions_total' in self.global_metrics[0]:
            collisions = [m['collisions_total'] for m in self.global_metrics]
            axes[1, 1].plot(steps, collisions, 'brown', linewidth=2)
            axes[1, 1].set_title('Collisions Cumulées')
            axes[1, 1].set_xlabel('Étape')
            axes[1, 1].set_ylabel('Nombre de collisions')
            axes[1, 1].grid(True, alpha=0.3)

        # Métriques collectives (si disponibles)
        if self.collective_metrics and len(self.collective_metrics) > 0:
            coll_steps = [m['step'] for m in self.collective_metrics]

            # Densité
            if 'density' in self.collective_metrics[0]:
                density = [m['density'] for m in self.collective_metrics]
                axes[1, 2].plot(coll_steps, density, 'purple', linewidth=2)
                axes[1, 2].set_title('Densité Collective')
                axes[1, 2].set_xlabel('Étape')
                axes[1, 2].set_ylabel('Densité')
                axes[1, 2].grid(True, alpha=0.3)

            # Clustering
            if 'clustering' in self.collective_metrics[0]:
                clustering = [m['clustering'] for m in self.collective_metrics]
                axes[2, 0].plot(coll_steps, clustering, 'green', linewidth=2)
                axes[2, 0].set_title('Coefficient de Clustering')
                axes[2, 0].set_xlabel('Étape')
                axes[2, 0].set_ylabel('Clustering')
                axes[2, 0].grid(True, alpha=0.3)

            # Polarisation
            if 'polarization' in self.collective_metrics[0]:
                polarization = [m['polarization'] for m in self.collective_metrics]
                axes[2, 1].plot(coll_steps, polarization, 'cyan', linewidth=2)
                axes[2, 1].set_title('Polarisation des Directions')
                axes[2, 1].set_xlabel('Étape')
                axes[2, 1].set_ylabel('Polarisation')
                axes[2, 1].grid(True, alpha=0.3)

        # Évacuation cumulative
        if 'Evacuated Agents' in self.model_data.columns:
            axes[2, 2].plot(self.model_data.index, self.model_data['Evacuated Agents'],
                          'g-', linewidth=2)
            axes[2, 2].set_title('Évacuation Cumulative')
            axes[2, 2].set_xlabel('Étape')
            axes[2, 2].set_ylabel('Agents évacués')
            axes[2, 2].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig(f"{output_dir}/temporal_evolution.png", dpi=150, bbox_inches='tight')
        plt.close()

    def _plot_spatial_analysis(self, output_dir: str):
        """Génère les graphiques d'analyse spatiale"""
        fig, axes = plt.subplots(2, 2, figsize=(14, 12))

        # Positions finales
        ax = axes[0, 0]

        # Dessiner les obstacles
        for obs in [(0, 0, 15, 0.2), (0, 11.8, 15, 0.2), (0, 0, 0.2, 12),
                   (14.8, 0, 0.2, 12), (5, 4, 2, 1), (10, 7, 1, 3)]:
            x, y, w, h = obs
            rect = patches.Rectangle((x, y), w, h, facecolor='gray', alpha=0.5, edgecolor='black')
            ax.add_patch(rect)

        # Dessiner les sorties
        for exit in [(7.5, 0, 1.5), (7.5, 12, 1.5), (0, 6, 1.0), (15, 6, 1.0)]:
            x, y, w = exit
            rect = patches.Rectangle((x - w/2, y - 0.1), w, 0.2,
                                    facecolor='green', alpha=0.7, edgecolor='darkgreen')
            ax.add_patch(rect)

        # Positions finales avec couleur selon état
        for agent in self.agent_data:
            if agent['final_position']:
                x, y = agent['final_position']

                # Couleur selon l'état final
                if agent['final_state'] == 'panic':
                    color = 'red'
                    size = 80
                    alpha = 0.8
                elif agent['final_state'] == 'evacuating':
                    color = 'orange'
                    size = 70
                    alpha = 0.7
                elif agent['evacuated']:
                    color = 'green'
                    size = 60
                    alpha = 0.6
                else:
                    color = 'blue'
                    size = 50
                    alpha = 0.5

                ax.scatter(x, y, color=color, s=size, alpha=alpha, edgecolors='black', linewidth=0.5)

        ax.set_xlim(0, 15)
        ax.set_ylim(0, 12)
        ax.set_aspect('equal')
        ax.set_title('Positions Finales des Agents')
        ax.set_xlabel('X (mètres)')
        ax.set_ylabel('Y (mètres)')

        # Heatmap de densité
        ax = axes[0, 1]
        if len(self.agent_data) > 0:
            positions = np.array([a['final_position'] for a in self.agent_data
                                if a['final_position'] is not None])

            if len(positions) > 0:
                # Histogramme 2D
                heatmap, xedges, yedges = np.histogram2d(
                    positions[:, 0], positions[:, 1],
                    bins=[30, 24], range=[[0, 15], [0, 12]]
                )

                im = ax.imshow(heatmap.T, origin='lower', aspect='auto',
                             extent=[0, 15, 0, 12], cmap='hot_r', alpha=0.8)
                ax.set_title('Heatmap de Densité Finale')
                ax.set_xlabel('X (mètres)')
                ax.set_ylabel('Y (mètres)')
                plt.colorbar(im, ax=ax, label='Nombre d\'agents')

        # Distribution des distances parcourues
        ax = axes[1, 0]
        distances = [a['distance_traveled'] for a in self.agent_data]
        if distances:
            ax.hist(distances, bins=20, color='blue', alpha=0.7, edgecolor='black')
            ax.set_title('Distribution des Distances Parcourues')
            ax.set_xlabel('Distance (mètres)')
            ax.set_ylabel('Nombre d\'agents')
            ax.grid(True, alpha=0.3)

            # Statistiques
            mean_dist = np.mean(distances)
            median_dist = np.median(distances)
            ax.axvline(mean_dist, color='red', linestyle='--', label=f'Moyenne: {mean_dist:.1f}m')
            ax.axvline(median_dist, color='green', linestyle='--', label=f'Médiane: {median_dist:.1f}m')
            ax.legend()

        # Distribution des temps d'évacuation
        ax = axes[1, 1]
        evacuation_times = [a['evacuation_time'] for a in self.agent_data
                          if a['evacuated'] and a['evacuation_time'] is not None]

        if evacuation_times:
            ax.hist(evacuation_times, bins=15, color='green', alpha=0.7, edgecolor='black')
            ax.set_title('Distribution des Temps d\'Évacuation')
            ax.set_xlabel('Temps (secondes)')
            ax.set_ylabel('Nombre d\'agents')
            ax.grid(True, alpha=0.3)

            # Courbe cumulative
            ax2 = ax.twinx()
            sorted_times = np.sort(evacuation_times)
            cumulative = np.arange(1, len(sorted_times) + 1) / len(sorted_times)
            ax2.plot(sorted_times, cumulative * 100, 'r-', linewidth=2)
            ax2.set_ylabel('Pourcentage cumulé (%)', color='red')
            ax2.tick_params(axis='y', labelcolor='red')

            # Indicateurs
            median_time = np.median(evacuation_times)
            ax.axvline(median_time, color='blue', linestyle='--',
                      label=f'Médiane: {median_time:.1f}s')
            ax.legend()

        plt.tight_layout()
        plt.savefig(f"{output_dir}/spatial_analysis.png", dpi=150, bbox_inches='tight')
        plt.close()

    def _plot_agent_analysis(self, output_dir: str):
        """Génère les graphiques d'analyse des agents"""
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))

        # Distribution des niveaux de panique
        panic_levels = [a['panic_level'] for a in self.agent_data]
        axes[0, 0].hist(panic_levels, bins=20, color='red', alpha=0.7, edgecolor='black')
        axes[0, 0].set_title('Distribution des Niveaux de Panique')
        axes[0, 0].set_xlabel('Niveau de panique (0-1)')
        axes[0, 0].set_ylabel('Nombre d\'agents')
        axes[0, 0].grid(True, alpha=0.3)

        # Panique vs distance
        distances = [a['distance_traveled'] for a in self.agent_data]
        if panic_levels and distances:
            axes[0, 1].scatter(panic_levels, distances, alpha=0.6, c='blue', edgecolors='black')
            axes[0, 1].set_title('Panique vs Distance Parcourue')
            axes[0, 1].set_xlabel('Niveau de panique')
            axes[0, 1].set_ylabel('Distance parcourue (m)')
            axes[0, 1].grid(True, alpha=0.3)

            # Régression linéaire
            if len(panic_levels) > 1:
                z = np.polyfit(panic_levels, distances, 1)
                p = np.poly1d(z)
                axes[0, 1].plot(panic_levels, p(panic_levels), "r--", alpha=0.8)

        # Distribution des collisions
        collisions = [a['collisions'] for a in self.agent_data]
        if collisions:
            axes[0, 2].hist(collisions, bins=range(0, int(max(collisions)) + 2),
                           color='orange', alpha=0.7, edgecolor='black')
            axes[0, 2].set_title('Distribution des Collisions')
            axes[0, 2].set_xlabel('Nombre de collisions')
            axes[0, 2].set_ylabel('Nombre d\'agents')
            axes[0, 2].grid(True, alpha=0.3)

        # Distribution des changements d'état
        state_changes = [a['state_changes'] for a in self.agent_data]
        if state_changes:
            axes[1, 0].hist(state_changes, bins=range(0, int(max(state_changes)) + 2),
                           color='green', alpha=0.7, edgecolor='black')
            axes[1, 0].set_title('Distribution des Changements d\'État')
            axes[1, 0].set_xlabel('Nombre de changements')
            axes[1, 0].set_ylabel('Nombre d\'agents')
            axes[1, 0].grid(True, alpha=0.3)

        # Type d'agent vs métriques
        agent_types = [a['type'] for a in self.agent_data]
        unique_types = list(set(agent_types))

        # Préparer les données par type
        type_data = {t: {'panic': [], 'distance': [], 'collisions': []} for t in unique_types}

        for i, agent in enumerate(self.agent_data):
            t = agent['type']
            type_data[t]['panic'].append(agent['panic_level'])
            type_data[t]['distance'].append(agent['distance_traveled'])
            type_data[t]['collisions'].append(agent['collisions'])

        # Boxplot de panique par type
        if unique_types:
            panic_data = [type_data[t]['panic'] for t in unique_types]
            bp = axes[1, 1].boxplot(panic_data, labels=unique_types, patch_artist=True)

            # Couleurs
            colors = ['lightblue', 'lightgreen', 'lightcoral', 'lightsalmon']
            for patch, color in zip(bp['boxes'], colors):
                patch.set_facecolor(color)

            axes[1, 1].set_title('Panique par Type d\'Agent')
            axes[1, 1].set_ylabel('Niveau de panique')
            axes[1, 1].grid(True, alpha=0.3, axis='y')

        # Diagramme radar des métriques globales
        if self.global_metrics and len(self.global_metrics) > 0:
            last_metrics = self.global_metrics[-1]

            # Métriques à afficher
            radar_metrics = ['average_speed', 'average_stress', 'average_panic']
            radar_labels = ['Vitesse', 'Stress', 'Panique']

            # Normalisation
            max_values = {
                'average_speed': 2.0,  # m/s maximum
                'average_stress': 1.0,
                'average_panic': 1.0
            }

            values = []
            for metric in radar_metrics:
                if metric in last_metrics:
                    value = last_metrics[metric] / max_values[metric]
                    values.append(min(value, 1.0))
                else:
                    values.append(0)

            # Compléter pour avoir un polygone fermé
            values = values + values[:1]
            radar_labels = radar_labels + radar_labels[:1]

            # Angles
            angles = np.linspace(0, 2 * np.pi, len(radar_labels), endpoint=True).tolist()

            # Radar plot
            ax = axes[1, 2]
            ax = fig.add_subplot(2, 3, 6, polar=True)
            ax.plot(angles, values, 'o-', linewidth=2, color='blue')
            ax.fill(angles, values, alpha=0.25, color='blue')
            ax.set_xticks(angles[:-1])
            ax.set_xticklabels(radar_labels[:-1])
            ax.set_ylim(0, 1)
            ax.set_title('Métriques Globales Finales', y=1.08)
            ax.grid(True)

        plt.tight_layout()
        plt.savefig(f"{output_dir}/agent_analysis.png", dpi=150, bbox_inches='tight')
        plt.close()

    def _plot_validation_results(self, output_dir: str):
        """Génère les graphiques de validation"""
        if not self.validation_metrics:
            return

        fig, axes = plt.subplots(2, 2, figsize=(12, 10))

        # Score de validation
        if 'validation_score' in self.validation_metrics:
            score = self.validation_metrics['validation_score']

            ax = axes[0, 0]
            bars = ax.bar(['Score'], [score], color='green' if score >= VALIDATION_THRESHOLD else 'red')
            ax.axhline(y=VALIDATION_THRESHOLD, color='red', linestyle='--',
                      label=f'Seuil: {VALIDATION_THRESHOLD}')
            ax.set_ylim(0, 1)
            ax.set_ylabel('Score (0-1)')
            ax.set_title('Score de Validation Global')
            ax.legend()

            # Annoter la barre
            ax.text(0, score + 0.02, f'{score:.3f}', ha='center', va='bottom', fontweight='bold')

        # Erreurs de position
        if 'position_errors' in self.validation_metrics:
            errors = self.validation_metrics['position_errors']

            error_types = ['RMSE', 'MAE']
            error_values = [errors.get('rmse_mean', 0), errors.get('mae_mean', 0)]

            ax = axes[0, 1]
            bars = ax.bar(error_types, error_values, color=['orange', 'blue'])
            ax.set_ylabel('Erreur (mètres)')
            ax.set_title('Erreurs de Position')
            ax.grid(True, alpha=0.3, axis='y')

            # Annoter les barres
            for bar, value in zip(bars, error_values):
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                       f'{value:.3f}', ha='center', va='bottom')

        # Correlation des vitesses
        if 'velocity_errors' in self.validation_metrics:
            vel_errors = self.validation_metrics['velocity_errors']

            if 'speed_correlation' in vel_errors:
                correlation = vel_errors['speed_correlation']

                ax = axes[1, 0]
                bars = ax.bar(['Corrélation'], [correlation],
                             color='green' if correlation >= 0.7 else 'orange')
                ax.set_ylim(-1, 1)
                ax.set_ylabel('Coefficient de corrélation')
                ax.set_title('Corrélation des Vitesses')
                ax.axhline(y=0, color='black', linewidth=0.5)

                # Annoter la barre
                ax.text(0, correlation + (0.1 if correlation >= 0 else -0.1),
                       f'{correlation:.3f}', ha='center', va='bottom' if correlation >= 0 else 'top',
                       fontweight='bold')

        # Similarité des trajectoires
        if 'trajectory_similarities' in self.validation_metrics:
            similarities = self.validation_metrics['trajectory_similarities']

            if 'avg_similarity' in similarities:
                similarity = similarities['avg_similarity']

                ax = axes[1, 1]
                bars = ax.bar(['Similarité'], [similarity],
                             color='green' if similarity >= 0.7 else 'orange')
                ax.set_ylim(0, 1)
                ax.set_ylabel('Similarité (0-1)')
                ax.set_title('Similarité Moyenne des Trajectoires')

                # Annoter la barre
                ax.text(0, similarity + 0.02, f'{similarity:.3f}',
                       ha='center', va='bottom', fontweight='bold')

        plt.tight_layout()
        plt.savefig(f"{output_dir}/validation_results.png", dpi=150, bbox_inches='tight')
        plt.close()


# INTERFACE UTILISATEUR ET EXÉCUTION PRINCIPALE

In [ ]:

# ============================================================================
# INTERFACE UTILISATEUR ET EXÉCUTION PRINCIPALE
# ============================================================================

def run_comprehensive_simulation():
    """Exécute une simulation complète avec toutes les améliorations"""
    print("="*80)
    print("🚀 SIMULATION MULTI-AGENTS COMPLÈTE - VERSION AMÉLIORÉE")
    print("="*80)

    # 1. Chargement des données réelles
    print("\n1. 📥 CHARGEMENT DES DONNÉES DE TRACKING")
    trajectories_path = input("Chemin du fichier CSV de trajectoires (défaut: trajectories.csv): ").strip()

    if not trajectories_path:
        trajectories_path = "trajectories.csv"

    try:
        data_loader = RealDataLoader(trajectories_path)
        print(f"✅ Fichier trouvé: {trajectories_path}")
    except FileNotFoundError as e:
        print(f"❌ {e}")
        print("💡 Veuillez placer votre fichier de données dans le répertoire courant.")
        return None

    # 2. Configuration de la simulation
    print("\n2. ⚙ CONFIGURATION DE LA SIMULATION")

    # Scénario
    print("\nScénarios disponibles:")
    print("  1. Normal (reproduction du comportement observé)")
    print("  2. Panique (niveau élevé de stress)")
    print("  3. Sorties bloquées (capacité réduite)")
    print("  4. Haute densité (plus d'agents)")
    print("  5. Évacuation dirigée (avec leaders)")

    choice = input("\nChoisissez un scénario (1-5, Enter pour 1): ").strip() or "1"

    scenarios = {
        "1": ("normal", 0.0),
        "2": ("panic", 0.5),
        "3": ("obstacle_blocked", 0.5),
        "4": ("high_density", 1.0),
        "5": ("leader_emergency", 0.8)
    }

    if choice in scenarios:
        scenario_name, intensity = scenarios[choice]
    else:
        scenario_name, intensity = "normal", 0.0

    print(f"✅ Scénario sélectionné: {scenario_name} (intensité: {intensity})")

    # Options avancées
    print("\nOptions avancées:")
    validate = input("Valider la simulation avant exécution? (o/n, défaut: o): ").strip().lower()
    validate = validate != "n"

    generate_video = input("Générer une vidéo? (o/n, défaut: o): ").strip().lower()
    generate_video = generate_video != "n"

    # 3. Création du modèle
    print("\n3. 🏗 CRÉATION DU MODÈLE MESA")

    model = CrowdModel(
        real_data_loader=data_loader,
        width=15.0,
        height=12.0,
        time_step=0.1,
        what_if_scenario=scenario_name,
        scenario_intensity=intensity,
        max_agents=30,
        verbose=True,
        validation_mode=False
    )

    # 4. Exécution avec ou sans vidéo
    if generate_video:
        print("\n4. 🎥 GÉNÉRATION DE VIDÉO")

        # Paramètres vidéo
        steps = input("Nombre de frames pour la vidéo (défaut: 200): ").strip()
        steps = int(steps) if steps else 200

        fps = input("Images par seconde (défaut: 20): ").strip()
        fps = int(fps) if fps else 20

        print("🎬 Préparation de la génération vidéo...")

        # Créer et exécuter le générateur de vidéo
        video_gen = VideoGenerator(model, steps=steps)
        video_gen.capture_simulation()

        # Sauvegarder la vidéo
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        video_filename = f"simulation_{scenario_name}_{timestamp}.mp4"

        gif_path, mp4_path = video_gen.save_video(video_filename, fps=fps)

        print(f"\n✅ Vidéo générée avec succès!")
        if gif_path:
            print(f"📁 GIF: {gif_path}")
        if mp4_path:
            print(f"📁 MP4: {mp4_path}")

        # Exécuter la simulation complète
        print("\n5. 🔄 EXÉCUTION DE LA SIMULATION COMPLÈTE")
        results = model.run_simulation(steps=300, validate_first=validate)

    else:
        # Exécution sans vidéo
        print("\n4. 🔄 EXÉCUTION DE LA SIMULATION")
        results = model.run_simulation(steps=300, validate_first=validate)

    # 6. Analyse des résultats
    print("\n6. 📊 ANALYSE DES RÉSULTATS")
    analyzer = SimulationAnalyzer(results)
    analyzer.create_comprehensive_report()

    # 7. Génération des visualisations
    print("\n7. 🎨 GÉNÉRATION DES VISUALISATIONS")
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = f"simulation_analysis_{scenario_name}_{timestamp}"

    analyzer.plot_comprehensive_results(output_dir)

    # 8. Sauvegarde des résultats
    print("\n8. 💾 SAUVEGARDE DES RÉSULTATS")
    results_dir = f"mesa_results_{scenario_name}_{timestamp}"
    os.makedirs(results_dir, exist_ok=True)

    # Sauvegarder les données
    pd.DataFrame(results['agent_data']).to_csv(f"{results_dir}/agent_results.csv", index=False)
    results['model_data'].to_csv(f"{results_dir}/model_metrics.csv")

    if results.get('agent_vars_data') is not None:
        results['agent_vars_data'].to_csv(f"{results_dir}/agent_vars.csv")

    with open(f"{results_dir}/global_metrics.json", 'w') as f:
        json.dump(results['global_metrics'], f, indent=2)

    with open(f"{results_dir}/collective_metrics.json", 'w') as f:
        json.dump(results['collective_metrics'], f, indent=2)

    with open(f"{results_dir}/simulation_summary.txt", 'w') as f:
        f.write(analyzer.create_comprehensive_report())

    # Copier les vidéos et visualisations
    import shutil
    if generate_video:
        if gif_path and os.path.exists(gif_path):
            shutil.copy2(gif_path, f"{results_dir}/{os.path.basename(gif_path)}")
        if mp4_path and os.path.exists(mp4_path):
            shutil.copy2(mp4_path, f"{results_dir}/{os.path.basename(mp4_path)}")

    if os.path.exists(output_dir):
        for file in os.listdir(output_dir):
            shutil.copy2(f"{output_dir}/{file}", f"{results_dir}/{file}")

    print(f"\n🎉 SIMULATION TERMINÉE AVEC SUCCÈS!")
    print(f"📁 Résultats complets dans: {results_dir}")
    print(f"📈 Visualisations dans: {output_dir}")

    if generate_video and (gif_path or mp4_path):
        print(f"🎥 Vidéos dans: {results_dir}")

    return results

# ============================================================================
# EXÉCUTION DIRECTE POUR TESTS
# ============================================================================

def run_test_simulation():
    """Exécute une simulation de test rapide"""
    print("🧪 EXÉCUTION DE TEST RAPIDE")

    # Créer un petit jeu de données de test
    test_data = {
        0: [{'frame': i, 'world_x': 2 + i*0.1, 'world_y': 2,
             'world_foot_x': 2 + i*0.1, 'world_foot_y': 2}
            for i in range(50)],
        1: [{'frame': i, 'world_x': 3, 'world_y': 2 + i*0.1,
             'world_foot_x': 3, 'world_foot_y': 2 + i*0.1}
            for i in range(50)]
    }

    # Sauvegarder temporairement
    test_df = pd.DataFrame([
        {'id': 0, 'frame': i, 'world_foot_x': 2 + i*0.1, 'world_foot_y': 2}
        for i in range(50)
    ] + [
        {'id': 1, 'frame': i, 'world_foot_x': 3, 'world_foot_y': 2 + i*0.1}
        for i in range(50)
    ])

    test_csv = "test_trajectories.csv"
    test_df.to_csv(test_csv, index=False)

    try:
        data_loader = RealDataLoader(test_csv)

        model = CrowdModel(
            real_data_loader=data_loader,
            width=15.0,
            height=12.0,
            time_step=0.1,
            what_if_scenario="normal",
            scenario_intensity=0.0,
            max_agents=5,
            verbose=True,
            validation_mode=True
        )

        # Validation
        validation_results = model.validate_simulation(max_steps=50)

        # Simulation courte
        results = model.run_simulation(steps=100, validate_first=False)

        # Nettoyer
        if os.path.exists(test_csv):
            os.remove(test_csv)

        return results

    except Exception as e:
        print(f"❌ Erreur lors du test: {e}")
        if os.path.exists(test_csv):
            os.remove(test_csv)
        return None
# ============================================================================
# FONCTION DE GÉNÉRATION VIDÉO SEULE
# ============================================================================

def generate_video_only():
    """Génère uniquement une vidéo sans analyse complète"""
    print("\n" + "="*60)
    print("🎬 GÉNÉRATEUR DE VIDÉO SEULE")
    print("="*60)

    # 1. Chargement des données
    print("\n1. 📥 CHARGEMENT DES DONNÉES")
    trajectories_path = input("Chemin du fichier CSV (défaut: trajectories.csv): ").strip()

    if not trajectories_path:
        trajectories_path = "trajectories.csv"

    try:
        data_loader = RealDataLoader(trajectories_path)
        print(f"✅ Fichier trouvé: {trajectories_path}")
    except FileNotFoundError as e:
        print(f"❌ {e}")
        print("💡 Création de données de test...")

        # Créer des données de test minimales
        test_data = pd.DataFrame([
            {'id': i, 'frame': j,
             'world_foot_x': 2 + i*0.5 + j*0.1,
             'world_foot_y': 2 + i*0.3 + random.uniform(-0.2, 0.2),
             'pixel_foot_x': 0, 'pixel_foot_y': 0, 'score': 0.8}
            for i in range(10) for j in range(50)
        ])

        test_data.to_csv("test_trajectories_temp.csv", index=False)
        data_loader = RealDataLoader("test_trajectories_temp.csv")

    # 2. Configuration de la vidéo
    print("\n2. ⚙ CONFIGURATION DE LA VIDÉO")

    print("\nScénarios disponibles:")
    print("  1. Normal (reproduction du comportement observé)")
    print("  2. Panique (niveau élevé de stress)")
    print("  3. Sorties bloquées (capacité réduite)")
    print("  4. Haute densité (plus d'agents)")
    print("  5. Évacuation dirigée (avec leaders)")

    choice = input("\nChoisissez un scénario (1-5, Enter pour 1): ").strip() or "1"

    scenarios = {
        "1": ("normal", 0.0),
        "2": ("panic", 0.7),
        "3": ("obstacle_blocked", 0.5),
        "4": ("high_density", 1.5),
        "5": ("leader_emergency", 0.8)
    }

    if choice in scenarios:
        scenario_name, intensity = scenarios[choice]
    else:
        scenario_name, intensity = "normal", 0.0

    print(f"✅ Scénario sélectionné: {scenario_name} (intensité: {intensity})")

    # Paramètres vidéo
    steps = input("\nNombre de frames (défaut: 150): ").strip()
    steps = int(steps) if steps else 150

    fps = input("Images par seconde (défaut: 20): ").strip()
    fps = int(fps) if fps else 20

    # 3. Création du modèle
    print("\n3. 🏗 CRÉATION DU MODÈLE")

    model = CrowdModel(
        real_data_loader=data_loader,
        width=15.0,
        height=12.0,
        time_step=0.1,
        what_if_scenario=scenario_name,
        scenario_intensity=intensity,
        max_agents=20,  # Limité pour performance vidéo
        verbose=True,
        validation_mode=True  # Pas de calibration pour aller plus vite
    )

    # 4. Génération de la vidéo
    print("\n4. 🎥 GÉNÉRATION DE LA VIDÉO")

    video_gen = VideoGenerator(model, steps=steps)
    video_gen.capture_simulation()

    # 5. Sauvegarde
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    video_filename = f"video_{scenario_name}_{timestamp}.mp4"

    print(f"\n💾 Sauvegarde de la vidéo...")
    gif_path, mp4_path = video_gen.save_video(video_filename, fps=fps)

    print(f"\n✅ VIDÉO GÉNÉRÉE AVEC SUCCÈS!")
    if gif_path:
        print(f"📁 GIF: {gif_path}")
        # Afficher le GIF dans Colab si possible
        try:
            from IPython.display import Image as IPImage
            display(IPImage(filename=gif_path))
        except:
            pass

    if mp4_path:
        print(f"📁 MP4: {mp4_path}")

    # Nettoyage
    if os.path.exists("test_trajectories_temp.csv"):
        os.remove("test_trajectories_temp.csv")

    return gif_path, mp4_path

# ============================================================================
# FONCTION DE COMPARAISON DE SCÉNARIOS
# ============================================================================

def compare_scenarios():
    """Compare plusieurs scénarios what-if"""
    import shutil # Added import statement for shutil
    print("\n" + "="*60)
    print("📊 COMPARAISON DE SCÉNARIOS WHAT-IF")
    print("="*60)

    # 1. Chargement des données
    print("\n1. 📥 CHARGEMENT DES DONNÉES")
    trajectories_path = input("Chemin du fichier CSV (défaut: trajectories.csv): ").strip()

    if not trajectories_path:
        trajectories_path = "trajectories.csv"

    try:
        data_loader = RealDataLoader(trajectories_path)
        print(f"✅ Fichier trouvé: {trajectories_path}")
    except FileNotFoundError as e:
        print(f"❌ {e}")
        print("💡 Utilisation de données synthétiques pour comparaison...")
        return None

    # 2. Sélection des scénarios à comparer
    print("\n2. 🎭 SÉLECTION DES SCÉNARIOS")

    scenarios_to_compare = []

    print("\nScénarios disponibles pour comparaison:")
    print("  1. Normal (baseline)")
    print("  2. Panique légère (intensité 0.3)")
    print("  3. Panique forte (intensité 0.7)")
    print("  4. 50% des sorties bloquées")
    print("  5. Haute densité (+50% d'agents)")
    print("  6. Évacuation dirigée par leaders")

    choices = input("\nEntrez les numéros des scénarios à comparer (ex: 1,3,5): ").strip()

    if not choices:
        choices = "1,3,5"  # Par défaut

    scenario_numbers = [c.strip() for c in choices.split(",")]

    scenario_definitions = {
        "1": ("normal", 0.0, "Baseline"),
        "2": ("panic", 0.3, "Panique légère"),
        "3": ("panic", 0.7, "Panique forte"),
        "4": ("obstacle_blocked", 0.5, "Sorties bloquées"),
        "5": ("high_density", 0.5, "Haute densité"),
        "6": ("leader_emergency", 0.8, "Évacuation dirigée")
    }

    for num in scenario_numbers:
        if num in scenario_definitions:
            name, intensity, label = scenario_definitions[num]
            scenarios_to_compare.append((name, intensity, label))
            print(f"   ✅ Ajouté: {label}")

    if not scenarios_to_compare:
        print("⚠ Aucun scénario valide sélectionné, utilisation par défaut")
        scenarios_to_compare = [
            ("normal", 0.0, "Baseline"),
            ("panic", 0.7, "Panique forte"),
            ("obstacle_blocked", 0.5, "Sorties bloquées")
        ]

    # 3. Exécution des simulations
    print("\n3. 🔄 EXÉCUTION DES SIMULATIONS")

    results = {}

    for scenario_name, intensity, label in scenarios_to_compare:
        print(f"\n{'='*40}")
        print(f"Scénario: {label}")
        print(f"{'='*40}")

        # Créer le modèle
        model = CrowdModel(
            real_data_loader=data_loader,
            width=15.0,
            height=12.0,
            time_step=0.1,
            what_if_scenario=scenario_name,
            scenario_intensity=intensity,
            max_agents=20,  # Limité pour comparaison rapide
            verbose=False,  # Moins de verbosité pour comparaison
            validation_mode=True  # Pas de calibration
        )

        # Exécuter la simulation
        sim_results = model.run_simulation(steps=150, validate_first=False)

        # Extraire les métriques clés
        agent_data = sim_results['agent_data']
        total_agents = len(agent_data)
        evacuated = sum(1 for a in agent_data if a['evacuated'])
        avg_panic = np.mean([a['panic_level'] for a in agent_data])
        avg_distance = np.mean([a['distance_traveled'] for a in agent_data])
        total_collisions = sum(a['collisions'] for a in agent_data)

        # Temps d'évacuation moyen
        evacuation_times = [a['evacuation_time'] for a in agent_data
                          if a['evacuated'] and a['evacuation_time'] is not None]
        avg_evacuation_time = np.mean(evacuation_times) if evacuation_times else None

        # Métriques de fin de simulation
        final_metrics = sim_results['global_metrics'][-1] if sim_results['global_metrics'] else {}

        results[label] = {
            'evacuation_rate': evacuated / total_agents if total_agents > 0 else 0,
            'avg_panic': avg_panic,
            'avg_distance': avg_distance,
            'total_collisions': total_collisions,
            'avg_evacuation_time': avg_evacuation_time,
            'final_speed': final_metrics.get('average_speed', 0),
            'final_stress': final_metrics.get('average_stress', 0),
            'agent_data': agent_data[:10],  # Prendre seulement les premiers pour économiser de la mémoire
            'model_data': sim_results['model_data'].tail(10)  # Dernières lignes seulement
        }

        print(f"  Taux d'évacuation: {results[label]['evacuation_rate']*100:.1f}%")
        print(f"  Panique moyenne: {results[label]['avg_panic']:.3f}")
        print(f"  Collisions: {results[label]['total_collisions']}")
        if avg_evacuation_time:
            print(f"  Temps d'évacuation moyen: {avg_evacuation_time:.1f}s")

    # 4. Analyse comparative
    print("\n4. 📊 ANALYSE COMPARATIVE")

    # Préparer les données pour visualisation
    labels = [scenario[2] for scenario in scenarios_to_compare]

    # Créer des visualisations comparatives
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Comparaison des Scénarios What-If', fontsize=16, fontweight='bold')

    # Taux d'évacuation
    evacuation_rates = [results[label]['evacuation_rate'] for label in labels]
    bars1 = axes[0, 0].bar(labels, evacuation_rates, color=['#2ecc71', '#e74c3c', '#3498db', '#f39c12', '#9b59b6'][:len(labels)])
    axes[0, 0].set_title('Taux d\'Évacuation')
    axes[0, 0].set_ylabel('Taux (0-1)')
    axes[0, 0].set_ylim(0, 1.1)
    axes[0, 0].tick_params(axis='x', rotation=45)

    # Annoter les barres
    for bar, rate in zip(bars1, evacuation_rates):
        height = bar.get_height()
        axes[0, 0].text(bar.get_x() + bar.get_width()/2., height + 0.02,
                       f'{rate*100:.1f}%', ha='center', va='bottom')

    # Panique moyenne
    panic_levels = [results[label]['avg_panic'] for label in labels]
    bars2 = axes[0, 1].bar(labels, panic_levels, color=['#e74c3c', '#c0392b', '#d35400', '#e67e22', '#f39c12'][:len(labels)])
    axes[0, 1].set_title('Panique Moyenne')
    axes[0, 1].set_ylabel('Niveau (0-1)')
    axes[0, 1].set_ylim(0, 1.1)
    axes[0, 1].tick_params(axis='x', rotation=45)

    # Annoter les barres
    for bar, panic in zip(bars2, panic_levels):
        height = bar.get_height()
        axes[0, 1].text(bar.get_x() + bar.get_width()/2., height + 0.02,
                       f'{panic:.3f}', ha='center', va='bottom')

    # Collisions totales
    collisions = [results[label]['total_collisions'] for label in labels]
    bars3 = axes[0, 2].bar(labels, collisions, color=['#e67e22', '#d35400', '#f39c12', '#e74c3c', '#c0392b'][:len(labels)])
    axes[0, 2].set_title('Nombre de Collisions')
    axes[0, 2].set_ylabel('Collisions')
    axes[0, 2].tick_params(axis='x', rotation=45)

    # Annoter les barres
    for bar, collision in zip(bars3, collisions):
        height = bar.get_height()
        axes[0, 2].text(bar.get_x() + bar.get_width()/2., height + 0.1,
                       f'{collision:.0f}', ha='center', va='bottom')

    # Distance moyenne parcourue
    distances = [results[label]['avg_distance'] for label in labels]
    bars4 = axes[1, 0].bar(labels, distances, color=['#3498db', '#2980b9', '#1abc9c', '#16a085', '#27ae60'][:len(labels)])
    axes[1, 0].set_title('Distance Moyenne Parcourue')
    axes[1, 0].set_ylabel('Distance (m)')
    axes[1, 0].tick_params(axis='x', rotation=45)

    # Annoter les barres
    for bar, distance in zip(bars4, distances):
        height = bar.get_height()
        axes[1, 0].text(bar.get_x() + bar.get_width()/2., height + 0.1,
                       f'{distance:.1f}m', ha='center', va='bottom')

    # Vitesse finale moyenne
    speeds = [results[label]['final_speed'] for label in labels]
    bars5 = axes[1, 1].bar(labels, speeds, color=['#9b59b6', '#8e44ad', '#5dade2', '#3498db', '#2ecc71'][:len(labels)])
    axes[1, 1].set_title('Vitesse Finale Moyenne')
    axes[1, 1].set_ylabel('Vitesse (m/s)')
    axes[1, 1].tick_params(axis='x', rotation=45)

    # Annoter les barres
    for bar, speed in zip(bars5, speeds):
        height = bar.get_height()
        axes[1, 1].text(bar.get_x() + bar.get_width()/2., height + 0.02,
                       f'{speed:.2f} m/s', ha='center', va='bottom')

    # Temps d'évacuation moyen (si disponible)
    evacuation_times = []
    valid_labels = []
    for label in labels:
        if results[label]['avg_evacuation_time'] is not None:
            evacuation_times.append(results[label]['avg_evacuation_time'])
            valid_labels.append(label)

    if evacuation_times:
        bars6 = axes[1, 2].bar(valid_labels, evacuation_times, color=['#1abc9c', '#16a085', '#2ecc71', '#27ae60'][:len(valid_labels)])
        axes[1, 2].set_title('Temps d\'Évacuation Moyen')
        axes[1, 2].set_ylabel('Temps (s)')
        axes[1, 2].tick_params(axis='x', rotation=45)

        # Annoter les barres
        for bar, time_val in zip(bars6, evacuation_times):
            height = bar.get_height()
            axes[1, 2].text(bar.get_x() + bar.get_width()/2., height + 0.1,
                           f'{time_val:.1f}s', ha='center', va='bottom')
    else:
        axes[1, 2].text(0.5, 0.5, 'Données non disponibles',
                       ha='center', va='center', transform=axes[1, 2].transAxes)
        axes[1, 2].set_title('Temps d\'Évacuation Moyen')

    plt.tight_layout()

    # Sauvegarder le graphique
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_file = f"scenario_comparison_{timestamp}.png"
    plt.savefig(output_file, dpi=150, bbox_inches='tight')
    print(f"\n📈 Graphique de comparaison sauvegardé: {output_file}")

    # Afficher le graphique
    plt.show()

    # 5. Tableau comparatif
    print("\n5. 📋 TABLEAU COMPARATIF")
    print("\n" + "="*100)
    print(f"{'Scénario':<20} {'Évacuation':<12} {'Panique':<10} {'Collisions':<12} {'Distance':<12} {'Vitesse':<12} {'Temps (s)':<10}")
    print("="*100)

    for label in labels:
        data = results[label]
        evacuation_percent = data['evacuation_rate'] * 100
        panic = data['avg_panic']
        collisions = data['total_collisions']
        distance = data['avg_distance']
        speed = data['final_speed']
        time_str = f"{data['avg_evacuation_time']:.1f}" if data['avg_evacuation_time'] else "N/A"

        print(f"{label:<20} {evacuation_percent:>10.1f}% {panic:>9.3f} {collisions:>11.0f} {distance:>11.1f}m {speed:>11.2f}m/s {time_str:>10}")

    print("="*100)

    # 6. Recommandations
    print("\n6. 💡 RECOMMANDATIONS")

    # Trouver le meilleur scénario pour chaque métrique
    best_evacuation = max(results.items(), key=lambda x: x[1]['evacuation_rate'])
    lowest_panic = min(results.items(), key=lambda x: x[1]['avg_panic'])
    lowest_collisions = min(results.items(), key=lambda x: x[1]['total_collisions'])

    print(f"\n🎯 Meilleur taux d'évacuation: {best_evacuation[0]} ({best_evacuation[1]['evacuation_rate']*100:.1f}%)")
    print(f"😌 Moins de panique: {lowest_panic[0]} (niveau {lowest_panic[1]['avg_panic']:.3f})")
    print(f"🤕 Moins de collisions: {lowest_collisions[0]} ({lowest_collisions[1]['total_collisions']} collisions)")

    # Recommandation globale
    scores = {}
    for label, data in results.items():
        # Score composite (plus élevé = meilleur)
        score = (
            data['evacuation_rate'] * 0.4 +  # 40% pour l'évacuation
            (1 - data['avg_panic']) * 0.3 +  # 30% pour éviter la panique (inverse)
            (1 / (data['total_collisions'] + 1)) * 0.2 +  # 20% pour éviter les collisions
            (1 / (data['avg_distance'] + 1)) * 0.1  # 10% pour les distances courtes
        )
        scores[label] = score

    best_overall = max(scores.items(), key=lambda x: x[1])
    print(f"\n🏆 Recommandation globale: {best_overall[0]} (score: {best_overall[1]:.3f})")

    # 7. Sauvegarde des résultats
    print("\n7. 💾 SAUVEGARDE DES RÉSULTATS")

    results_dir = f"scenario_comparison_{timestamp}"
    os.makedirs(results_dir, exist_ok=True)

    # Sauvegarder les données
    comparison_data = {}
    for label in labels:
        comparison_data[label] = {
            'evacuation_rate': results[label]['evacuation_rate'],
            'avg_panic': results[label]['avg_panic'],
            'total_collisions': results[label]['total_collisions'],
            'avg_distance': results[label]['avg_distance'],
            'final_speed': results[label]['final_speed'],
            'final_stress': results[label]['final_stress'],
            'avg_evacuation_time': results[label]['avg_evacuation_time']
        }

    with open(f"{results_dir}/comparison_results.json", 'w') as f:
        json.dump(comparison_data, f, indent=2)

    # Copier le graphique
    shutil.copy2(output_file, f"{results_dir}/{output_file}")

    # Générer un rapport texte
    with open(f"{results_dir}/comparison_report.txt", 'w') as f:
        f.write("Rapport de Comparaison de Scénarios\n")
        f.write("="*50 + "\n\n")
        f.write(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Nombre de scénarios comparés: {len(scenarios_to_compare)}\n\n")

        f.write("Résultats détaillés:\n")
        f.write("-"*80 + "\n")
        for label in labels:
            data = results[label]
            f.write(f"\nScénario: {label}\n")
            f.write(f"  Taux d'évacuation: {data['evacuation_rate']*100:.1f}%\n")
            f.write(f"  Panique moyenne: {data['avg_panic']:.3f}\n")
            f.write(f"  Collisions totales: {data['total_collisions']}\n")
            f.write(f"  Distance moyenne: {data['avg_distance']:.1f}m\n")
            if data['avg_evacuation_time']:
                f.write(f"  Temps d'évacuation moyen: {data['avg_evacuation_time']:.1f}s\n")

        f.write("\n" + "="*50 + "\n")
        f.write("RECOMMANDATIONS:\n")
        f.write(f"Meilleur taux d'évacuation: {best_evacuation[0]}\n")
        f.write(f"Moins de panique: {lowest_panic[0]}\n")
        f.write(f"Moins de collisions: {lowest_collisions[0]}\n")
        f.write(f"Recommandation globale: {best_overall[0]}\n")

    print(f"\n✅ Résultats sauvegardés dans: {results_dir}")
    print(f"   • Graphique: {output_file}")
    print(f"   • Données: comparison_results.json")
    print(f"   • Rapport: comparison_report.txt")

    return results

# ============================================================================
# POINT D'ENTRÉE PRINCIPAL
# ============================================================================

if __name__ == "__main__":
    print("\n" + "="*80)
    print("🎯 SMA DE SIMULATION DE FOULE - VERSION AMÉLIORÉE")
    print("   Validation scientifique + Calibration Helbing + Métriques collectives")
    print("="*80)

    # Vérifier les dépendances
    print("\n🔍 VÉRIFICATION DES DÉPENDANCES...")
    try:
        import mesa
        import numpy as np
        import pandas as pd
        import networkx as nx
        import matplotlib.pyplot as plt
        print("✅ Dépendances principales: OK")
    except ImportError as e:
        print(f"❌ Dépendance manquante: {e}")
        print("💡 Installez avec: pip install mesa numpy pandas networkx matplotlib")
        sys.exit(1)

    # Menu principal
    while True:
        print("\n" + "="*50)
        print("MENU PRINCIPAL")
        print("="*50)
        print("1. Exécuter une simulation complète")
        print("2. Exécuter un test rapide")
        print("3. Comparer plusieurs scénarios")
        print("4. Générer uniquement une vidéo")
        print("5. Quitter")

        choice = input("\nChoisissez une option (1-5): ").strip()

        if choice == "1":
            run_comprehensive_simulation()
        elif choice == "2":
            run_test_simulation()
        elif choice == "3":
            compare_scenarios()
        elif choice == "4":
            generate_video_only()
        elif choice == "5":
            print("\n👋 Au revoir!")
            break
        else:
            print("❌ Choix invalide, veuillez réessayer.")


🎯 SMA DE SIMULATION DE FOULE - VERSION AMÉLIORÉE
   Validation scientifique + Calibration Helbing + Métriques collectives

🔍 VÉRIFICATION DES DÉPENDANCES...
✅ Dépendances principales: OK

MENU PRINCIPAL
1. Exécuter une simulation complète
2. Exécuter un test rapide
3. Comparer plusieurs scénarios
4. Générer uniquement une vidéo
5. Quitter

Choisissez une option (1-5): 3

📊 COMPARAISON DE SCÉNARIOS WHAT-IF

1. 📥 CHARGEMENT DES DONNÉES
Chemin du fichier CSV (défaut: trajectories.csv): /content/trajectories_corrected.csv
✅ Fichier trouvé: /content/trajectories_corrected.csv

2. 🎭 SÉLECTION DES SCÉNARIOS

Scénarios disponibles pour comparaison:
  1. Normal (baseline)
  2. Panique légère (intensité 0.3)
  3. Panique forte (intensité 0.7)
  4. 50% des sorties bloquées
  5. Haute densité (+50% d'agents)
  6. Évacuation dirigée par leaders

Entrez les numéros des scénarios à comparer (ex: 1,3,5): 1,2,3,4,5,6
   ✅ Ajouté: Baseline
   ✅ Ajouté: Panique légère
   ✅ Ajouté: Panique forte
   ✅ Ajo